#I. Download libraries

In [ ]:
!pip install pydicom opencv-python tqdm scikit-learn
!pip install tensorflow --quiet
!pip install openpyxl --quiet

In [ ]:
#Necessary imports
import tensorflow as tf
from tensorflow.keras.applications import Xception
from tensorflow.keras.models import Model
from sklearn.utils.class_weight import compute_class_weight
import matplotlib.pyplot as plt
from tqdm import tqdm
import seaborn as sns
import pandas as pd
from keras.callbacks import ModelCheckpoint
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import to_categorical, Sequence
from keras.callbacks import ModelCheckpoint, EarlyStopping
from tensorflow.keras.callbacks import ReduceLROnPlateau
from tensorflow.keras import regularizers
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from itertools import cycle
from sklearn.metrics import auc, roc_curve
from sklearn.metrics import RocCurveDisplay
from sklearn.metrics import precision_recall_fscore_support

##Call API from TCIA

In [ ]:
BASE_URL = "https://services.cancerimagingarchive.net/nbia-api/services/v1"

# List all collections
import requests
r = requests.get(f"{BASE_URL}/getCollectionValues?format=json")
r.raise_for_status()
collections = [c['Collection'] for c in r.json()]
print(collections)

# Get all series for a collection
collection = "CMMD"
r = requests.get(f"{BASE_URL}/getSeries?Collection={collection}&format=json")
r.raise_for_status()
series = r.json()
print(f"{len(series)} series found in {collection}")

# Download one series as a ZIP
import os, zipfile
os.makedirs("cmmd_data", exist_ok=True)
series_uid = series[0]['SeriesInstanceUID']
url = f"{BASE_URL}/getImage?SeriesInstanceUID={series_uid}"
resp = requests.get(url, stream=True)
with open(f"cmmd_data/{series_uid}.zip", "wb") as f:
    for chunk in resp.iter_content(8192):
        if chunk:
            f.write(chunk)
# Extract
with zipfile.ZipFile(f"cmmd_data/{series_uid}.zip", "r") as z:
    z.extractall(f"cmmd_data/{series_uid}")
print(f"Downloaded and extracted {series_uid}")

In [ ]:
import requests

BASE_URL = "https://services.cancerimagingarchive.net/nbia-api/services/v1"
COLLECTION = "CMMD"

r = requests.get(f"{BASE_URL}/getSeries?Collection={COLLECTION}&format=json")
r.raise_for_status()
series_list = r.json()  # this defines the variable
print(f"Found {len(series_list)} series in {COLLECTION}")

In [ ]:
#You must get the metadata file from TCIA
from google.colab import files
uploaded = files.upload()  # This will open a file picker

# Once uploaded, check the filename
clinical_xlsx = list(uploaded.keys())[0]

# Now you can read it
import pandas as pd
clin = pd.read_excel(clinical_xlsx)

In [ ]:
print("Unique classification values:")
print(clin["classification"].dropna().unique())

print("\nUnique subtype values:")
print(clin["subtype"].dropna().unique())

#II. Preprocessing

##Preprocessing v1.0

In [ ]:
import os
import zipfile
import io
import requests
import pydicom
import cv2
import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical

# --- CONFIGURATION ---
download_root = "/content/cmmd_data"
output_dir = "/content/CMMD_preprocessed/v1.0"
os.makedirs(download_root, exist_ok=True)
os.makedirs(output_dir, exist_ok=True)

# --- METADATA MAPPING ---
tumor_map = {'none':0, 'mass':1, 'calcification':2, 'both':3}
y_map = {
    'benign':0,
    'luminal a':1,
    'luminal b':2,
    'her2':3, 'her2-enriched':3,
    'triple negative':4, 'tn':4
}
num_classes = 5

# --- 1. HELPER: SPECTRAL RESIDUAL SALIENCY ---
def get_saliency_map(img):
    """
    Computes Spectral Residual Saliency Map (Hou & Zhang, CVPR 2007).
    """
    img_float = img.astype(np.float32)
    f = np.fft.fft2(img_float)
    magnitude = np.abs(f)
    phase = np.angle(f)
    log_amplitude = np.log(magnitude + 1e-9)
    avg_log_amplitude = cv2.blur(log_amplitude, (3, 3))
    spectral_residual = log_amplitude - avg_log_amplitude
    f_new = np.exp(spectral_residual + 1j * phase)
    saliency_map = np.abs(np.fft.ifft2(f_new))**2
    saliency_map = cv2.GaussianBlur(saliency_map, (9, 9), 2.5)
    saliency_map = cv2.normalize(saliency_map, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    return saliency_map

# --- 2. ENSEMBLE CROPPER ---
def crop_tumor_ensemble(img, size=(299, 299)):
    # Pre-processing
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    img_search = clahe.apply(img)
    img_search = cv2.GaussianBlur(img_search, (3, 3), 0)

    # Dynamic Kernel Size
    h, w = img.shape[:2]
    k_dynamic = max(3, int(min(h, w) * 0.015))
    if k_dynamic % 2 == 0: k_dynamic += 1

    # ============================================================
    # PHASE 1: THE SAFE ZONE (EROSION)
    # ============================================================
    blur = cv2.GaussianBlur(img_search, (5, 5), 0)
    _, mask_breast = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    kernel_safe = np.ones((k_dynamic, k_dynamic), np.uint8)
    mask_safe = cv2.erode(mask_breast, kernel_safe, iterations=2)

    # ============================================================
    # PHASE 2: GRAVITY PULL
    # ============================================================
    img_masked = cv2.bitwise_and(img_search, img_search, mask=mask_safe)
    M = cv2.moments(img_masked, binaryImage=False)

    if M["m00"] != 0:
        grav_x = int(M["m10"] / M["m00"])
        grav_y = int(M["m01"] / M["m00"])
    else:
        grav_x, grav_y = w//2, h//2

    # ============================================================
    # PHASE 3: DEFINE FOCUS ZONE
    # ============================================================
    focus_radius = min(h, w) // 3
    mask_focus = np.zeros_like(mask_safe)
    cv2.circle(mask_focus, (grav_x, grav_y), focus_radius, 255, -1)
    mask_final_search = cv2.bitwise_and(mask_safe, mask_focus)

    # ============================================================
    # HELPER: SMART SCORE (Log-Area * Brightness)
    # ============================================================
    def get_center_smart(mask, img_intensity):
        contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        if not contours: return None

        valid_contours = [c for c in contours if cv2.contourArea(c) > 20]
        if not valid_contours: return None

        best_c = None
        max_score = -1.0

        for c in valid_contours:
            area = cv2.contourArea(c)

            # 1. Measure Brightness
            c_mask = np.zeros_like(mask)
            cv2.drawContours(c_mask, [c], -1, 255, -1)
            mean_val = cv2.mean(img_intensity, mask=c_mask)[0]

            # 2. Calculate Hybrid Score
            # Score = Brightness * log(Area + 1)
            score = mean_val * np.log1p(area)

            if score > max_score:
                max_score = score
                M = cv2.moments(c)
                if M["m00"] != 0:
                    best_c = (int(M["m10"] / M["m00"]), int(M["m01"] / M["m00"]))

        return best_c

    # ============================================================
    # PHASE 4: VOTERS (Using Smart Scoring)
    # ============================================================

    # --- Voter 1: Otsu ---
    mask1 = cv2.bitwise_and(mask_breast, mask_breast, mask=mask_final_search)
    c1 = get_center_smart(mask1, img_search)

    # --- Voter 2: Saliency ---
    sal_map = get_saliency_map(img_search)
    _, mask2_raw = cv2.threshold(sal_map, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    mask2 = cv2.bitwise_and(mask2_raw, mask2_raw, mask=mask_final_search)
    c2 = get_center_smart(mask2, img_search)

    # --- Voter 3: DoG (Adaptive) ---
    g1 = cv2.GaussianBlur(img_search, (5, 5), 0)
    g2 = cv2.GaussianBlur(img_search, (9, 9), 0)
    dog = cv2.subtract(g1, g2)

    # Adaptive Threshold for DoG
    _, mask3_raw = cv2.threshold(dog, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    kernel_small = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    mask3_clean = cv2.morphologyEx(mask3_raw, cv2.MORPH_OPEN, kernel_small)
    mask3 = cv2.bitwise_and(mask3_clean, mask3_clean, mask=mask_final_search)

    c3 = get_center_smart(mask3, img_search)

    # --- Consensus Logic ---
    candidates = [c for c in [c1, c2, c3] if c is not None]
    final_center = (grav_x, grav_y)

    tolerance_dist = max(75, 0.10 * min(h, w))

    if candidates:
        if len(candidates) == 1:
            final_center = candidates[0]
        else:
            agreed = []
            for i in range(len(candidates)):
                for j in range(i + 1, len(candidates)):
                    dist = np.sqrt((candidates[i][0]-candidates[j][0])**2 + (candidates[i][1]-candidates[j][1])**2)
                    if dist < tolerance_dist:
                        agreed.append(candidates[i])
                        agreed.append(candidates[j])

            if agreed:
                avg_x = int(sum([p[0] for p in agreed]) / len(agreed))
                avg_y = int(sum([p[1] for p in agreed]) / len(agreed))
                final_center = (avg_x, avg_y)
            else:
                final_center = c2 if c2 else (c1 if c1 else c3)

    # --- Crop ---
    centerX, centerY = final_center
    half = size[0] // 2
    start_x = max(centerX - half, 0)
    start_y = max(centerY - half, 0)

    if start_x + size[0] > img.shape[1]: start_x = max(0, img.shape[1] - size[0])
    if start_y + size[1] > img.shape[0]: start_y = max(0, img.shape[0] - size[1])

    roi = img[start_y:start_y+size[1], start_x:start_x+size[0]]
    if roi.shape[0] != size[0] or roi.shape[1] != size[1]:
        roi = cv2.resize(roi, size)
    return roi

# ============================================================
# PHASE 5: STANDARD FUNCTIONS
# ============================================================
def resize_with_padding(img, target_size=(299, 299)):
    old_size = img.shape[:2]
    ratio = min(float(target_size[0])/old_size[0], float(target_size[1])/old_size[1])
    new_size = tuple([int(x*ratio) for x in old_size])
    img = cv2.resize(img, (new_size[1], new_size[0]))
    delta_w = target_size[1] - new_size[1]
    delta_h = target_size[0] - new_size[0]
    top, bottom = delta_h//2, delta_h-(delta_h//2)
    left, right = delta_w//2, delta_w-(delta_w//2)
    return cv2.copyMakeBorder(img, top, bottom, left, right, cv2.BORDER_CONSTANT, value=0)

def crop_breast_contour(img):
    _, thresh = cv2.threshold(img, 5, 255, cv2.THRESH_BINARY)
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if contours:
        c = max(contours, key=cv2.contourArea)
        x, y, w, h = cv2.boundingRect(c)
        return img[y:y+h, x:x+w]
    return img

def download_series(series_uid):
    extract_path = os.path.join(download_root, series_uid)
    if os.path.exists(extract_path) and len(os.listdir(extract_path)) > 0:
        return extract_path

    url = f"https://services.cancerimagingarchive.net/nbia-api/services/v1/getImage?SeriesInstanceUID={series_uid}" # From TCIA
    try:
        resp = requests.get(url, stream=True)
        resp.raise_for_status()
        z = zipfile.ZipFile(io.BytesIO(resp.content))
        z.extractall(extract_path)
        return extract_path
    except: return None

def load_dicom_dual(path):
    try:
        ds = pydicom.dcmread(path)
        img = ds.pixel_array

        # Handle Photometric Interpretation (Invert if needed)
        if hasattr(ds, 'PhotometricInterpretation') and ds.PhotometricInterpretation == 'MONOCHROME1':
            img = np.max(img) - img

        img = img.astype(np.float32)

        # -----------------------------------------------------------
        # 1. ROBUST SCALING
        # -----------------------------------------------------------
        # Instead of max(), we use 99th percentile to ignore metal clips
        # We perform this BEFORE cropping to ensure we get the intensity logic right
        max_val = np.percentile(img, 99)
        min_val = np.min(img)

        # Clip outlier pixels (metal tags) to the max_val
        img = np.clip(img, min_val, max_val)

        # Normalize to [0, 1] based on robust range
        if max_val > min_val:
            img = (img - min_val) / (max_val - min_val)
        else:
            img = np.zeros_like(img) # Edge case: image is flat color

        # -----------------------------------------------------------
        # 2. CONVERT TO uint8 [0, 255]
        # -----------------------------------------------------------
        # We map the robust [0, 1] range to [0, 255] integers.
        # This keeps storage efficient and contrast high.
        img = (img * 255.0).astype(np.uint8)

        # -----------------------------------------------------------
        # 3. CROP & RESIZE
        # -----------------------------------------------------------
        img_clean = crop_breast_contour(img)

        # Global Image
        img_g = resize_with_padding(img_clean, (299, 299))
        img_g = np.stack([img_g]*3, axis=-1)

        # Local Image
        img_l = crop_tumor_ensemble(img_clean, (299, 299))
        img_l = np.stack([img_l]*3, axis=-1)

        return img_g, img_l
    except Exception as e:
        print(f"Error loading {path}: {e}")
        return None, None

# --- MAIN LOOP ---
X_global, X_local, X_meta, y_labels = [], [], [], []
counter = {k:0 for k in ['benign','malignant','luminal a','luminal b','her2','triple negative']}

print(f" Starting Processing...")

# We need this for min-max normalization
valid_ages = clin['Age'].dropna()
AGE_MIN = valid_ages.min()
AGE_MAX = valid_ages.max()
print(f" Age Normalization Range: {AGE_MIN} - {AGE_MAX}")

for uid_entry in tqdm(series_list, desc="Processing Series"):
    uid = uid_entry['SeriesInstanceUID']
    folder_path = download_series(uid)
    if not folder_path: continue
    dicom_files = [f for f in os.listdir(folder_path) if f.endswith('.dcm')]
    if not dicom_files: continue

    try:
        patient_id = pydicom.dcmread(os.path.join(folder_path, dicom_files[0])).PatientID
        subject_rows = clin[clin['ID1'] == patient_id]
        if subject_rows.empty: continue
        row = subject_rows.iloc[0]

        classification_raw = str(row.get("classification","")).strip().lower()
        subtype_raw = str(row.get("subtype","")).strip().lower()
        abnormality_raw = str(row.get("abnormality","")).strip().lower()
        age_value = row.get("Age", np.nan)

        include_record = False
        if classification_raw == "benign":
            include_record = True
            final_class = "benign"
            final_subtype = "benign"
        elif classification_raw == "malignant" and pd.notna(age_value) and pd.notna(row.get("subtype")):
            include_record = True
            final_class = "malignant"
            final_subtype = subtype_raw

        if not include_record: continue

        y_label = y_map["benign"] if final_class=="benign" else y_map[final_subtype]
        # Age Normalization (Min-Max)
        if pd.isna(age_value):
            age_norm = 0.5 # Default to mean/middle if missing
        else:
            # Formula: (x - min) / (max - min)
            age_norm = (age_value - AGE_MIN) / (AGE_MAX - AGE_MIN)
        # Abnormality One-Hot Encoding
        # tumor_map has 4 keys (0, 1, 2, 3)
        tumor_idx = tumor_map.get(abnormality_raw, 0) # Default to 'none' (0)
        abnormality_onehot = [0] * 4
        abnormality_onehot[tumor_idx] = 1

        # Combine: [Age, Ab0, Ab1, Ab2, Ab3] -> Shape (5,)
        meta_vector = [age_norm] + abnormality_onehot

        for f in dicom_files:
            img_path = os.path.join(folder_path, f)
            img_g, img_l = load_dicom_dual(img_path)
            if img_g is not None:
                X_global.append(img_g)
                X_local.append(img_l)
                X_meta.append(meta_vector) #Append the new vector
                y_labels.append(y_label)

                # CORRECTED COUNTING LOGIC
                if final_class == "benign":
                    counter["benign"] += 1
                else:
                    counter["malignant"] += 1
                    if final_subtype == "luminal a": counter["luminal a"] += 1
                    elif final_subtype == "luminal b": counter["luminal b"] += 1
                    # Handles 'her2' and 'her2-enriched'
                    elif final_subtype in ["her2", "her2-enriched"]: counter["her2"] += 1
                    elif final_subtype in ["triple negative", "tn"]: counter["triple negative"] += 1
    except: continue

# --- SPLIT & SAVE ---
print("\n Splitting Data...")
X_global = np.array(X_global, dtype=np.uint8)
X_local = np.array(X_local, dtype=np.uint8)
X_meta = np.array(X_meta, dtype=np.float32) #Change to float32
y_labels = np.array(y_labels, dtype=np.int32)

Xg_train, Xg_temp, Xl_train, Xl_temp, Xm_train, Xm_temp, y_train, y_temp = train_test_split(
    X_global, X_local, X_meta, y_labels,
    test_size=0.28, random_state=42, stratify=y_labels
)
Xg_val, Xg_test, Xl_val, Xl_test, Xm_val, Xm_test, y_val, y_test = train_test_split(
    Xg_temp, Xl_temp, Xm_temp, y_temp,
    test_size=0.36, random_state=42, stratify=y_temp
)

y_train = to_categorical(y_train, num_classes)
y_val   = to_categorical(y_val,   num_classes)
y_test  = to_categorical(y_test,  num_classes)

print(f"\n Saving to {output_dir}...")
np.save(output_dir+'/X_train_global.npy', Xg_train)
np.save(output_dir+'/X_val_global.npy',   Xg_val)
np.save(output_dir+'/X_test_global.npy',  Xg_test)
np.save(output_dir+'/X_train_local.npy', Xl_train)
np.save(output_dir+'/X_val_local.npy',   Xl_val)
np.save(output_dir+'/X_test_local.npy',  Xl_test)
np.save(output_dir+'/X_train_metadata.npy', Xm_train)
np.save(output_dir+'/X_val_metadata.npy',   Xm_val)
np.save(output_dir+'/X_test_metadata.npy',  Xm_test)
np.save(output_dir+'/y_train.npy', y_train)
np.save(output_dir+'/y_val.npy',   y_val)
np.save(output_dir+'/y_test.npy',  y_test)

print("\n Dataset Summary:")
for k, v in counter.items():
    print(f"{k.title()}: {v}")
print(f"Total Training: {len(y_train)}")
print(f"Total Validation: {len(y_val)}")
print(f"Total Test: {len(y_test)}")
print("\n Done! Data is ready.")

#III. Image check

##Image check v1.0

In [ ]:
import matplotlib.pyplot as plt

print("Global Shape:", Xg_train[1].shape)
print("Local Shape:", Xl_train[1].shape)

fig, ax = plt.subplots(1, 2, figsize=(10, 5))
ax[0].imshow(Xg_train[2889])
ax[0].set_title("Global (Whole Breast)")
ax[1].imshow(Xl_train[2889])
ax[1].set_title("Local (ROI Crop)")
plt.show()

##Tower's image check v1.0

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt
import numpy as np
import cv2
import os

# ==========================================
# 1. CONFIGURATION & SETUP
# ==========================================
DATA_DIR = "/content/drive/MyDrive/CMMD_preprocessed/v1.0"
CLASS_NAMES = ['Benign', 'Luminal A', 'Luminal B', 'HER2', 'Triple Negative']

print(f" Looking for data in: {DATA_DIR}")
if not os.path.exists(DATA_DIR):
    print(f" WARNING: The directory {DATA_DIR} does not exist.")
    print("Please check where your .npy files are saved.")

# ==========================================
# 2. IMAGE PROCESSING UTILS
# ==========================================
def build_gamma_table(gamma):
    invGamma = 1.0 / gamma
    return np.array([((i / 255.0) ** invGamma) * 255 for i in np.arange(0, 256)]).astype("uint8")

def apply_gamma(img, table):
    return cv2.LUT(img, table)

def apply_sigmoid(img, gain, cutoff):
    img_f = img.astype(np.float32) / 255.0
    sigmoid = 1.0 / (1.0 + np.exp(-gain * (img_f - cutoff)))
    return (sigmoid * 255).astype(np.uint8)

# Pre-initialize tools to speed up the loop
clahe_mild = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
clahe_strong = cv2.createCLAHE(clipLimit=4.0, tileGridSize=(8,8))
gamma_mild_table = build_gamma_table(0.6)
gamma_strong_table = build_gamma_table(1.5)

# ==========================================
# 3. DATA LOADING HELPERS
# ==========================================
# We use a cache so we don't reload the huge .npy files every time you click a button
data_cache = {
    'train': {'Xg': None, 'Xl': None, 'y': None},
    'val':   {'Xg': None, 'Xl': None, 'y': None},
    'test':  {'Xg': None, 'Xl': None, 'y': None}
}

def load_split_data(split_name):
    """Lazy loader for dataset splits."""
    if data_cache[split_name]['y'] is not None:
        return data_cache[split_name] # Return cached

    print(f"⏳ Loading {split_name} data from disk... (One-time setup)")
    try:
        # Load Y fully (small)
        y_path = os.path.join(DATA_DIR, f"y_{split_name}.npy")
        y = np.load(y_path)

        # Load X with mmap (fast, low RAM)
        g_path = os.path.join(DATA_DIR, f"X_{split_name}_global.npy")
        l_path = os.path.join(DATA_DIR, f"X_{split_name}_local.npy")

        Xg = np.load(g_path, mmap_mode='r')
        Xl = np.load(l_path, mmap_mode='r')

        data_cache[split_name] = {'Xg': Xg, 'Xl': Xl, 'y': y}
        print(f" Loaded {len(y)} samples for {split_name}.")
        return data_cache[split_name]

    except FileNotFoundError as e:
        print(f" Error loading {split_name}: {e}")
        return None

# ==========================================
# 4. VISUALIZATION LOGIC
# ==========================================
def visualize_towers(split, index):
    data = load_split_data(split)
    if not data: return

    # 1. Retrieve Raw Images
    # The .npy shape is (N, 299, 299, 3) but all channels are identical grayscale.
    # We take channel 0 to get the raw 2D image.
    ig = data['Xg'][index][:,:,0]
    il = data['Xl'][index][:,:,0]
    label_onehot = data['y'][index]
    label_str = CLASS_NAMES[np.argmax(label_onehot)]

    # 2. Apply Tower Transformations
    # --- Tower 1: Global Gamma ---
    t1_c1 = ig
    t1_c2 = apply_gamma(ig, gamma_mild_table)
    t1_c3 = apply_gamma(ig, gamma_strong_table)

    # --- Tower 2: Global CLAHE ---
    t2_c1 = ig
    t2_c2 = clahe_mild.apply(ig)
    t2_c3 = clahe_strong.apply(ig)

    # --- Tower 3: Local Sigmoid ---
    t3_c1 = il
    t3_c2 = apply_sigmoid(il, gain=8, cutoff=0.75)
    t3_c3 = apply_sigmoid(il, gain=15, cutoff=0.60)

    # --- Tower 4: Local CLAHE ---
    t4_c1 = il
    t4_c2 = clahe_mild.apply(il)
    t4_c3 = clahe_strong.apply(il)

    # 3. Plotting
    fig, axes = plt.subplots(4, 3, figsize=(12, 16), constrained_layout=True)
    fig.suptitle(f"Dataset: {split.upper()} | Index: {index} | Class: {label_str}", fontsize=16, weight='bold')

    rows = [
        ("Tower 1: Global Gamma", [t1_c1, t1_c2, t1_c3], ["Original", "Gamma 0.6 (Brighten)", "Gamma 1.5 (Darken)"]),
        ("Tower 2: Global CLAHE", [t2_c1, t2_c2, t2_c3], ["Original", "CLAHE (Clip 2.0)", "CLAHE (Clip 4.0)"]),
        ("Tower 3: Local Sigmoid", [t3_c1, t3_c2, t3_c3], ["Original Local", "Sigmoid (G=8, C=0.75)", "Sigmoid (G=15, C=0.60)"]),
        ("Tower 4: Local CLAHE", [t4_c1, t4_c2, t4_c3], ["Original Local", "CLAHE (Clip 2.0)", "CLAHE (Clip 4.0)"])
    ]

    for row_idx, (tower_name, images, labels) in enumerate(rows):
        for col_idx, (img, txt) in enumerate(zip(images, labels)):
            ax = axes[row_idx, col_idx]
            ax.imshow(img, cmap='gray', vmin=0, vmax=255)
            ax.set_xticks([])
            ax.set_yticks([])

            # Row Title (Tower Name)
            if col_idx == 1:
                ax.set_title(tower_name, fontsize=14, fontweight='bold', pad=10)

            # Column Label (Modification Type)
            ax.set_xlabel(txt, fontsize=11)

    plt.show()

# ==========================================
# 5. WIDGET DASHBOARD
# ==========================================
style = {'description_width': 'initial'}
layout = widgets.Layout(width='300px')

# Widgets
dd_split = widgets.Dropdown(options=['train', 'val', 'test'], value='train', description='1️⃣ Dataset:', style=style, layout=layout)
dd_class = widgets.Dropdown(options=CLASS_NAMES, value='Benign', description='2️⃣ Filter Class:', style=style, layout=layout)
btn_scan = widgets.Button(description='Step 1: Scan for Cases', button_style='info', icon='search', layout=layout)

# Dropdown for ID (Disabled initially)
dd_id = widgets.Dropdown(options=[], description='3️⃣ Select ID:', disabled=True, style=style, layout=layout)
btn_run = widgets.Button(description='Step 2: Visualize Towers', button_style='success', icon='eye', disabled=True, layout=layout)

out_log = widgets.Output()
out_plot = widgets.Output()

# --- Callback: SCAN ---
def on_scan_click(b):
    out_log.clear_output()
    out_plot.clear_output() # Clear previous plot to avoid confusion

    split = dd_split.value
    target_class_name = dd_class.value
    target_class_idx = CLASS_NAMES.index(target_class_name)

    with out_log:
        print(f" Scanning '{split}' set for '{target_class_name}'...")

        data = load_split_data(split)
        if data is None:
            print(" Could not load data.")
            return

        # Filter indices where label matches
        # y is one-hot, so we use argmax
        y_integers = np.argmax(data['y'], axis=1)
        matching_indices = np.where(y_integers == target_class_idx)[0]

        if len(matching_indices) > 0:
            print(f" Found {len(matching_indices)} matching images.")

            # Update ID Dropdown
            dd_id.options = tuple(matching_indices)
            dd_id.value = matching_indices[0]
            dd_id.disabled = False
            btn_run.disabled = False
        else:
            print(f" No images found for class '{target_class_name}' in this split.")
            dd_id.options = []
            dd_id.disabled = True
            btn_run.disabled = True

# --- Callback: VISUALIZE ---
def on_run_click(b):
    out_plot.clear_output()
    with out_plot:
        try:
            visualize_towers(dd_split.value, dd_id.value)
        except Exception as e:
            print(f" Error during visualization: {e}")

# Bind Events
btn_scan.on_click(on_scan_click)
btn_run.on_click(on_run_click)

# Display UI
ui = widgets.VBox([
    widgets.HBox([dd_split, dd_class]),
    widgets.HBox([btn_scan, dd_id]),
    btn_run
])

print(" QUAD-STREAM INPUT VISUALIZER")
display(ui)
display(out_log)
display(out_plot)

#IV. Label Distribution

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import pydicom

# ------------------------------------------------------------
# STEP 1 — Identify all actual patients found in DICOM folders
# ------------------------------------------------------------
actual_patients = {}

for series_id in all_series:
    folder = os.path.join(download_root, series_id)
    if not os.path.isdir(folder):
        continue

    dicoms = [f for f in os.listdir(folder) if f.lower().endswith('.dcm')]
    if len(dicoms) == 0:
        continue

    try:
        ds = pydicom.dcmread(os.path.join(folder, dicoms[0]))
        pid = ds.PatientID
        actual_patients.setdefault(pid, []).append(series_id)
    except:
        continue

print("Number of patients with DICOM data:", len(actual_patients))

# ------------------------------------------------------------
# STEP 2 — INCLUSION RULES
# ------------------------------------------------------------
included_patients = set()

for pid in actual_patients.keys():
    row = clin[clin["ID1"] == pid]
    if row.empty:
        continue

    row = row.iloc[0]

    classification = str(row.get("classification", "")).strip().lower()
    subtype_raw   = str(row.get("subtype", "")).strip().lower()
    abnormality   = str(row.get("abnormality", "")).strip().lower()
    age_val       = row.get("Age", np.nan)

    include = False

    if classification == "benign":
        include = True

    elif classification == "malignant":
        if pd.notna(age_val) and pd.notna(row.get("subtype")) and pd.notna(row.get("abnormality")):
            include = True

    if include:
        included_patients.add(pid)

print(" Final kept patients:", len(included_patients))

# ------------------------------------------------------------
# STEP 3 — Filter clinical dataframe to only included patients
# ------------------------------------------------------------
clin_kept = clin[clin["ID1"].isin(included_patients)].copy()
clin_kept = clin_kept.drop_duplicates(subset=["ID1"], keep='first')

print("Clinical rows after filtering:", len(clin_kept))

# ------------------------------------------------------------
# STEP 4 — AGE distribution and statistics
# ------------------------------------------------------------
ages = clin_kept["Age"].dropna()

# --- Statistics ---
age_stats = {
    "Count": len(ages),
    "Mean": ages.mean(),
    "Median": ages.median(),
    "Std Dev": ages.std(),
    "Min": ages.min(),
    "Max": ages.max(),
    "25th percentile": ages.quantile(0.25),
    "75th percentile": ages.quantile(0.75)
}

print("\n Age Statistics (Kept Patients):")
for k, v in age_stats.items():
    print(f"{k}: {v:.2f}" if isinstance(v, float) else f"{k}: {v}")

# --- Age groups ---
age_groups = pd.cut(
    ages,
    bins=[0, 39, 49, 59, 69, 120],
    labels=["<40", "40–49", "50–59", "60–69", "70+"]
)
age_group_counts = age_groups.value_counts().sort_index()

# ------------------------------------------------------------
# STEP 5 — ABNORMALITY distribution
# ------------------------------------------------------------
abnorm = (
    clin_kept["abnormality"]
    .astype(str)
    .str.strip()
    .str.lower()
    .replace({
        "none": "none",
        "mass": "mass",
        "calcification": "calcification",
        "both": "both"
    })
)
abnorm_counts = abnorm.value_counts()

# ------------------------------------------------------------
# STEP 6 — PLOTS
# ------------------------------------------------------------
# — Age histogram —
plt.figure(figsize=(8,5))
sns.histplot(ages, bins=20, kde=True)
plt.title("Age Distribution (Kept Patients Only)")
plt.xlabel("Age")
plt.ylabel("Patients")
plt.show()

# — Age groups —
plt.figure(figsize=(8,5))
sns.barplot(x=age_group_counts.index, y=age_group_counts.values, palette="viridis")
plt.title("Age Groups (Kept Patients Only)")
plt.xlabel("Age Group")
plt.ylabel("Patients")
for i, v in enumerate(age_group_counts.values):
    plt.text(i, v, str(v), ha="center", va="bottom")
plt.show()

# — Abnormality —
plt.figure(figsize=(8,5))
sns.barplot(x=abnorm_counts.index.str.title(), y=abnorm_counts.values, palette="magma")
plt.title("Abnormality Types (Kept Patients Only)")
plt.xlabel("Abnormality")
plt.ylabel("Patients")
for i, v in enumerate(abnorm_counts.values):
    plt.text(i, v, str(v), ha="center", va="bottom")
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# --- Helper to convert labels to integer class IDs ---
def to_class_idx(arr):
    if arr.ndim == 2: # one-hot
        return np.argmax(arr, axis=1)
    return arr.astype(int)


# === PREPARE LABEL ARRAYS ===
labels_train_idx = to_class_idx(y_train)
labels_val_idx   = to_class_idx(y_val)
labels_test_idx  = to_class_idx(y_test)

label_names = ['Benign', 'Luminal A', 'Luminal B', 'HER2', 'Triple Negative']


# === FUNCTION TO PLOT A DISTRIBUTION ===
def plot_distribution(label_idx, title):
    counts = np.bincount(label_idx, minlength=5)

    print("\n" + title)
    print("-------------------------------------")
    print("Total images:", counts.sum())
    for i, name in enumerate(label_names):
        print(f"{name}: {counts[i]}")

    df = pd.DataFrame({'ClassIdx': np.arange(5), 'Count': counts})

    plt.figure(figsize=(9,5))
    ax = sns.barplot(x='ClassIdx', y='Count', data=df, palette='viridis')
    ax.set_xticks(range(5))
    ax.set_xticklabels(label_names, rotation=45)
    ax.set_xlabel('')
    ax.set_ylabel('Number of Images')
    ax.set_title(title)

    # Annotate counts on bars
    for p in ax.patches:
        h = int(p.get_height())
        ax.annotate(f'{h}', (p.get_x() + p.get_width()/2, h),
                    ha='center', va='bottom', fontsize=11)

    plt.tight_layout()
    plt.show()


# === PLOT ALL 3 DISTRIBUTIONS ===
plot_distribution(labels_train_idx, "Training Dataset Distribution")

plot_distribution(labels_val_idx,   "Validation Dataset Distribution")

plot_distribution(labels_test_idx,  "Test Dataset Distribution")

#V. Saving .npy files to Drive

##Import npy files to Drive v1.0

In [ ]:
from google.colab import drive
import shutil
import os

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Define Paths
source_dir = "/content/CMMD_preprocessed/v1.0"

# Destination to your Google Drive
drive_output = '/content/drive/MyDrive/CMMD_preprocessed/v1.0'
os.makedirs(drive_output, exist_ok=True)

# 3. List of ALL 12 Files
npy_files = [
    # Global Images (Whole Breast)
    'X_train_global.npy', 'X_val_global.npy', 'X_test_global.npy',

    # Local Images (ROI Crops)
    'X_train_local.npy', 'X_val_local.npy', 'X_test_local.npy',

    # Metadata
    'X_train_metadata.npy', 'X_val_metadata.npy', 'X_test_metadata.npy',

    # Labels
    'y_train.npy', 'y_val.npy', 'y_test.npy'
]

# 4. Move Loop
print(f"🚀 Moving {len(npy_files)} files to Google Drive...")

for f in npy_files:
    src = os.path.join(source_dir, f)
    dst = os.path.join(drive_output, f)

    if os.path.exists(src):
        # If space is an issue, change shutil.copy to shutil.move
        shutil.copy(src, dst)
        print(f"✅ Saved: {f}")
    else:
        print(f" File not found (skipped): {f}")

print(f"\n All data saved to: {drive_output}")

#VI. Loading npy data files from Drive

##Load .npy files v1.0

In [ ]:
!fusermount -u /content/drive
!rm -rf /content/drive
!mkdir /content/drive
from google.colab import drive
import numpy as np
import os

# --- 1. Mount Google Drive ---
drive.mount('/content/drive', force_remount=True)

# --- 2. Set path to preprocessed files ---
files_path = '/content/drive/MyDrive/CMMD_preprocessed/v1.0'

print(f" Loading data from: {files_path}")

# --- 3. Load GLOBAL Images (Whole Breast) ---
print(" Loading Global Images...")
X_train_global = np.load(os.path.join(files_path, 'X_train_global.npy'))
X_val_global   = np.load(os.path.join(files_path, 'X_val_global.npy'))
X_test_global  = np.load(os.path.join(files_path, 'X_test_global.npy'))

# --- 4. Load LOCAL Images (ROI Crop) ---
print(" Loading Local Images...")
X_train_local = np.load(os.path.join(files_path, 'X_train_local.npy'))
X_val_local   = np.load(os.path.join(files_path, 'X_val_local.npy'))
X_test_local  = np.load(os.path.join(files_path, 'X_test_local.npy'))

# --- 5. Load Metadata & Labels ---
print(" Loading Metadata & Labels...")
X_train_metadata = np.load(os.path.join(files_path, 'X_train_metadata.npy'))
X_val_metadata   = np.load(os.path.join(files_path, 'X_val_metadata.npy'))
X_test_metadata  = np.load(os.path.join(files_path, 'X_test_metadata.npy'))

y_train = np.load(os.path.join(files_path, 'y_train.npy'))
y_val   = np.load(os.path.join(files_path, 'y_val.npy'))
y_test  = np.load(os.path.join(files_path, 'y_test.npy'))

# --- 6. VERIFICATION ---
print("\n Data Loaded Successfully!")
print("="*40)

# 1. Training Set
print(f"TRAIN SET: {len(y_train)} samples")
print(f"  - Global: {X_train_global.shape}")
print(f"  - Local:  {X_train_local.shape}")
print(f"  - Meta:   {X_train_metadata.shape}")
print(f"  - Labels: {y_train.shape}")

print("-" * 20)

# 2. Validation Set
print(f"VALIDATION SET: {len(y_val)} samples")
print(f"  - Global: {X_val_global.shape}")
print(f"  - Local:  {X_val_local.shape}")
print(f"  - Meta:   {X_val_metadata.shape}")
print(f"  - Labels: {y_val.shape}")

print("-" * 20)

# 3. Test Set
print(f"TEST SET: {len(y_test)} samples")
print(f"  - Global: {X_test_global.shape}")
print(f"  - Local:  {X_test_local.shape}")
print(f"  - Meta:   {X_test_metadata.shape}")
print(f"  - Labels: {y_test.shape}")
print("="*40)

#VII. Visualize images from the npy files

####Visualize images v1.0

In [ ]:
import matplotlib.pyplot as plt
import random
import cv2
import numpy as np

def visualize_v1_13_selective_sigmoid(X_global, X_local, y, dataset_name, num_samples=3):
    print(f"\n🔍 Visualizing {num_samples} samples from: {dataset_name} SET")

    # --- 1. Setup Helpers ---
    clahe_mild = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8,8))

    invGamma = 1.0 / 0.6
    gamma_table = np.array([((i / 255.0) ** invGamma) * 255 for i in np.arange(0, 256)]).astype("uint8")

    # Highly Selective Sigmoid
    # Targets only the brightest pixels (tumor core & calcifications)
    def apply_selective_sigmoid(img):
        # Convert to float 0-1
        img_f = img.astype(np.float32) / 255.0

        # Gain: Slightly softer to preserve a tiny bit of edge gradient
        gain = 8.0
        # Cutoff: Much higher. Only pixels > 0.75 brightness get boosted.
        cutoff = 0.75

        # Formula
        sigmoid = 1.0 / (1.0 + np.exp(-gain * (img_f - cutoff)))

        # Normalize back to 0-255
        sigmoid = (sigmoid * 255).astype(np.uint8)

        return sigmoid

    indices = random.sample(range(len(y)), num_samples)
    fig, axes = plt.subplots(num_samples, 6, figsize=(24, 4 * num_samples))
    if num_samples == 1: axes = [axes]

    for row_idx, data_idx in enumerate(indices):
        raw_global = X_global[data_idx]
        raw_local = X_local[data_idx]

        # --- ENSURE UINT8 & COPY ---
        if len(raw_global.shape) == 3: gray_g = raw_global[:,:,0].copy()
        else: gray_g = raw_global.copy()
        if gray_g.dtype != np.uint8: gray_g = gray_g.astype(np.uint8)

        if len(raw_local.shape) == 3: gray_l = raw_local[:,:,0].copy()
        else: gray_l = raw_local.copy()
        if gray_l.dtype != np.uint8: gray_l = gray_l.astype(np.uint8)

        # --- GENERATE VIEWS ---
        g_orig = gray_g
        g_clahe = clahe_mild.apply(gray_g)
        g_gamma = cv2.LUT(gray_g, gamma_table)

        l_orig = gray_l
        l_clahe = clahe_mild.apply(gray_l)

        # APPLY SELECTIVE SIGMOID
        l_sigmoid = apply_selective_sigmoid(gray_l)

        label = np.argmax(y[data_idx])

        # --- PLOTTING ---
        axes[row_idx][0].imshow(g_orig, cmap='gray')
        axes[row_idx][0].set_title(f"GLOBAL (Label: {label})\nCh1: Original")
        axes[row_idx][0].axis('off')

        axes[row_idx][1].imshow(g_clahe, cmap='gray')
        axes[row_idx][1].set_title("GLOBAL\nCh2: CLAHE")
        axes[row_idx][1].axis('off')

        axes[row_idx][2].imshow(g_gamma, cmap='gray')
        axes[row_idx][2].set_title("GLOBAL\nCh3: Gamma 0.6")
        axes[row_idx][2].axis('off')

        axes[row_idx][3].imshow(l_orig, cmap='gray')
        axes[row_idx][3].set_title("LOCAL\nCh1: Original")
        axes[row_idx][3].axis('off')

        axes[row_idx][4].imshow(l_clahe, cmap='gray')
        axes[row_idx][4].set_title("LOCAL\nCh2: CLAHE")
        axes[row_idx][4].axis('off')

        # NEW VISUALIZATION
        axes[row_idx][5].imshow(l_sigmoid, cmap='gray')
        axes[row_idx][5].set_title("LOCAL\nCh3: Selective Sigmoid\n(High Density Core)")
        axes[row_idx][5].axis('off')

    plt.tight_layout()
    plt.show()

if 'X_train_global' in locals():
    visualize_v1_13_selective_sigmoid(X_train_global, X_train_local, y_train, "TRAIN", num_samples=3)

Demonstrate the ensemble cropping

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import random

# ==========================================
# 1. SETUP & DATA CHECK
# ==========================================
if 'X_train_global' not in locals() or 'X_train_local' not in locals():
    print(" Data not found!")
    X_train_global = np.random.rand(10, 299, 299, 3).astype(np.float32)
    X_train_local  = np.random.rand(10, 299, 299, 3).astype(np.float32)
    y_train = np.array([[1,0,0,0,0], [0,1,0,0,0]]*5)

# ==========================================
# 2. ENSEMBLE VISUALIZATION LOGIC
# ==========================================
def get_saliency_map(img):
    img_float = img.astype(np.float32)
    f = np.fft.fft2(img_float)
    magnitude = np.abs(f)
    phase = np.angle(f)
    log_amplitude = np.log(magnitude + 1e-9)
    avg_log_amplitude = cv2.blur(log_amplitude, (3, 3))
    spectral_residual = log_amplitude - avg_log_amplitude
    f_new = np.exp(spectral_residual + 1j * phase)
    saliency_map = np.abs(np.fft.ifft2(f_new))**2
    saliency_map = cv2.GaussianBlur(saliency_map, (9, 9), 2.5)
    return cv2.normalize(saliency_map, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)

def visualize_voting_on_global(img_input):
    # Safe convert to uint8
    if img_input.max() <= 1.5:
        img_uint8 = (img_input * 255).astype(np.uint8)
    else:
        img_uint8 = img_input.astype(np.uint8)

    if img_uint8.ndim == 3 and img_uint8.shape[-1] == 3:
        img_gray = cv2.cvtColor(img_uint8, cv2.COLOR_RGB2GRAY)
        img_vis = img_uint8.copy()
    else:
        img_gray = img_uint8.squeeze()
        img_vis = cv2.cvtColor(img_gray, cv2.COLOR_GRAY2RGB)

    # Pre-processing
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    img_search = clahe.apply(img_gray)
    img_search = cv2.GaussianBlur(img_search, (3, 3), 0)

    h, w = img_gray.shape[:2]
    k_dynamic = max(3, int(min(h, w) * 0.015))
    if k_dynamic % 2 == 0: k_dynamic += 1

    # ============================================================
    # HELPER: SMART SCORE (Log-Area * Brightness)
    # ============================================================
    def get_center_smart(mask, img_intensity):
        contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        if not contours: return None

        valid_contours = [c for c in contours if cv2.contourArea(c) > 20]
        if not valid_contours: return None

        best_c = None
        max_score = -1.0

        for c in valid_contours:
            area = cv2.contourArea(c)

            c_mask = np.zeros_like(mask)
            cv2.drawContours(c_mask, [c], -1, 255, -1)
            mean_val = cv2.mean(img_intensity, mask=c_mask)[0]

            # The Logic: Score = Brightness * log(Area + 1)
            score = mean_val * np.log1p(area)

            if score > max_score:
                max_score = score
                M = cv2.moments(c)
                if M["m00"] != 0:
                    best_c = (int(M["m10"] / M["m00"]), int(M["m01"] / M["m00"]))

        return best_c

    # --- PHASE 1: SAFE ZONE ---
    blur = cv2.GaussianBlur(img_search, (5, 5), 0)
    _, mask_breast = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    kernel_safe = np.ones((k_dynamic, k_dynamic), np.uint8)
    mask_safe = cv2.erode(mask_breast, kernel_safe, iterations=2)

    # --- PHASE 2: GRAVITY ---
    img_masked = cv2.bitwise_and(img_search, img_search, mask=mask_safe)
    M = cv2.moments(img_masked, binaryImage=False)

    if M["m00"] != 0:
        grav_x = int(M["m10"] / M["m00"])
        grav_y = int(M["m01"] / M["m00"])
    else:
        grav_x, grav_y = w//2, h//2

    # --- PHASE 3: FOCUS ZONE ---
    focus_radius = min(h, w) // 3
    mask_focus = np.zeros_like(mask_safe)
    cv2.circle(mask_focus, (grav_x, grav_y), focus_radius, 255, -1)
    mask_final_search = cv2.bitwise_and(mask_safe, mask_focus)

    # --- PHASE 4: VOTERS (Smart Scoring) ---
    # 1. Otsu
    mask1 = cv2.bitwise_and(mask_breast, mask_breast, mask=mask_final_search)
    c1 = get_center_smart(mask1, img_search)

    # 2. Saliency
    sal_map = get_saliency_map(img_search)
    _, mask2_raw = cv2.threshold(sal_map, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    mask2 = cv2.bitwise_and(mask2_raw, mask2_raw, mask=mask_final_search)
    c2 = get_center_smart(mask2, img_search)

    # 3. DoG (Adaptive Otsu)
    g1 = cv2.GaussianBlur(img_search, (5, 5), 0)
    g2 = cv2.GaussianBlur(img_search, (9, 9), 0)
    dog = cv2.subtract(g1, g2)
    _, mask3_raw = cv2.threshold(dog, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    # Gentle Cleaning for DoG
    kernel_small = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    mask3_clean = cv2.morphologyEx(mask3_raw, cv2.MORPH_OPEN, kernel_small)
    mask3 = cv2.bitwise_and(mask3_clean, mask3_clean, mask=mask_final_search)
    c3 = get_center_smart(mask3, img_search)

    # --- VOTING ---
    candidates = [c for c in [c1, c2, c3] if c is not None]
    final_center = (grav_x, grav_y)
    method_str = "Gravity Anchor (Fallback)"
    tolerance_dist = max(20, 0.05 * min(h, w))

    if candidates:
        if len(candidates) == 1:
            final_center = candidates[0]
            method_str = "Single Candidate"
        else:
            agreed = []
            pairs = []
            if c1 and c2: pairs.append((c1, c2, "Otsu+Sal"))
            if c1 and c3: pairs.append((c1, c3, "Otsu+DoG"))
            if c2 and c3: pairs.append((c2, c3, "Sal+DoG"))

            pairs_found = False
            for p1, p2, name in pairs:
                dist = np.sqrt((p1[0]-p2[0])**2 + (p1[1]-p2[1])**2)
                if dist < tolerance_dist:
                    agreed.append(p1)
                    agreed.append(p2)
                    method_str = f"Consensus ({name})"
                    pairs_found = True
                    break

            if pairs_found:
                avg_x = int(sum([p[0] for p in agreed])/len(agreed))
                avg_y = int(sum([p[1] for p in agreed])/len(agreed))
                final_center = (avg_x, avg_y)
            else:
                method_str = "Hierarchy (Saliency>Otsu)"
                final_center = c2 if c2 else (c1 if c1 else c3)

    # --- DRAWING ---
    contours_safe, _ = cv2.findContours(mask_safe, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    cv2.drawContours(img_vis, contours_safe, -1, (255, 255, 255), 1)
    cv2.circle(img_vis, (grav_x, grav_y), focus_radius, (0, 255, 255), 1)
    cv2.drawMarker(img_vis, (grav_x, grav_y), (255, 255, 255), markerType=cv2.MARKER_CROSS, markerSize=15, thickness=2)

    # Draw Voters (Concentric: Red > Green > Blue)
    if c1: cv2.circle(img_vis, c1, 18, (255, 0, 0), -1)   # Red
    if c2: cv2.circle(img_vis, c2, 11, (0, 255, 0), -1)   # Green
    if c3: cv2.circle(img_vis, c3, 5,  (0, 0, 255), -1)   # Blue

    cx, cy = final_center
    box_size = 40
    cv2.rectangle(img_vis, (cx-box_size, cy-box_size), (cx+box_size, cy+box_size), (255, 255, 0), 2)

    return img_vis, method_str

# ==========================================
# 3. SELECT SAMPLES
# ==========================================
benign_idxs = []
malignant_idxs = []
scan_limit = min(300, len(y_train))
scan_range = random.sample(range(len(y_train)), scan_limit)

for idx in scan_range:
    lbl = y_train[idx]
    if lbl.ndim > 0 and lbl.size > 1: class_id = np.argmax(lbl)
    elif lbl.ndim > 0 and lbl.size == 1: class_id = int(lbl.item())
    else: class_id = int(lbl)

    if class_id == 0: benign_idxs.append(idx)
    else: malignant_idxs.append(idx)

if len(benign_idxs) < 1 or len(malignant_idxs) < 2:
    selected_indices = random.sample(range(len(y_train)), 3)
else:
    selected_indices = [random.choice(benign_idxs)] + random.sample(malignant_idxs, 2)
    print(f" Selected indices: {selected_indices}")

# ==========================================
# 4. GENERATE PLOT
# ==========================================
clahe_strong = cv2.createCLAHE(clipLimit=4.0, tileGridSize=(8,8))

fig, axes = plt.subplots(3, 3, figsize=(12, 12))
rows_labels = ['Original Global', 'Smart Scoring', 'True Local Input']

for ax, row in zip(axes[:,0], rows_labels):
    ax.set_ylabel(row, rotation=90, size='large', weight='bold')

col_headers = ['Column 1: Benign', 'Column 2: Malignant', 'Column 3: Malignant']

for i, idx in enumerate(selected_indices):
    img_global = X_train_global[idx]
    img_local_true = X_train_local[idx]
    vis_voting, method = visualize_voting_on_global(img_global)

    if img_local_true.max() <= 1.5:
        local_uint8 = (img_local_true * 255).astype(np.uint8)
    else:
        local_uint8 = img_local_true.astype(np.uint8)

    if local_uint8.ndim == 3: local_gray = local_uint8[:,:,0]
    else: local_gray = local_uint8
    local_display = clahe_strong.apply(local_gray)

    axes[0, i].imshow(img_global)
    axes[0, i].set_title(f"{col_headers[i]}\n(Idx: {idx})")
    axes[0, i].axis('off')
    axes[1, i].imshow(vis_voting)
    axes[1, i].set_title(f"Focus Zone & Votes\n{method}")
    axes[1, i].axis('off')
    axes[2, i].imshow(local_display, cmap='gray')
    axes[2, i].set_title(f"Dataset Input (Local)\n{local_uint8.shape}")
    axes[2, i].axis('off')

plt.tight_layout()
plt.show()

#VIII. Xception model

###Training model v1.0

#####Phase 1 training

In [ ]:
import os
import glob
import re
import pickle
import numpy as np
import cv2
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.utils import Sequence
from tensorflow.keras.applications.xception import preprocess_input
from tensorflow.keras import layers, Model, Input
import albumentations as A

# ==========================================
# 1. CUSTOM LOSS FUNCTION (CLASS-WEIGHTED FOCAL LOSS)
# ==========================================
def class_weighted_focal_loss(gamma=2.5, alpha_weights=None):
    """
    gamma: Reduced to 2.5. Gamma 3.0 was too aggressive and suppressed gradients
           for "moderately hard" samples (like Luminal A), hurting their learning.
    """
    # Convert alpha to a tensor constant for efficiency
    if alpha_weights is not None:
        alpha_tensor = tf.constant(alpha_weights, dtype=tf.float32)
    else:
        alpha_tensor = None

    def focal_loss_fixed(y_true, y_pred):
        epsilon = tf.keras.backend.epsilon()
        y_pred = tf.clip_by_value(y_pred, epsilon, 1. - epsilon)

        # Step 1: Get probability of TRUE class only
        # y_true: (batch, 5), y_pred: (batch, 5)
        y_true = tf.cast(y_true, tf.float32)

        # p_t = probability assigned to the true class for each sample
        # Shape: (batch_size,)
        pt = tf.reduce_sum(y_true * y_pred, axis=-1)  # sum(y_true_i * y_pred_i) = p_true_class

        # Step 2: Basic cross entropy for true class
        ce = -tf.math.log(pt + epsilon)  # Shape: (batch_size,)

        # Step 3: Focal weighting based on how confident we are in TRUE class
        focal_weight = tf.math.pow(1 - pt, gamma)  # Shape: (batch_size,)

        # 3. Calculate Class Weight Term
        # Multiply by alpha corresponding to the true class
        if alpha_tensor is not None:
            # Get alpha for the true class of each sample
            # y_true * alpha gives (batch, 5) with alpha only at true class position
            # Then sum to get shape (batch_size,)
            alpha_t = tf.reduce_sum(y_true * alpha_tensor, axis=-1)
            loss = alpha_t * focal_weight * ce
        else:
            loss = focal_weight * ce

        return tf.reduce_mean(loss)  # Mean over batch

    return focal_loss_fixed

# ==========================================
# 2. QUAD-STREAM GENERATOR
# ==========================================
class QuadStreamGenerator(Sequence):
    def __init__(self, x_global, x_local, x_meta, y, batch_size,
                 augment=False, shuffle=True,
                 oversample_ratio=0.0):
        super().__init__()
        self.x_global = x_global
        self.x_local = x_local
        self.x_meta = x_meta
        self.y = y
        self.batch_size = batch_size
        self.augment = augment
        self.shuffle = shuffle

        # Oversampling Setup
        self.oversample_ratio = oversample_ratio
        self.indexes = np.arange(len(self.x_global))

        # Calculate Sampling Probabilities if oversampling is enabled
        self.sampling_probs = None
        if self.oversample_ratio > 0:
            self.sampling_probs = self._calculate_sampling_probs()

        # --- ENHANCEMENT OBJECTS ---
        self.clahe_mild = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
        self.clahe_strong = cv2.createCLAHE(clipLimit=4.0, tileGridSize=(8,8))
        self.gamma_mild_table = self._build_gamma_table(0.6)
        self.gamma_strong_table = self._build_gamma_table(1.5)

        # --- SPECIALIZED AUGMENTATION PIPELINES ---
        # Using Albumentations to treat each class biologically correctly
        if self.augment:
            self.aug_pipelines = self._build_augmentation_pipelines()

        self.on_epoch_end()

    def _build_augmentation_pipelines(self):
        # 1. Base Geometric (Safe for all)
        # We use additional_targets to apply SAME transform to all 4 towers
        base_aug = [
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.5),
            A.Affine(scale=(0.9, 1.1), translate_percent=(0.1, 0.1), rotate=(-20, 20), p=0.5),
        ]

        # 2. Define Class-Specific Pipelines
        pipelines = {}

        # --- Class 0: BENIGN ---
        pipelines[0] = A.Compose(base_aug + [
            A.MedianBlur(blur_limit=5, p=0.3),
        ], additional_targets={'image2': 'image', 'image3': 'image', 'image4': 'image'})

        # --- Class 1: LUMINAL A ---
        pipelines[1] = A.Compose(base_aug + [
             A.ISONoise(color_shift=(0.01, 0.05), intensity=(0.1, 0.5), p=0.2),
        ], additional_targets={'image2': 'image', 'image3': 'image', 'image4': 'image'})

        # --- Class 2: LUMINAL B (TARGET) ---
        pipelines[2] = A.Compose(base_aug + [
            # 1. Geometry: Create the spicule shape physically
            A.ElasticTransform(alpha=1, sigma=50, p=0.5),

            # 2. Structure: Highlight the edges (MILD settings)
            A.Emboss(alpha=(0.2, 0.5), strength=(0.3, 0.5), p=0.4),
        ], additional_targets={'image2': 'image', 'image3': 'image', 'image4': 'image'})

        # --- Class 3: HER2+ ---
        pipelines[3] = A.Compose(base_aug + [
            A.Sharpen(alpha=(0.2, 0.5), lightness=(0.5, 1.0), p=0.5),
        ], additional_targets={'image2': 'image', 'image3': 'image', 'image4': 'image'})

        # --- Class 4: TN (TARGET) ---
        pipelines[4] = A.Compose(base_aug + [
            # 1. Geometry: Create the "Bulge" / Mass Effect
            A.GridDistortion(num_steps=5, distort_limit=0.3, p=0.5),

            # 2. Structure: Define the sharp margin (MILD)
            A.Sharpen(alpha=(0.1, 0.3), lightness=(0.5, 1.0), p=0.4),

            # 3. Density: High Density (Dark/Bright shift)
            # Safe Range: (85, 115)
            A.RandomGamma(gamma_limit=(85, 115), p=0.5),
        ], additional_targets={'image2': 'image', 'image3': 'image', 'image4': 'image'})

        return pipelines

    def _calculate_sampling_probs(self):
        # 1. Get integer labels
        y_int = np.argmax(self.y, axis=1)
        classes, counts = np.unique(y_int, return_counts=True)
        # Calculate weight using Inverse Frequency (1.0 / count)
        # This ensures every class has roughly EQUAL probability of being picked per batch
        class_weight_map = {c: 1.0/count for c, count in zip(classes, counts)}

        sample_weights = np.array([class_weight_map[label] for label in y_int])

        # Normalize to sum to 1.0 for np.random.choice
        return sample_weights / sample_weights.sum()

    def _build_gamma_table(self, gamma):
        invGamma = 1.0 / gamma
        return np.array([((i / 255.0) ** invGamma) * 255 for i in np.arange(0, 256)]).astype("uint8")

    def apply_gamma(self, img, table):
        return cv2.LUT(img, table)

    def apply_sigmoid(self, img, gain, cutoff):
        img_f = img.astype(np.float32) / 255.0
        sigmoid = 1.0 / (1.0 + np.exp(-gain * (img_f - cutoff)))
        return (sigmoid * 255).astype(np.uint8)

    def __len__(self):
        return int(np.ceil(len(self.x_global) / self.batch_size))

    def on_epoch_end(self):
        # Logic to handle Weighted Random Sampling
        if self.sampling_probs is not None and self.shuffle:
            # Sample WITH REPLACEMENT based on calculated probabilities
            # This creates a new list of indexes where minority classes appear more often
            self.indexes = np.random.choice(
                np.arange(len(self.x_global)),
                size=len(self.x_global),
                p=self.sampling_probs,
                replace=True
            )
        elif self.shuffle:
            np.random.shuffle(self.indexes)

    def __getitem__(self, index):
        idxs = self.indexes[index*self.batch_size:(index+1)*self.batch_size]

        s1, s2, s3, s4 = [], [], [], []
        bm = self.x_meta[idxs]
        by = self.y[idxs]

        for i in range(len(idxs)):
            # Identify Class for Specialization
            # You must extract the integer label (0-4) to pick the right pipeline
            current_label_idx = np.argmax(by[i])

            # Load raw images
            ig_raw = self.x_global[idxs[i]]
            il_raw = self.x_local[idxs[i]]

            if ig_raw.ndim == 3: ig = ig_raw[:,:,0].astype(np.uint8)
            else: ig = ig_raw.astype(np.uint8)

            if il_raw.ndim == 3: il = il_raw[:,:,0].astype(np.uint8)
            else: il = il_raw.astype(np.uint8)

            # --- PREPARE STACKS ---
            # Tower 1: Global Gamma
            stack_1 = np.stack([
                ig,
                self.apply_gamma(ig, self.gamma_mild_table),
                self.apply_gamma(ig, self.gamma_strong_table)
            ], axis=-1)

            # Tower 2: Global CLAHE
            stack_2 = np.stack([
                ig,
                self.clahe_mild.apply(ig),
                self.clahe_strong.apply(ig)
            ], axis=-1)

            # Tower 3: Local Sigmoid
            stack_3 = np.stack([
                il,
                self.apply_sigmoid(il, gain=8, cutoff=0.75),
                self.apply_sigmoid(il, gain=15, cutoff=0.60)
            ], axis=-1)

            # Tower 4: Local CLAHE
            stack_4 = np.stack([
                il,
                self.clahe_mild.apply(il),
                self.clahe_strong.apply(il)
            ], axis=-1)

            # --- APPLY CLASS-SPECIFIC AUGMENTATION ---
            if self.augment:
                # Select the correct pipeline for this specific image's class
                pipeline = self.aug_pipelines[current_label_idx]

                # Apply to all 4 towers simultaneously to keep them geometrically aligned
                # Albumentations 'image' is primary, 'image2/3/4' are additional
                augmented = pipeline(
                    image=stack_1,
                    image2=stack_2,
                    image3=stack_3,
                    image4=stack_4
                )

                stack_1 = augmented['image']
                stack_2 = augmented['image2']
                stack_3 = augmented['image3']
                stack_4 = augmented['image4']

            s1.append(preprocess_input(stack_1.astype(np.float32)))
            s2.append(preprocess_input(stack_2.astype(np.float32)))
            s3.append(preprocess_input(stack_3.astype(np.float32)))
            s4.append(preprocess_input(stack_4.astype(np.float32)))

        inputs = (np.array(s1), np.array(s2), np.array(s3), np.array(s4), np.array(bm))

        # Targets must be a TUPLE (), not a List []
        targets = (by, by, by, by, by)

        return inputs, targets

# ==========================================
# 3. DEFINE THE QUAD MODEL
# ==========================================
def create_quad_model(num_classes=5):
    def build_tower(input_shape, name_suffix):
        inp = Input(shape=input_shape, name=f"in_{name_suffix}")
        base = tf.keras.applications.Xception(
            weights='imagenet', include_top=False, input_shape=input_shape,
            name=f"xception_{name_suffix}"
        )
        base.trainable = False

        x = base(inp)
        x = layers.GlobalAveragePooling2D()(x)

        # "Dense -> BN -> Swish -> Dropout" pattern.
        # This prevents "dead neurons" early in the network and stabilizes the features entering concatenation.
        # Tower Layer 1
        x = layers.Dense(1024, kernel_initializer='he_normal', use_bias=False)(x) # he_normal for Swish/ReLU stability
        x = layers.BatchNormalization()(x) # Normalize before activation (Pre-Activation)
        x = layers.Activation('swish')(x)  # Swish is superior to ReLU for deep networks
        x = layers.Dropout(0.5)(x)

        # Tower Layer 2
        # Compress to standard feature size
        x = layers.Dense(512, kernel_initializer='he_normal', use_bias=False)(x)
        x = layers.BatchNormalization()(x)
        x = layers.Activation('swish')(x)
        x = layers.Dropout(0.5)(x)

        # Attention Gate (Standard)
        # This allows the model to gate specific features instead of the whole image
        att = layers.Dense(512, activation='sigmoid', name=f"att_gate_{name_suffix}")(x)
        boost = layers.Multiply()([x, att])
        x_res = layers.Add(name=f"feat_{name_suffix}")([x, boost])

        aux_out = layers.Dense(num_classes, activation='softmax', name=f"aux_{name_suffix}")(x_res)
        return inp, x_res, aux_out

    # This function allows "Source" tower to look at "Target" tower and borrow features
    def apply_cross_attention(source_feat, target_feat, suffix):
        # 1. Compress target features (The "Key" / "Value")
        context = layers.Dense(128, activation='relu', name=f"ctx_ex_{suffix}")(target_feat)

        # 2. Compress source features (The "Query")
        query = layers.Dense(128, activation='relu', name=f"query_ex_{suffix}")(source_feat)

        # 3. Compute Relevance: Concatenate Query and Key -> Sigmoid Gate
        # (Using Concatenate is more stable than Dot product for batch vectors)
        concat_qk = layers.Concatenate()([query, context])
        attention_score = layers.Dense(512, activation='sigmoid', name=f"att_score_{suffix}")(concat_qk)

        # 4. Gated Context: Scale the target features by how relevant they are
        # Rescale context back to 512 to match source
        context_scaled = layers.Dense(512, activation='relu', kernel_initializer='zeros', name=f"ctx_up_{suffix}")(context)
        gated_context = layers.Multiply()([context_scaled, attention_score])

        return gated_context

    in1, feat1, aux1 = build_tower((299,299,3), "t1_global_gamma")
    in2, feat2, aux2 = build_tower((299,299,3), "t2_global_clahe")
    in3, feat3, aux3 = build_tower((299,299,3), "t3_local_sigmoid")
    in4, feat4, aux4 = build_tower((299,299,3), "t4_local_clahe")

    # TOWER 1 Enriched: T1 + Context from T2, T3, T4
    c1_2 = apply_cross_attention(feat1, feat2, "t1_from_t2")
    c1_3 = apply_cross_attention(feat1, feat3, "t1_from_t3")
    c1_4 = apply_cross_attention(feat1, feat4, "t1_from_t4")
    feat1_enriched = layers.Add(name="t1_enriched")([feat1, c1_2, c1_3, c1_4])

    # TOWER 2 Enriched: T2 + Context from T1, T3, T4
    c2_1 = apply_cross_attention(feat2, feat1, "t2_from_t1")
    c2_3 = apply_cross_attention(feat2, feat3, "t2_from_t3")
    c2_4 = apply_cross_attention(feat2, feat4, "t2_from_t4")
    feat2_enriched = layers.Add(name="t2_enriched")([feat2, c2_1, c2_3, c2_4])

    # TOWER 3 Enriched: T3 + Context from T1, T2, T4
    c3_1 = apply_cross_attention(feat3, feat1, "t3_from_t1")
    c3_2 = apply_cross_attention(feat3, feat2, "t3_from_t2")
    c3_4 = apply_cross_attention(feat3, feat4, "t3_from_t4")
    feat3_enriched = layers.Add(name="t3_enriched")([feat3, c3_1, c3_2, c3_4])

    # TOWER 4 Enriched: T4 + Context from T1, T2, T3
    c4_1 = apply_cross_attention(feat4, feat1, "t4_from_t1")
    c4_2 = apply_cross_attention(feat4, feat2, "t4_from_t2")
    c4_3 = apply_cross_attention(feat4, feat3, "t4_from_t3")
    feat4_enriched = layers.Add(name="t4_enriched")([feat4, c4_1, c4_2, c4_3])

    in_meta = Input(shape=(5,), name="in_meta")
    feat_meta = layers.Dense(64, activation='relu')(in_meta)

    # --- CONCATENATION ---
    # We concatenate the ENRICHED features, not the raw ones
    concat = layers.Concatenate()([feat1_enriched, feat2_enriched, feat3_enriched, feat4_enriched, feat_meta])

    # Stabilized SE Block
    se_filters = 2112 // 16

    # Reduction (Squeeze)
    se = layers.Dense(se_filters, kernel_initializer='he_normal', use_bias=False)(concat)
    se = layers.BatchNormalization()(se) # BN added here to stabilize reduction
    se = layers.Activation('swish')(se)

    # Expansion (Excitation)
    se = layers.Dense(2112, kernel_initializer='he_normal')(se)
    se = layers.Activation('sigmoid')(se) # Sigmoid must be last for gating

    fused_weighted = layers.Multiply()([concat, se])

    # Applied "Dense -> BN -> Swish -> Dropout" pattern
    # Layer 1
    z = layers.Dense(1024, kernel_initializer='he_normal', use_bias=False)(fused_weighted)
    z = layers.BatchNormalization()(z) # Standard placement
    z = layers.Activation('swish')(z)
    z = layers.Dropout(0.3)(z)

    # Layer 2
    z = layers.Dense(512, kernel_initializer='he_normal', use_bias=False)(z)
    z = layers.BatchNormalization()(z)
    z = layers.Activation('swish')(z)
    z = layers.Dropout(0.3)(z)

    final_out = layers.Dense(num_classes, activation='softmax', name="final_output")(z)

    model = Model(
        inputs=[in1, in2, in3, in4, in_meta],
        outputs=[final_out, aux1, aux2, aux3, aux4],
        name="Quad_Expert_System"
    )
    return model

# ==========================================
# 4. CUSTOM CALLBACK FOR SAFE SAVING
# ==========================================
class ContinuousHistorySaver(tf.keras.callbacks.Callback):
    def __init__(self, filepath, previous_history=None):
        super().__init__()
        self.filepath = filepath
        # If we have old history, start with that. Otherwise, empty dict.
        self.history_storage = previous_history if previous_history else {}

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        # Update storage with the latest metrics from this epoch
        for k, v in logs.items():
            if k not in self.history_storage:
                self.history_storage[k] = []
            self.history_storage[k].append(v)

        # Save immediately to Drive
        try:
            with open(self.filepath, 'wb') as f:
                pickle.dump(self.history_storage, f)
        except Exception as e:
            print(f"\n Warning: Could not save history for epoch {epoch}: {e}")

# ==========================================
# 5. TRAINING LOOP
# ==========================================
def run_phase_1_quad(save_dir):
    CHECKPOINT_DIR = os.path.join(save_dir, 'checkpoints_v1.0')
    os.makedirs(CHECKPOINT_DIR, exist_ok=True)
    print(f" Saving checkpoints to: {CHECKPOINT_DIR}")

    # Since the Generator now guarantees a balanced stream (1:1:1:1:1),
    # We set the loss weights to be equal [1.0] for all classes.
    alpha_balanced = np.array([1.0, 1.0, 1.0, 1.0, 1.0], dtype=float)
    print("\n Balanced Alphas:", alpha_balanced)

    # --- FRESH METRIC INSTANCES ---
    # We create a function to generate FRESH metrics for each head
    def get_metrics(name_prefix):
        return [
            'accuracy',
            tf.keras.metrics.AUC(name=f'{name_prefix}_auc'),
            tf.keras.metrics.F1Score(average='macro', name=f'{name_prefix}_f1')
        ]

    # oversample_ratio is technically not used in the new calculation formula,
    # But we pass > 0 to trigger the "balanced" sampling mode logic.
    # 2. Pass Weights to Generator
    train_gen = QuadStreamGenerator(
        X_train_global, X_train_local, X_train_metadata, y_train,
        batch_size=12, augment=True,
        oversample_ratio=1 # Triggers the balanced sampling
    )

    val_gen = QuadStreamGenerator(
        X_val_global, X_val_local, X_val_metadata, y_val,
        batch_size=12, augment=False,
        oversample_ratio=0.0 # Validation must remain natural distribution
    )

    # Resume Logic
    initial_epoch = 0
    model = None
    existing_checkpoints = glob.glob(os.path.join(CHECKPOINT_DIR, "epoch_*.keras"))

    if existing_checkpoints:
        def get_epoch_num(filepath):
            match = re.search(r"epoch_(\d+).keras", filepath)
            return int(match.group(1)) if match else -1
        latest_ckpt = max(existing_checkpoints, key=get_epoch_num)
        print(f" Found checkpoint! Resuming from: {latest_ckpt}")
        try:
            model = tf.keras.models.load_model(
                latest_ckpt,
                # Ensure Gamma=2.5 is passed when loading custom object
                custom_objects={'focal_loss_fixed': class_weighted_focal_loss(gamma=2.5, alpha_weights=alpha_balanced)}
            )
            found_epoch = get_epoch_num(latest_ckpt)
            initial_epoch = found_epoch
            print(f" Resuming training at Epoch {initial_epoch + 1}")
        except Exception as e:
            print(f" Error loading checkpoint {e}. Starting fresh.")
            model = None

    if model is None:
        print("✨ Starting training from scratch...")
        model = create_quad_model()

        # We pass the alpha_final list computed above
        loss_fn = class_weighted_focal_loss(gamma=2.5, alpha_weights=alpha_balanced)
        loss_list = [loss_fn for _ in range(5)]
        loss_weights_list = [1.0, 0.1, 0.1, 0.1, 0.1]

        # Apply specific metrics to specific outputs using a Dictionary
        # This is much safer than a list of lists
        metrics_dict = {
            'final_output': get_metrics('final'),
            'aux_t1_global_gamma': get_metrics('aux1'),
            'aux_t2_global_clahe': get_metrics('aux2'),
            'aux_t3_local_sigmoid': get_metrics('aux3'),
            'aux_t4_local_clahe': get_metrics('aux4')
        }

        model.compile(
            optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
            loss=loss_list,
            loss_weights=loss_weights_list,
            metrics=metrics_dict,
            run_eagerly=False
        )
    else:
        print(" Optimizer state restored from checkpoint (Compilation skipped).")

    # --- 5. HISTORY RETRIEVAL ---
    # We load history NOW to find the previous best score
    history_path = os.path.join(save_dir, 'phase1_v1.0_history.pkl')
    previous_best_f1 = -np.inf
    old_history = {}

    if os.path.exists(history_path):
        print(" Found existing history file. Scanning for previous best F1...")
        try:
            with open(history_path, 'rb') as f:
                old_history = pickle.load(f)

            target_metric = 'val_final_output_final_f1'
            if target_metric in old_history and len(old_history[target_metric]) > 0:
                previous_best_f1 = max(old_history[target_metric])
                print(f" Previous Best Val F1 was: {previous_best_f1:.4f}")
            else:
                print(" Metric not found in history. Assuming -inf.")
        except Exception as e:
            print(f" Could not read history file: {e}")

    # --- 6. DEFINE CALLBACKS WITH MEMORY ---
    # Manually create the checkpoint callback so we can modify it
    best_ckpt = tf.keras.callbacks.ModelCheckpoint(
        filepath=os.path.join(save_dir, 'phase1_v1.0_best.keras'),
        save_best_only=True,
        monitor='val_final_output_final_f1',
        verbose=1,
        mode='max'
    )

    # INJECT THE MEMORY
    # If we found a previous best, we force the callback to respect it.
    if previous_best_f1 > -np.inf:
        best_ckpt.best = previous_best_f1
        print(f" Injected previous best F1 ({previous_best_f1:.4f}) into ModelCheckpoint.")

    # We pass 'old_history' so it continues appending to it, rather than starting fresh
    history_saver = ContinuousHistorySaver(filepath=history_path, previous_history=old_history)

    callbacks = [
        tf.keras.callbacks.EarlyStopping(
            monitor='val_final_output_final_f1',
            patience=6,
            restore_best_weights=True,
            mode='max'
        ),

        tf.keras.callbacks.ReduceLROnPlateau(
            monitor='val_final_output_final_f1',
            factor=0.2,
            patience=2,
            min_lr=1e-6,
            mode='max'
        ),

        tf.keras.callbacks.ModelCheckpoint(
            filepath=os.path.join(CHECKPOINT_DIR, "epoch_{epoch:02d}.keras"),
            save_best_only=False, verbose=1
        ),
        #The custom one we created above with memory
        best_ckpt,
        history_saver # Saves history.pkl every epoch
    ]

    print(f" Running Phase 1 v1.0 (Quad-Expert)...")

    # 3. Fit without class_weight arg
    history = model.fit(
        train_gen,
        validation_data=val_gen,
        initial_epoch=initial_epoch,
        epochs=30, # Increased epochs slightly as oversampling changes epoch dynamics
        callbacks=callbacks
    )

    print(f"\n Saving final Model to: {save_dir}")
    model.save(os.path.join(save_dir, 'phase1_v1.0_final.keras'))
    new_history = history.history

    if os.path.exists(history_path):
        print("found existing history, merging...")
        with open(history_path, 'rb') as f:
            old_history = pickle.load(f)

        # IMPROVED MERGE LOGIC:
        # Loop through NEW history to ensure we capture everything
        for key, val in new_history.items():
            if key in old_history:
                # Append new epochs to existing list
                old_history[key].extend(val)
            else:
                # If a new metric appeared (rare, but possible), add it
                old_history[key] = val

        final_history = old_history
    else:
        final_history = new_history

    # Save the combined history
    with open(history_path, 'wb') as f:
        pickle.dump(final_history, f)

    return model, history

# RUN
MY_DRIVE_PATH = '/content/drive/My Drive/Xception model/v1.0'
os.makedirs(MY_DRIVE_PATH, exist_ok=True)

model, history = run_phase_1_quad(save_dir=MY_DRIVE_PATH)

#####Phase 2 training (Unfreeze block 14 for fine-tuning)

In [ ]:
import os
import glob
import re
import pickle
import numpy as np
import cv2
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.utils import Sequence
from tensorflow.keras.applications.xception import preprocess_input
from tensorflow.keras import layers, Model, Input
import albumentations as A

# ==========================================
# 1. RE-DEFINE CUSTOM OBJECTS (REQUIRED FOR LOADING)
# ==========================================
def class_weighted_focal_loss(gamma=2.5, alpha_weights=None):
    """
    gamma: Reduced to 2.5. Gamma 3.0 was too aggressive and suppressed gradients
           for "moderately hard" samples (like Luminal A), hurting their learning.
    """
    # Convert alpha to a tensor constant for efficiency
    if alpha_weights is not None:
        alpha_tensor = tf.constant(alpha_weights, dtype=tf.float32)
    else:
        alpha_tensor = None

    def focal_loss_fixed(y_true, y_pred):
        epsilon = tf.keras.backend.epsilon()
        y_pred = tf.clip_by_value(y_pred, epsilon, 1. - epsilon)

        # Step 1: Get probability of TRUE class only
        # y_true: (batch, 5), y_pred: (batch, 5)
        y_true = tf.cast(y_true, tf.float32)

        # p_t = probability assigned to the true class for each sample
        # Shape: (batch_size,)
        pt = tf.reduce_sum(y_true * y_pred, axis=-1)  # sum(y_true_i * y_pred_i) = p_true_class

        # Step 2: Basic cross entropy for true class
        ce = -tf.math.log(pt + epsilon)  # Shape: (batch_size,)

        # Step 3: Focal weighting based on how confident we are in TRUE class
        focal_weight = tf.math.pow(1 - pt, gamma)  # Shape: (batch_size,)

        # 3. Calculate Class Weight Term
        # Multiply by alpha corresponding to the true class
        if alpha_tensor is not None:
            # Get alpha for the true class of each sample
            # y_true * alpha gives (batch, 5) with alpha only at true class position
            # Then sum to get shape (batch_size,)
            alpha_t = tf.reduce_sum(y_true * alpha_tensor, axis=-1)
            loss = alpha_t * focal_weight * ce
        else:
            loss = focal_weight * ce

        return tf.reduce_mean(loss)  # Mean over batch

    return focal_loss_fixed

# ==========================================
# 2. RE-DEFINE GENERATOR (REQUIRED FOR DATA)
# ==========================================
class QuadStreamGenerator(Sequence):
    def __init__(self, x_global, x_local, x_meta, y, batch_size,
                 augment=False, shuffle=True,
                 oversample_ratio=0.0):
        super().__init__()
        self.x_global = x_global
        self.x_local = x_local
        self.x_meta = x_meta
        self.y = y
        self.batch_size = batch_size
        self.augment = augment
        self.shuffle = shuffle

        # Oversampling Setup
        self.oversample_ratio = oversample_ratio
        self.indexes = np.arange(len(self.x_global))

        # Calculate Sampling Probabilities if oversampling is enabled
        self.sampling_probs = None
        if self.oversample_ratio > 0:
            self.sampling_probs = self._calculate_sampling_probs()

        # --- ENHANCEMENT OBJECTS ---
        self.clahe_mild = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
        self.clahe_strong = cv2.createCLAHE(clipLimit=4.0, tileGridSize=(8,8))
        self.gamma_mild_table = self._build_gamma_table(0.6)
        self.gamma_strong_table = self._build_gamma_table(1.5)

        # --- SPECIALIZED AUGMENTATION PIPELINES ---
        # Using Albumentations to treat each class biologically correctly
        if self.augment:
            self.aug_pipelines = self._build_augmentation_pipelines()

        self.on_epoch_end()

    def _build_augmentation_pipelines(self):
        # 1. Base Geometric (Safe for all)
        # We use additional_targets to apply SAME transform to all 4 towers
        base_aug = [
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.5),
            A.Affine(scale=(0.9, 1.1), translate_percent=(0.1, 0.1), rotate=(-20, 20), p=0.5),
        ]

        # 2. Define Class-Specific Pipelines
        pipelines = {}

        # --- Class 0: BENIGN ---
        pipelines[0] = A.Compose(base_aug + [
            A.MedianBlur(blur_limit=5, p=0.3),
        ], additional_targets={'image2': 'image', 'image3': 'image', 'image4': 'image'})

        # --- Class 1: LUMINAL A ---
        pipelines[1] = A.Compose(base_aug + [
             A.ISONoise(color_shift=(0.01, 0.05), intensity=(0.1, 0.5), p=0.2),
        ], additional_targets={'image2': 'image', 'image3': 'image', 'image4': 'image'})

        # --- Class 2: LUMINAL B (TARGET) ---
        pipelines[2] = A.Compose(base_aug + [
            # 1. Geometry: Create the spicule shape physically
            A.ElasticTransform(alpha=1, sigma=50, p=0.5),

            # 2. Structure: Highlight the edges (MILD settings)
            A.Emboss(alpha=(0.2, 0.5), strength=(0.3, 0.5), p=0.4),
        ], additional_targets={'image2': 'image', 'image3': 'image', 'image4': 'image'})

        # --- Class 3: HER2+ ---
        pipelines[3] = A.Compose(base_aug + [
            A.Sharpen(alpha=(0.2, 0.5), lightness=(0.5, 1.0), p=0.5),
        ], additional_targets={'image2': 'image', 'image3': 'image', 'image4': 'image'})

        # --- Class 4: TN (TARGET) ---
        pipelines[4] = A.Compose(base_aug + [
            # 1. Geometry: Create the "Bulge" / Mass Effect
            A.GridDistortion(num_steps=5, distort_limit=0.3, p=0.5),

            # 2. Structure: Define the sharp margin (MILD)
            A.Sharpen(alpha=(0.1, 0.3), lightness=(0.5, 1.0), p=0.4),

            # 3. Density: High Density (Dark/Bright shift)
            # Safe Range: (85, 115)
            A.RandomGamma(gamma_limit=(85, 115), p=0.5),
        ], additional_targets={'image2': 'image', 'image3': 'image', 'image4': 'image'})

        return pipelines

    def _calculate_sampling_probs(self):
        # 1. Get integer labels
        y_int = np.argmax(self.y, axis=1)
        classes, counts = np.unique(y_int, return_counts=True)
        # Calculate weight using Inverse Frequency (1.0 / count)
        # This ensures every class has roughly EQUAL probability of being picked per batch.
        class_weight_map = {c: 1.0/count for c, count in zip(classes, counts)}

        sample_weights = np.array([class_weight_map[label] for label in y_int])

        # Normalize to sum to 1.0 for np.random.choice
        return sample_weights / sample_weights.sum()

    def _build_gamma_table(self, gamma):
        invGamma = 1.0 / gamma
        return np.array([((i / 255.0) ** invGamma) * 255 for i in np.arange(0, 256)]).astype("uint8")

    def apply_gamma(self, img, table):
        return cv2.LUT(img, table)

    def apply_sigmoid(self, img, gain, cutoff):
        img_f = img.astype(np.float32) / 255.0
        sigmoid = 1.0 / (1.0 + np.exp(-gain * (img_f - cutoff)))
        return (sigmoid * 255).astype(np.uint8)

    def __len__(self):
        return int(np.ceil(len(self.x_global) / self.batch_size))

    def on_epoch_end(self):
        # Logic to handle Weighted Random Sampling
        if self.sampling_probs is not None and self.shuffle:
            # Sample WITH REPLACEMENT based on calculated probabilities
            # This creates a new list of indexes where minority classes appear more often
            self.indexes = np.random.choice(
                np.arange(len(self.x_global)),
                size=len(self.x_global),
                p=self.sampling_probs,
                replace=True
            )
        elif self.shuffle:
            np.random.shuffle(self.indexes)

    def __getitem__(self, index):
        idxs = self.indexes[index*self.batch_size:(index+1)*self.batch_size]

        s1, s2, s3, s4 = [], [], [], []
        bm = self.x_meta[idxs]
        by = self.y[idxs]

        for i in range(len(idxs)):
            # Identify Class for Specialization
            current_label_idx = np.argmax(by[i])

            # Load raw images
            ig_raw = self.x_global[idxs[i]]
            il_raw = self.x_local[idxs[i]]

            if ig_raw.ndim == 3: ig = ig_raw[:,:,0].astype(np.uint8)
            else: ig = ig_raw.astype(np.uint8)

            if il_raw.ndim == 3: il = il_raw[:,:,0].astype(np.uint8)
            else: il = il_raw.astype(np.uint8)

            # --- PREPARE STACKS ---
            # Tower 1: Global Gamma
            stack_1 = np.stack([
                ig,
                self.apply_gamma(ig, self.gamma_mild_table),
                self.apply_gamma(ig, self.gamma_strong_table)
            ], axis=-1)

            # Tower 2: Global CLAHE
            stack_2 = np.stack([
                ig,
                self.clahe_mild.apply(ig),
                self.clahe_strong.apply(ig)
            ], axis=-1)

            # Tower 3: Local Sigmoid
            stack_3 = np.stack([
                il,
                self.apply_sigmoid(il, gain=8, cutoff=0.75),
                self.apply_sigmoid(il, gain=15, cutoff=0.60)
            ], axis=-1)

            # Tower 4: Local CLAHE
            stack_4 = np.stack([
                il,
                self.clahe_mild.apply(il),
                self.clahe_strong.apply(il)
            ], axis=-1)

            # --- APPLY CLASS-SPECIFIC AUGMENTATION ---
            if self.augment:
                # Select the correct pipeline for this specific image's class
                pipeline = self.aug_pipelines[current_label_idx]

                # Apply to all 4 towers simultaneously to keep them geometrically aligned
                # Albumentations 'image' is primary, 'image2/3/4' are additional
                augmented = pipeline(
                    image=stack_1,
                    image2=stack_2,
                    image3=stack_3,
                    image4=stack_4
                )

                stack_1 = augmented['image']
                stack_2 = augmented['image2']
                stack_3 = augmented['image3']
                stack_4 = augmented['image4']

            s1.append(preprocess_input(stack_1.astype(np.float32)))
            s2.append(preprocess_input(stack_2.astype(np.float32)))
            s3.append(preprocess_input(stack_3.astype(np.float32)))
            s4.append(preprocess_input(stack_4.astype(np.float32)))

        inputs = (np.array(s1), np.array(s2), np.array(s3), np.array(s4), np.array(bm))

        # Targets must be a TUPLE (), not a List []
        targets = (by, by, by, by, by)

        return inputs, targets

# ==========================================
# 3. HELPER: UNFREEZE LOGIC
# ==========================================
def unfreeze_xception_blocks(model, start_block=10):
    """
    Traverses the 4 Xception towers in the Quad model and unfreezes
    blocks from 'start_block' up to 14.
    """
    print(f"\n Unfreezing blocks {start_block}-14 in all Xception towers...")

    # We define the regex to match the block names
    target_blocks = [f"block{i}_" for i in range(start_block, 15)]

    unfrozen_count = 0

    # Iterate through the main model layers to find the Xception Bases
    for layer in model.layers:
        # Our Xception bases are named "xception_t1...", "xception_t2..."
        if "xception" in layer.name and isinstance(layer, Model):
            print(f"   -> Processing Tower: {layer.name}")

            # Set the Container Model (the Xception Base) to trainable
            layer.trainable = True

            # Now traverse the layers INSIDE the Xception Base
            for inner_layer in layer.layers:
                # Check if layer name matches our target blocks (10, 11, 12, 13, 14)
                should_unfreeze = any(block in inner_layer.name for block in target_blocks)

                if should_unfreeze:
                    # CRITICAL: KEEP BATCH NORM FROZEN
                    if isinstance(inner_layer, layers.BatchNormalization):
                        inner_layer.trainable = False
                    else:
                        inner_layer.trainable = True
                        unfrozen_count += 1
                else:
                    # Keep everything else (block 1-9) frozen
                    inner_layer.trainable = False

    print(f" Total layers unfrozen across all towers: {unfrozen_count}")
    return model

# ==========================================
# 4. CUSTOM CALLBACK FOR SAFE SAVING
# ==========================================
class ContinuousHistorySaver(tf.keras.callbacks.Callback):
    def __init__(self, filepath, previous_history=None):
        super().__init__()
        self.filepath = filepath
        # If we have old history, start with that. Otherwise, empty dict.
        self.history_storage = previous_history if previous_history else {}

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        # Update storage with the latest metrics from this epoch
        for k, v in logs.items():
            if k not in self.history_storage:
                self.history_storage[k] = []
            self.history_storage[k].append(v)

        # Save immediately to Drive
        try:
            with open(self.filepath, 'wb') as f:
                pickle.dump(self.history_storage, f)
        except Exception as e:
            print(f"\n Warning: Could not save history for epoch {epoch}: {e}")

# ==========================================
# 5. PHASE 2 TRAINING LOOP
# ==========================================
def run_phase_2_finetune(save_dir, resume_from_dir):
    # Setup directories
    CHECKPOINT_DIR = os.path.join(save_dir, 'checkpoints_v1.0_phase2')
    os.makedirs(CHECKPOINT_DIR, exist_ok=True)
    print(f" Saving Phase 2 checkpoints to: {CHECKPOINT_DIR}")

    # Setup Alphas & Metrics (Same as before)
    alpha_balanced = np.array([1.0, 1.0, 1.0, 1.0, 1.0], dtype=float)
    def get_metrics(name_prefix):
        return [
            'accuracy',
            tf.keras.metrics.AUC(name=f'{name_prefix}_auc'),
            tf.keras.metrics.F1Score(average='macro', name=f'{name_prefix}_f1')
        ]

    # --- LOAD DATA ---
    print(" Initializing Generators...")
    train_gen = QuadStreamGenerator(
        X_train_global, X_train_local, X_train_metadata, y_train,
        batch_size=12, augment=True, oversample_ratio=1
    )
    val_gen = QuadStreamGenerator(
        X_val_global, X_val_local, X_val_metadata, y_val,
        batch_size=12, augment=False, oversample_ratio=0.0
    )

    # --- SMART RESUMPTION LOGIC ---
    initial_epoch = 0
    model = None

    # 1. Check if we have interrupted Phase 2 checkpoints
    phase2_checkpoints = glob.glob(os.path.join(CHECKPOINT_DIR, "p2_epoch_*.keras"))

    if phase2_checkpoints:
        # Helper to find the highest epoch number
        def get_epoch_num(filepath):
            match = re.search(r"p2_epoch_(\d+).keras", filepath)
            return int(match.group(1)) if match else -1

        latest_ckpt = max(phase2_checkpoints, key=get_epoch_num)
        print(f" Found interrupted Phase 2 training! Resuming from: {latest_ckpt}")

        try:
            # Load the checkpoint.
            # This restores: Weights, Optimizer State, and "Unfrozen" status.
            model = tf.keras.models.load_model(
                latest_ckpt,
                custom_objects={'focal_loss_fixed': class_weighted_focal_loss(gamma=2.5, alpha_weights=alpha_balanced)}
            )
            found_epoch = get_epoch_num(latest_ckpt)
            initial_epoch = found_epoch
            print(f" Jumping ahead to Epoch {initial_epoch + 1}")
        except Exception as e:
            print(f" Error loading checkpoint {e}. Falling back to start.")
            model = None

    # 2. If no Phase 2 checkpoint found (or error), Start Fresh from Phase 1 Model
    if model is None:
        print(" No Phase 2 progress found. Starting Fine-Tuning from scratch...")

        phase1_path = os.path.join(resume_from_dir, 'phase1_v1.0_final.keras')
        if not os.path.exists(phase1_path):
            phase1_path = os.path.join(resume_from_dir, 'phase1_v1.0_best.keras')

        if not os.path.exists(phase1_path):
            raise FileNotFoundError(" Could not find Phase 1 model to fine-tune!")

        print(f" Loading Phase 1 Model from: {phase1_path}")
        model = tf.keras.models.load_model(
            phase1_path,
            custom_objects={'focal_loss_fixed': class_weighted_focal_loss(gamma=2.5, alpha_weights=alpha_balanced)}
        )

        # --- UNFREEZE & COMPILE (Only needed for fresh start) ---
        model = unfreeze_xception_blocks(model, start_block=14)

        fine_tune_lr = 1e-6
        print(f"⚙️ Recompiling with Fine-Tuning LR: {fine_tune_lr}")

        loss_fn = class_weighted_focal_loss(gamma=2.5, alpha_weights=alpha_balanced)
        metrics_dict = {
            'final_output': get_metrics('final'),
            'aux_t1_global_gamma': get_metrics('aux1'),
            'aux_t2_global_clahe': get_metrics('aux2'),
            'aux_t3_local_sigmoid': get_metrics('aux3'),
            'aux_t4_local_clahe': get_metrics('aux4')
        }

        model.compile(
            optimizer=tf.keras.optimizers.Adam(learning_rate=fine_tune_lr),
            loss=[loss_fn] * 5,
            loss_weights=[1.0, 0.1, 0.1, 0.1, 0.1],
            metrics=metrics_dict
        )
    else:
        print(" Optimizer state restored from checkpoint (Compilation skipped).")

    # --- HISTORY HANDLING ---
    phase1_hist_path = os.path.join(resume_from_dir, 'phase1_v1.0_history.pkl')
    phase2_hist_path = os.path.join(save_dir, 'phase2_v1.0_history.pkl')

    previous_best_f1 = -np.inf
    if os.path.exists(phase1_hist_path):
        with open(phase1_hist_path, 'rb') as f:
            p1_hist = pickle.load(f)
            if 'val_final_output_final_f1' in p1_hist:
                previous_best_f1 = max(p1_hist['val_final_output_final_f1'])

    # Load existing Phase 2 history if available
    current_p2_history = {}
    if os.path.exists(phase2_hist_path):
        with open(phase2_hist_path, 'rb') as f:
            current_p2_history = pickle.load(f)

    # --- CALLBACKS ---
    best_ckpt = tf.keras.callbacks.ModelCheckpoint(
        filepath=os.path.join(save_dir, 'phase2_v1.0_best.keras'),
        save_best_only=True,
        monitor='val_final_output_final_f1',
        verbose=1,
        mode='max'
    )
    if previous_best_f1 > -np.inf:
        best_ckpt.best = previous_best_f1

    history_saver = ContinuousHistorySaver(filepath=phase2_hist_path, previous_history=current_p2_history)

    callbacks = [
        tf.keras.callbacks.EarlyStopping(monitor='val_final_output_final_f1', patience=8, restore_best_weights=True, mode='max'),
        tf.keras.callbacks.ReduceLROnPlateau(monitor='val_final_output_final_f1', factor=0.2, patience=3, min_lr=1e-7, mode='max'),
        tf.keras.callbacks.ModelCheckpoint(filepath=os.path.join(CHECKPOINT_DIR, "p2_epoch_{epoch:02d}.keras"), save_best_only=False, verbose=1),
        best_ckpt,
        history_saver
    ]

    print(f" Running Phase 2 (Fine-Tuning) starting from Epoch {initial_epoch + 1}...")

    history = model.fit(
        train_gen,
        validation_data=val_gen,
        initial_epoch=initial_epoch, # Tells Keras where to start counting
        epochs=20,
        callbacks=callbacks
    )

    print(f"\n Saving Final Phase 2 Model to: {save_dir}")
    model.save(os.path.join(save_dir, 'phase2_v1.0_final.keras'))

    return model, history

# ==========================================
# 6. RUN EXECUTION
# ==========================================
MY_DRIVE_PATH = '/content/drive/My Drive/Xception model/v1.0'

model_p2, history_p2 = run_phase_2_finetune(save_dir=MY_DRIVE_PATH, resume_from_dir=MY_DRIVE_PATH)

#IX. DenseNet121 model

##Training model v1.0

###Phase 1 training

In [ ]:
import os
import glob
import re
import pickle
import numpy as np
import cv2
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.utils import Sequence
from tensorflow.keras.applications.densenet import DenseNet121, preprocess_input ##Import DenseNest121
from tensorflow.keras import layers, Model, Input
import albumentations as A

# ==========================================
# 1. CUSTOM LOSS FUNCTION (CLASS-WEIGHTED FOCAL LOSS)
# ==========================================
def class_weighted_focal_loss(gamma=2.5, alpha_weights=None):
    """
    gamma: Reduced to 2.5. Gamma 3.0 was too aggressive and suppressed gradients
           for "moderately hard" samples (like Luminal A), hurting their learning.
    """
    # Convert alpha to a tensor constant for efficiency
    if alpha_weights is not None:
        alpha_tensor = tf.constant(alpha_weights, dtype=tf.float32)
    else:
        alpha_tensor = None

    def focal_loss_fixed(y_true, y_pred):
        epsilon = tf.keras.backend.epsilon()
        y_pred = tf.clip_by_value(y_pred, epsilon, 1. - epsilon)

        # Step 1: Get probability of TRUE class only
        # y_true: (batch, 5), y_pred: (batch, 5)
        y_true = tf.cast(y_true, tf.float32)

        # p_t = probability assigned to the true class for each sample
        # Shape: (batch_size,)
        pt = tf.reduce_sum(y_true * y_pred, axis=-1)  # sum(y_true_i * y_pred_i) = p_true_class

        # Step 2: Basic cross entropy for true class
        ce = -tf.math.log(pt + epsilon)  # Shape: (batch_size,)

        # Step 3: Focal weighting based on how confident we are in TRUE class
        focal_weight = tf.math.pow(1 - pt, gamma)  # Shape: (batch_size,)

        # 3. Calculate Class Weight Term
        # Multiply by alpha corresponding to the true class
        if alpha_tensor is not None:
            # Get alpha for the true class of each sample
            # y_true * alpha gives (batch, 5) with alpha only at true class position
            # Then sum to get shape (batch_size,)
            alpha_t = tf.reduce_sum(y_true * alpha_tensor, axis=-1)
            loss = alpha_t * focal_weight * ce
        else:
            loss = focal_weight * ce

        return tf.reduce_mean(loss)  # Mean over batch

    return focal_loss_fixed

# ==========================================
# 2. QUAD-STREAM GENERATOR
# ==========================================
class QuadStreamGenerator(Sequence):
    def __init__(self, x_global, x_local, x_meta, y, batch_size,
                 augment=False, shuffle=True,
                 oversample_ratio=0.0):
        super().__init__()
        self.x_global = x_global
        self.x_local = x_local
        self.x_meta = x_meta
        self.y = y
        self.batch_size = batch_size
        self.augment = augment
        self.shuffle = shuffle

        # Oversampling Setup
        self.oversample_ratio = oversample_ratio
        self.indexes = np.arange(len(self.x_global))

        # Calculate Sampling Probabilities if oversampling is enabled
        self.sampling_probs = None
        if self.oversample_ratio > 0:
            self.sampling_probs = self._calculate_sampling_probs()

        # --- ENHANCEMENT OBJECTS ---
        self.clahe_mild = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
        self.clahe_strong = cv2.createCLAHE(clipLimit=4.0, tileGridSize=(8,8))
        self.gamma_mild_table = self._build_gamma_table(0.6)
        self.gamma_strong_table = self._build_gamma_table(1.5)

        # --- SPECIALIZED AUGMENTATION PIPELINES ---
        # Using Albumentations to treat each class biologically correctly
        if self.augment:
            self.aug_pipelines = self._build_augmentation_pipelines()

        self.on_epoch_end()

    def _build_augmentation_pipelines(self):
        # 1. Base Geometric (Safe for all)
        # We use additional_targets to apply SAME transform to all 4 towers
        base_aug = [
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.5),
            A.Affine(scale=(0.9, 1.1), translate_percent=(0.1, 0.1), rotate=(-20, 20), p=0.5),
        ]

        # 2. Define Class-Specific Pipelines
        pipelines = {}

        # --- Class 0: BENIGN ---
        pipelines[0] = A.Compose(base_aug + [
            A.MedianBlur(blur_limit=5, p=0.3),
        ], additional_targets={'image2': 'image', 'image3': 'image', 'image4': 'image'})

        # --- Class 1: LUMINAL A ---
        pipelines[1] = A.Compose(base_aug + [
             A.ISONoise(color_shift=(0.01, 0.05), intensity=(0.1, 0.5), p=0.2),
        ], additional_targets={'image2': 'image', 'image3': 'image', 'image4': 'image'})

        # --- Class 2: LUMINAL B (TARGET) ---
        pipelines[2] = A.Compose(base_aug + [
            # 1. Geometry: Create the spicule shape physically
            A.ElasticTransform(alpha=1, sigma=50, p=0.5),

            # 2. Structure: Highlight the edges (MILD settings)
            A.Emboss(alpha=(0.2, 0.5), strength=(0.3, 0.5), p=0.4),
        ], additional_targets={'image2': 'image', 'image3': 'image', 'image4': 'image'})

        # --- Class 3: HER2+ ---
        pipelines[3] = A.Compose(base_aug + [
            A.Sharpen(alpha=(0.2, 0.5), lightness=(0.5, 1.0), p=0.5),
        ], additional_targets={'image2': 'image', 'image3': 'image', 'image4': 'image'})

        # --- Class 4: TN (TARGET) ---
        pipelines[4] = A.Compose(base_aug + [
            # 1. Geometry: Create the "Bulge" / Mass Effect
            A.GridDistortion(num_steps=5, distort_limit=0.3, p=0.5),

            # 2. Structure: Define the sharp margin (MILD)
            A.Sharpen(alpha=(0.1, 0.3), lightness=(0.5, 1.0), p=0.4),

            # 3. Density: High Density (Dark/Bright shift)
            # Safe Range: (85, 115)
            A.RandomGamma(gamma_limit=(85, 115), p=0.5),
        ], additional_targets={'image2': 'image', 'image3': 'image', 'image4': 'image'})

        return pipelines

    def _calculate_sampling_probs(self):
        # 1. Get integer labels
        y_int = np.argmax(self.y, axis=1)
        classes, counts = np.unique(y_int, return_counts=True)
        # Calculate weight using Inverse Frequency (1.0 / count)
        # This ensures every class has roughly EQUAL probability of being picked per batch.
        class_weight_map = {c: 1.0/count for c, count in zip(classes, counts)}

        sample_weights = np.array([class_weight_map[label] for label in y_int])

        # Normalize to sum to 1.0 for np.random.choice
        return sample_weights / sample_weights.sum()

    def _build_gamma_table(self, gamma):
        invGamma = 1.0 / gamma
        return np.array([((i / 255.0) ** invGamma) * 255 for i in np.arange(0, 256)]).astype("uint8")

    def apply_gamma(self, img, table):
        return cv2.LUT(img, table)

    def apply_sigmoid(self, img, gain, cutoff):
        img_f = img.astype(np.float32) / 255.0
        sigmoid = 1.0 / (1.0 + np.exp(-gain * (img_f - cutoff)))
        return (sigmoid * 255).astype(np.uint8)

    def __len__(self):
        return int(np.ceil(len(self.x_global) / self.batch_size))

    def on_epoch_end(self):
        # Logic to handle Weighted Random Sampling
        if self.sampling_probs is not None and self.shuffle:
            # Sample WITH REPLACEMENT based on calculated probabilities
            # This creates a new list of indexes where minority classes appear more often
            self.indexes = np.random.choice(
                np.arange(len(self.x_global)),
                size=len(self.x_global),
                p=self.sampling_probs,
                replace=True
            )
        elif self.shuffle:
            np.random.shuffle(self.indexes)

    def __getitem__(self, index):
        idxs = self.indexes[index*self.batch_size:(index+1)*self.batch_size]

        s1, s2, s3, s4 = [], [], [], []
        bm = self.x_meta[idxs]
        by = self.y[idxs]

        for i in range(len(idxs)):
            # Identify Class for Specialization
            current_label_idx = np.argmax(by[i])

            # Load raw images
            ig_raw = self.x_global[idxs[i]]
            il_raw = self.x_local[idxs[i]]

            if ig_raw.ndim == 3: ig = ig_raw[:,:,0].astype(np.uint8)
            else: ig = ig_raw.astype(np.uint8)

            if il_raw.ndim == 3: il = il_raw[:,:,0].astype(np.uint8)
            else: il = il_raw.astype(np.uint8)

            # --- PREPARE STACKS ---
            # Tower 1: Global Gamma
            stack_1 = np.stack([
                ig,
                self.apply_gamma(ig, self.gamma_mild_table),
                self.apply_gamma(ig, self.gamma_strong_table)
            ], axis=-1)

            # Tower 2: Global CLAHE
            stack_2 = np.stack([
                ig,
                self.clahe_mild.apply(ig),
                self.clahe_strong.apply(ig)
            ], axis=-1)

            # Tower 3: Local Sigmoid
            stack_3 = np.stack([
                il,
                self.apply_sigmoid(il, gain=8, cutoff=0.75),
                self.apply_sigmoid(il, gain=15, cutoff=0.60)
            ], axis=-1)

            # Tower 4: Local CLAHE
            stack_4 = np.stack([
                il,
                self.clahe_mild.apply(il),
                self.clahe_strong.apply(il)
            ], axis=-1)

            # --- APPLY CLASS-SPECIFIC AUGMENTATION ---
            if self.augment:
                # Select the correct pipeline for this specific image's class
                pipeline = self.aug_pipelines[current_label_idx]

                # Apply to all 4 towers simultaneously to keep them geometrically aligned
                # Albumentations 'image' is primary, 'image2/3/4' are additional
                augmented = pipeline(
                    image=stack_1,
                    image2=stack_2,
                    image3=stack_3,
                    image4=stack_4
                )

                stack_1 = augmented['image']
                stack_2 = augmented['image2']
                stack_3 = augmented['image3']
                stack_4 = augmented['image4']

            s1.append(preprocess_input(stack_1.astype(np.float32)))
            s2.append(preprocess_input(stack_2.astype(np.float32)))
            s3.append(preprocess_input(stack_3.astype(np.float32)))
            s4.append(preprocess_input(stack_4.astype(np.float32)))

        inputs = (np.array(s1), np.array(s2), np.array(s3), np.array(s4), np.array(bm))

        # Targets must be a TUPLE (), not a List []
        targets = (by, by, by, by, by)

        return inputs, targets

# ==========================================
# 3. DEFINE THE QUAD MODEL
# ==========================================
def create_quad_model(num_classes=5):
    def build_tower(input_shape, name_suffix):
        inp = Input(shape=input_shape, name=f"in_{name_suffix}")
        # Use DenseNet121 instead of Xception
        base = DenseNet121(
            weights='imagenet',
            include_top=False,
            input_shape=input_shape,
            name=f"densenet_{name_suffix}"
        )
        base.trainable = False

        x = base(inp)
        x = layers.GlobalAveragePooling2D()(x)

        # Tower Layer 1
        x = layers.Dense(1024, kernel_initializer='he_normal', use_bias=False)(x) # he_normal for Swish/ReLU stability
        x = layers.BatchNormalization()(x) # Normalize before activation (Pre-Activation)
        x = layers.Activation('swish')(x)  # Swish is superior to ReLU for deep networks
        x = layers.Dropout(0.5)(x)

        # Tower Layer 2
        # Compress to standard feature size
        x = layers.Dense(512, kernel_initializer='he_normal', use_bias=False)(x)
        x = layers.BatchNormalization()(x)
        x = layers.Activation('swish')(x)
        x = layers.Dropout(0.5)(x)

        # Attention Gate (Standard)
        # This allows the model to gate specific features instead of the whole image
        att = layers.Dense(512, activation='sigmoid', name=f"att_gate_{name_suffix}")(x)
        boost = layers.Multiply()([x, att])
        x_res = layers.Add(name=f"feat_{name_suffix}")([x, boost])

        aux_out = layers.Dense(num_classes, activation='softmax', name=f"aux_{name_suffix}")(x_res)
        return inp, x_res, aux_out

    # This function allows "Source" tower to look at "Target" tower and borrow features
    def apply_cross_attention(source_feat, target_feat, suffix):
        # 1. Compress target features (The "Key" / "Value")
        context = layers.Dense(128, activation='relu', name=f"ctx_ex_{suffix}")(target_feat)

        # 2. Compress source features (The "Query")
        query = layers.Dense(128, activation='relu', name=f"query_ex_{suffix}")(source_feat)

        # 3. Compute Relevance: Concatenate Query and Key -> Sigmoid Gate
        # (Using Concatenate is more stable than Dot product for batch vectors)
        concat_qk = layers.Concatenate()([query, context])
        attention_score = layers.Dense(512, activation='sigmoid', name=f"att_score_{suffix}")(concat_qk)

        # 4. Gated Context: Scale the target features by how relevant they are
        # Rescale context back to 512 to match source
        context_scaled = layers.Dense(512, activation='relu', kernel_initializer='zeros', name=f"ctx_up_{suffix}")(context)
        gated_context = layers.Multiply()([context_scaled, attention_score])

        return gated_context

    in1, feat1, aux1 = build_tower((299,299,3), "t1_global_gamma")
    in2, feat2, aux2 = build_tower((299,299,3), "t2_global_clahe")
    in3, feat3, aux3 = build_tower((299,299,3), "t3_local_sigmoid")
    in4, feat4, aux4 = build_tower((299,299,3), "t4_local_clahe")

    # TOWER 1 Enriched: T1 + Context from T2, T3, T4
    c1_2 = apply_cross_attention(feat1, feat2, "t1_from_t2")
    c1_3 = apply_cross_attention(feat1, feat3, "t1_from_t3")
    c1_4 = apply_cross_attention(feat1, feat4, "t1_from_t4")
    feat1_enriched = layers.Add(name="t1_enriched")([feat1, c1_2, c1_3, c1_4])

    # TOWER 2 Enriched: T2 + Context from T1, T3, T4
    c2_1 = apply_cross_attention(feat2, feat1, "t2_from_t1")
    c2_3 = apply_cross_attention(feat2, feat3, "t2_from_t3")
    c2_4 = apply_cross_attention(feat2, feat4, "t2_from_t4")
    feat2_enriched = layers.Add(name="t2_enriched")([feat2, c2_1, c2_3, c2_4])

    # TOWER 3 Enriched: T3 + Context from T1, T2, T4
    c3_1 = apply_cross_attention(feat3, feat1, "t3_from_t1")
    c3_2 = apply_cross_attention(feat3, feat2, "t3_from_t2")
    c3_4 = apply_cross_attention(feat3, feat4, "t3_from_t4")
    feat3_enriched = layers.Add(name="t3_enriched")([feat3, c3_1, c3_2, c3_4])

    # TOWER 4 Enriched: T4 + Context from T1, T2, T3
    c4_1 = apply_cross_attention(feat4, feat1, "t4_from_t1")
    c4_2 = apply_cross_attention(feat4, feat2, "t4_from_t2")
    c4_3 = apply_cross_attention(feat4, feat3, "t4_from_t3")
    feat4_enriched = layers.Add(name="t4_enriched")([feat4, c4_1, c4_2, c4_3])

    in_meta = Input(shape=(5,), name="in_meta")
    feat_meta = layers.Dense(64, activation='relu')(in_meta)

    # --- CONCATENATION ---
    # We concatenate the ENRICHED features, not the raw ones
    concat = layers.Concatenate()([feat1_enriched, feat2_enriched, feat3_enriched, feat4_enriched, feat_meta])

    # Stabilized SE Block
    se_filters = 2112 // 16

    # Reduction (Squeeze)
    se = layers.Dense(se_filters, kernel_initializer='he_normal', use_bias=False)(concat)
    se = layers.BatchNormalization()(se) # BN added here to stabilize reduction
    se = layers.Activation('swish')(se)

    # Expansion (Excitation)
    se = layers.Dense(2112, kernel_initializer='he_normal')(se)
    se = layers.Activation('sigmoid')(se) # Sigmoid must be last for gating

    fused_weighted = layers.Multiply()([concat, se])

    # Applied "Dense -> BN -> Swish -> Dropout" pattern
    # Layer 1
    z = layers.Dense(1024, kernel_initializer='he_normal', use_bias=False)(fused_weighted)
    z = layers.BatchNormalization()(z) # Standard placement
    z = layers.Activation('swish')(z)
    z = layers.Dropout(0.3)(z)

    # Layer 2
    z = layers.Dense(512, kernel_initializer='he_normal', use_bias=False)(z)
    z = layers.BatchNormalization()(z)
    z = layers.Activation('swish')(z)
    z = layers.Dropout(0.3)(z)

    final_out = layers.Dense(num_classes, activation='softmax', name="final_output")(z)

    model = Model(
        inputs=[in1, in2, in3, in4, in_meta],
        outputs=[final_out, aux1, aux2, aux3, aux4],
        name="v1.0_Quad_Expert_System"
    )
    return model

# ==========================================
# 4. CUSTOM CALLBACK FOR SAFE SAVING
# ==========================================
class ContinuousHistorySaver(tf.keras.callbacks.Callback):
    def __init__(self, filepath, previous_history=None):
        super().__init__()
        self.filepath = filepath
        # If we have old history, start with that. Otherwise, empty dict.
        self.history_storage = previous_history if previous_history else {}

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        # Update storage with the latest metrics from this epoch
        for k, v in logs.items():
            if k not in self.history_storage:
                self.history_storage[k] = []
            self.history_storage[k].append(v)

        # Save immediately to Drive
        try:
            with open(self.filepath, 'wb') as f:
                pickle.dump(self.history_storage, f)
        except Exception as e:
            print(f"\n Warning: Could not save history for epoch {epoch}: {e}")

# ==========================================
# 5. TRAINING LOOP
# ==========================================
def run_phase_1_quad(save_dir):
    CHECKPOINT_DIR = os.path.join(save_dir, 'checkpoints_v1.0')
    os.makedirs(CHECKPOINT_DIR, exist_ok=True)
    print(f" Saving checkpoints to: {CHECKPOINT_DIR}")

    # We set the loss weights to be equal [1.0] for all classes.
    alpha_balanced = np.array([1.0, 1.0, 1.0, 1.0, 1.0], dtype=float)
    print("\n Balanced Alphas:", alpha_balanced)

    # --- FRESH METRIC INSTANCES ---
    # We create a function to generate FRESH metrics for each head
    def get_metrics(name_prefix):
        return [
            'accuracy',
            tf.keras.metrics.AUC(name=f'{name_prefix}_auc'),
            tf.keras.metrics.F1Score(average='macro', name=f'{name_prefix}_f1')
        ]

    # oversample_ratio is technically not used in the new calculation formula,
    # But we pass > 0 to trigger the "balanced" sampling mode logic.
    # 2. Pass Weights to Generator
    train_gen = QuadStreamGenerator(
        X_train_global, X_train_local, X_train_metadata, y_train,
        batch_size=12, augment=True,
        oversample_ratio=1 # Triggers the balanced sampling
    )

    val_gen = QuadStreamGenerator(
        X_val_global, X_val_local, X_val_metadata, y_val,
        batch_size=12, augment=False,
        oversample_ratio=0.0 # Validation must remain natural distribution
    )

    # Resume Logic
    initial_epoch = 0
    model = None
    existing_checkpoints = glob.glob(os.path.join(CHECKPOINT_DIR, "epoch_*.keras"))

    if existing_checkpoints:
        def get_epoch_num(filepath):
            match = re.search(r"epoch_(\d+).keras", filepath)
            return int(match.group(1)) if match else -1
        latest_ckpt = max(existing_checkpoints, key=get_epoch_num)
        print(f" Found checkpoint! Resuming from: {latest_ckpt}")
        try:
            model = tf.keras.models.load_model(
                latest_ckpt,
                # Ensure Gamma=2.5 is passed when loading custom object
                custom_objects={'focal_loss_fixed': class_weighted_focal_loss(gamma=2.5, alpha_weights=alpha_balanced)}
            )
            found_epoch = get_epoch_num(latest_ckpt)
            initial_epoch = found_epoch
            print(f" Resuming training at Epoch {initial_epoch + 1}")
        except Exception as e:
            print(f" Error loading checkpoint {e}. Starting fresh.")
            model = None

    if model is None:
        print(" Starting v1.0 training from scratch...")
        model = create_quad_model()

        # Compile with Gamma=2.5
        # We pass the alpha_final list computed above
        loss_fn = class_weighted_focal_loss(gamma=2.5, alpha_weights=alpha_balanced)
        loss_list = [loss_fn for _ in range(5)]
        loss_weights_list = [1.0, 0.1, 0.1, 0.1, 0.1]

        # Apply specific metrics to specific outputs using a Dictionary
        metrics_dict = {
            'final_output': get_metrics('final'),
            'aux_t1_global_gamma': get_metrics('aux1'),
            'aux_t2_global_clahe': get_metrics('aux2'),
            'aux_t3_local_sigmoid': get_metrics('aux3'),
            'aux_t4_local_clahe': get_metrics('aux4')
        }

        model.compile(
            optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
            loss=loss_list,
            loss_weights=loss_weights_list,
            metrics=metrics_dict,
            run_eagerly=False
        )
    else:
        print(" Optimizer state restored from checkpoint (Compilation skipped).")

    # --- 5. HISTORY RETRIEVAL ---
    # We load history NOW to find the previous best score
    history_path = os.path.join(save_dir, 'phase1_v1.0_history.pkl')
    previous_best_f1 = -np.inf
    old_history = {}

    if os.path.exists(history_path):
        print(" Found existing history file. Scanning for previous best F1...")
        try:
            with open(history_path, 'rb') as f:
                old_history = pickle.load(f)

            target_metric = 'val_final_output_final_f1'
            if target_metric in old_history and len(old_history[target_metric]) > 0:
                previous_best_f1 = max(old_history[target_metric])
                print(f" Previous Best Val F1 was: {previous_best_f1:.4f}")
            else:
                print(" Metric not found in history. Assuming -inf.")
        except Exception as e:
            print(f" Could not read history file: {e}")

    # --- 6. DEFINE CALLBACKS WITH MEMORY ---

    # Manually create the checkpoint callback so we can modify it
    best_ckpt = tf.keras.callbacks.ModelCheckpoint(
        filepath=os.path.join(save_dir, 'phase1_v1.0_best.keras'),
        save_best_only=True,
        monitor='val_final_output_final_f1',
        verbose=1,
        mode='max'
    )

    # INJECT THE MEMORY
    # If we found a previous best, we force the callback to respect it.
    if previous_best_f1 > -np.inf:
        best_ckpt.best = previous_best_f1
        print(f" Injected previous best F1 ({previous_best_f1:.4f}) into ModelCheckpoint.")

    # 2. Continuous History Saver
    # We pass 'old_history' so it continues appending to it, rather than starting fresh
    history_saver = ContinuousHistorySaver(filepath=history_path, previous_history=old_history)

    callbacks = [
        tf.keras.callbacks.EarlyStopping(
            monitor='val_final_output_final_f1',
            patience=6,
            restore_best_weights=True,
            mode='max'
        ),

        tf.keras.callbacks.ReduceLROnPlateau(
            monitor='val_final_output_final_f1',
            factor=0.2,
            patience=2,
            min_lr=1e-6,
            mode='max'
        ),

        tf.keras.callbacks.ModelCheckpoint(
            filepath=os.path.join(CHECKPOINT_DIR, "epoch_{epoch:02d}.keras"),
            save_best_only=False, verbose=1
        ),
        #The custom one we created above with memory
        best_ckpt,
        history_saver   # Saves history.pkl every epoch
    ]

    print(f" Running Phase 1 v1.0 (Quad-Expert)...")

    # 3. Fit without class_weight arg
    history = model.fit(
        train_gen,
        validation_data=val_gen,
        initial_epoch=initial_epoch,
        epochs=30,
        callbacks=callbacks
    )

    print(f"\n Saving Final v1.0 Model to: {save_dir}")
    model.save(os.path.join(save_dir, 'phase1_v1.0_final.keras'))
    new_history = history.history

    if os.path.exists(history_path):
        print("found existing history, merging...")
        with open(history_path, 'rb') as f:
            old_history = pickle.load(f)

        # IMPROVED MERGE LOGIC:
        # Loop through NEW history to ensure we capture everything
        for key, val in new_history.items():
            if key in old_history:
                # Append new epochs to existing list
                old_history[key].extend(val)
            else:
                # If a new metric appeared (rare, but possible), add it
                old_history[key] = val

        final_history = old_history
    else:
        final_history = new_history

    # Save the combined history
    with open(history_path, 'wb') as f:
        pickle.dump(final_history, f)

    return model, history

# RUN
MY_DRIVE_PATH = '/content/drive/My Drive/DenseNet121 model/v1.0'
os.makedirs(MY_DRIVE_PATH, exist_ok=True)

model, history = run_phase_1_quad(save_dir=MY_DRIVE_PATH)

###Phase 2 training (Unfreeze block 14 for fine-tuning)

In [ ]:
import os
import glob
import re
import pickle
import numpy as np
import cv2
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.utils import Sequence
from tensorflow.keras.applications.densenet import DenseNet121, preprocess_input
from tensorflow.keras import layers, Model, Input
import albumentations as A

# ==========================================
# 1. RE-DEFINE CUSTOM OBJECTS (REQUIRED FOR LOADING)
# ==========================================
def class_weighted_focal_loss(gamma=2.5, alpha_weights=None):
    """
    gamma: Reduced to 2.5. Gamma 3.0 was too aggressive and suppressed gradients
           for "moderately hard" samples (like Luminal A), hurting their learning.
    """
    # Convert alpha to a tensor constant for efficiency
    if alpha_weights is not None:
        alpha_tensor = tf.constant(alpha_weights, dtype=tf.float32)
    else:
        alpha_tensor = None

    def focal_loss_fixed(y_true, y_pred):
        epsilon = tf.keras.backend.epsilon()
        y_pred = tf.clip_by_value(y_pred, epsilon, 1. - epsilon)

        # Step 1: Get probability of TRUE class only
        # y_true: (batch, 5), y_pred: (batch, 5)
        y_true = tf.cast(y_true, tf.float32)

        # p_t = probability assigned to the true class for each sample
        # Shape: (batch_size,)
        pt = tf.reduce_sum(y_true * y_pred, axis=-1)  # sum(y_true_i * y_pred_i) = p_true_class

        # Step 2: Basic cross entropy for true class
        ce = -tf.math.log(pt + epsilon)  # Shape: (batch_size,)

        # Step 3: Focal weighting based on how confident we are in TRUE class
        focal_weight = tf.math.pow(1 - pt, gamma)  # Shape: (batch_size,)

        # 3. Calculate Class Weight Term
        # Multiply by alpha corresponding to the true class
        if alpha_tensor is not None:
            # Get alpha for the true class of each sample
            # y_true * alpha gives (batch, 5) with alpha only at true class position
            # Then sum to get shape (batch_size,)
            alpha_t = tf.reduce_sum(y_true * alpha_tensor, axis=-1)
            loss = alpha_t * focal_weight * ce
        else:
            loss = focal_weight * ce

        return tf.reduce_mean(loss)  # Mean over batch

    return focal_loss_fixed

# ==========================================
# 2. RE-DEFINE GENERATOR
# ==========================================
class QuadStreamGenerator(Sequence):
    def __init__(self, x_global, x_local, x_meta, y, batch_size,
                 augment=False, shuffle=True,
                 oversample_ratio=0.0):
        super().__init__()
        self.x_global = x_global
        self.x_local = x_local
        self.x_meta = x_meta
        self.y = y
        self.batch_size = batch_size
        self.augment = augment
        self.shuffle = shuffle

        # Oversampling Setup
        self.oversample_ratio = oversample_ratio
        self.indexes = np.arange(len(self.x_global))

        # Calculate Sampling Probabilities if oversampling is enabled
        self.sampling_probs = None
        if self.oversample_ratio > 0:
            self.sampling_probs = self._calculate_sampling_probs()

        # --- ENHANCEMENT OBJECTS ---
        self.clahe_mild = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
        self.clahe_strong = cv2.createCLAHE(clipLimit=4.0, tileGridSize=(8,8))
        self.gamma_mild_table = self._build_gamma_table(0.6)
        self.gamma_strong_table = self._build_gamma_table(1.5)

        # --- SPECIALIZED AUGMENTATION PIPELINES ---
        # Using Albumentations to treat each class biologically correctly
        if self.augment:
            self.aug_pipelines = self._build_augmentation_pipelines()

        self.on_epoch_end()

    def _build_augmentation_pipelines(self):
        # 1. Base Geometric (Safe for all)
        # We use additional_targets to apply SAME transform to all 4 towers
        base_aug = [
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.5),
            A.Affine(scale=(0.9, 1.1), translate_percent=(0.1, 0.1), rotate=(-20, 20), p=0.5),
        ]

        # 2. Define Class-Specific Pipelines
        pipelines = {}

        # --- Class 0: BENIGN ---
        pipelines[0] = A.Compose(base_aug + [
            A.MedianBlur(blur_limit=5, p=0.3),
        ], additional_targets={'image2': 'image', 'image3': 'image', 'image4': 'image'})

        # --- Class 1: LUMINAL A ---
        pipelines[1] = A.Compose(base_aug + [
             A.ISONoise(color_shift=(0.01, 0.05), intensity=(0.1, 0.5), p=0.2),
        ], additional_targets={'image2': 'image', 'image3': 'image', 'image4': 'image'})

        # --- Class 2: LUMINAL B (TARGET) ---
        pipelines[2] = A.Compose(base_aug + [
            # 1. Geometry: Create the spicule shape physically
            A.ElasticTransform(alpha=1, sigma=50, p=0.5),

            # 2. Structure: Highlight the edges (MILD settings)
            A.Emboss(alpha=(0.2, 0.5), strength=(0.3, 0.5), p=0.4),
        ], additional_targets={'image2': 'image', 'image3': 'image', 'image4': 'image'})

        # --- Class 3: HER2+ ---
        pipelines[3] = A.Compose(base_aug + [
            A.Sharpen(alpha=(0.2, 0.5), lightness=(0.5, 1.0), p=0.5),
        ], additional_targets={'image2': 'image', 'image3': 'image', 'image4': 'image'})

        # --- Class 4: TN (TARGET) ---
        pipelines[4] = A.Compose(base_aug + [
            # 1. Geometry: Create the "Bulge" / Mass Effect
            A.GridDistortion(num_steps=5, distort_limit=0.3, p=0.5),

            # 2. Structure: Define the sharp margin (MILD)
            A.Sharpen(alpha=(0.1, 0.3), lightness=(0.5, 1.0), p=0.4),

            # 3. Density: High Density (Dark/Bright shift)
            # Safe Range: (85, 115)
            A.RandomGamma(gamma_limit=(85, 115), p=0.5),
        ], additional_targets={'image2': 'image', 'image3': 'image', 'image4': 'image'})

        return pipelines

    def _calculate_sampling_probs(self):
        # 1. Get integer labels
        y_int = np.argmax(self.y, axis=1)
        classes, counts = np.unique(y_int, return_counts=True)
        # Calculate weight using Inverse Frequency (1.0 / count)
        # This ensures every class has roughly EQUAL probability of being picked per batch
        class_weight_map = {c: 1.0/count for c, count in zip(classes, counts)}

        sample_weights = np.array([class_weight_map[label] for label in y_int])

        # Normalize to sum to 1.0 for np.random.choice
        return sample_weights / sample_weights.sum()

    def _build_gamma_table(self, gamma):
        invGamma = 1.0 / gamma
        return np.array([((i / 255.0) ** invGamma) * 255 for i in np.arange(0, 256)]).astype("uint8")

    def apply_gamma(self, img, table):
        return cv2.LUT(img, table)

    def apply_sigmoid(self, img, gain, cutoff):
        img_f = img.astype(np.float32) / 255.0
        sigmoid = 1.0 / (1.0 + np.exp(-gain * (img_f - cutoff)))
        return (sigmoid * 255).astype(np.uint8)

    def __len__(self):
        return int(np.ceil(len(self.x_global) / self.batch_size))

    def on_epoch_end(self):
        # Logic to handle Weighted Random Sampling
        if self.sampling_probs is not None and self.shuffle:
            # Sample WITH REPLACEMENT based on calculated probabilities
            # This creates a new list of indexes where minority classes appear more often
            self.indexes = np.random.choice(
                np.arange(len(self.x_global)),
                size=len(self.x_global),
                p=self.sampling_probs,
                replace=True
            )
        elif self.shuffle:
            np.random.shuffle(self.indexes)

    def __getitem__(self, index):
        idxs = self.indexes[index*self.batch_size:(index+1)*self.batch_size]

        s1, s2, s3, s4 = [], [], [], []
        bm = self.x_meta[idxs]
        by = self.y[idxs]

        for i in range(len(idxs)):
            # Identify Class for Specialization
            current_label_idx = np.argmax(by[i])

            # Load raw images
            ig_raw = self.x_global[idxs[i]]
            il_raw = self.x_local[idxs[i]]

            if ig_raw.ndim == 3: ig = ig_raw[:,:,0].astype(np.uint8)
            else: ig = ig_raw.astype(np.uint8)

            if il_raw.ndim == 3: il = il_raw[:,:,0].astype(np.uint8)
            else: il = il_raw.astype(np.uint8)

            # --- PREPARE STACKS ---
            # Tower 1: Global Gamma
            stack_1 = np.stack([
                ig,
                self.apply_gamma(ig, self.gamma_mild_table),
                self.apply_gamma(ig, self.gamma_strong_table)
            ], axis=-1)

            # Tower 2: Global CLAHE
            stack_2 = np.stack([
                ig,
                self.clahe_mild.apply(ig),
                self.clahe_strong.apply(ig)
            ], axis=-1)

            # Tower 3: Local Sigmoid
            stack_3 = np.stack([
                il,
                self.apply_sigmoid(il, gain=8, cutoff=0.75),
                self.apply_sigmoid(il, gain=15, cutoff=0.60)
            ], axis=-1)

            # Tower 4: Local CLAHE
            stack_4 = np.stack([
                il,
                self.clahe_mild.apply(il),
                self.clahe_strong.apply(il)
            ], axis=-1)

            # --- APPLY CLASS-SPECIFIC AUGMENTATION ---
            if self.augment:
                # Select the correct pipeline for this specific image's class
                pipeline = self.aug_pipelines[current_label_idx]

                # Apply to all 4 towers simultaneously to keep them geometrically aligned
                # Albumentations 'image' is primary, 'image2/3/4' are additional
                augmented = pipeline(
                    image=stack_1,
                    image2=stack_2,
                    image3=stack_3,
                    image4=stack_4
                )

                stack_1 = augmented['image']
                stack_2 = augmented['image2']
                stack_3 = augmented['image3']
                stack_4 = augmented['image4']

            s1.append(preprocess_input(stack_1.astype(np.float32)))
            s2.append(preprocess_input(stack_2.astype(np.float32)))
            s3.append(preprocess_input(stack_3.astype(np.float32)))
            s4.append(preprocess_input(stack_4.astype(np.float32)))

        inputs = (np.array(s1), np.array(s2), np.array(s3), np.array(s4), np.array(bm))

        # Targets must be a TUPLE (), not a List []
        targets = (by, by, by, by, by)

        return inputs, targets

# ==========================================
# 3. HELPER: UNFREEZE LOGIC (DENSENET VERSION)
# ==========================================
def unfreeze_densenet_blocks(model, target_block_str="conv5"):
    """
    Traverses the 4 DenseNet towers and unfreezes the top-most blocks.
    For DenseNet121, the last dense block layers start with 'conv5_'.
    """
    print(f"\n Unfreezing '{target_block_str}' layers in all DenseNet towers...")

    unfrozen_count = 0

    # Iterate through the main model layers to find the DenseNet Bases
    for layer in model.layers:
        # We renamed them to "densenet_t1...", "densenet_t2..." in Phase 1
        if "densenet" in layer.name and isinstance(layer, Model):
            print(f"   -> Processing Tower: {layer.name}")

            # Set the Container Model to trainable
            layer.trainable = True

            # Now traverse the layers INSIDE the DenseNet Base
            for inner_layer in layer.layers:
                # DenseNet naming convention: conv5_block1_..., conv5_block16_...
                if target_block_str in inner_layer.name:
                    # KEEP BATCH NORM FROZEN
                    if isinstance(inner_layer, layers.BatchNormalization):
                        inner_layer.trainable = False
                    else:
                        inner_layer.trainable = True
                        unfrozen_count += 1
                else:
                    # Keep earlier blocks (conv1, conv2, conv3, conv4) frozen
                    inner_layer.trainable = False

    print(f" Total layers unfrozen across all towers: {unfrozen_count}")
    return model

# ==========================================
# 4. CUSTOM CALLBACK FOR SAFE SAVING
# ==========================================
class ContinuousHistorySaver(tf.keras.callbacks.Callback):
    def __init__(self, filepath, previous_history=None):
        super().__init__()
        self.filepath = filepath
        # If we have old history, start with that. Otherwise, empty dict.
        self.history_storage = previous_history if previous_history else {}

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        # Update storage with the latest metrics from this epoch
        for k, v in logs.items():
            if k not in self.history_storage:
                self.history_storage[k] = []
            self.history_storage[k].append(v)

        # Save immediately to Drive
        try:
            with open(self.filepath, 'wb') as f:
                pickle.dump(self.history_storage, f)
        except Exception as e:
            print(f"\n Warning: Could not save history for epoch {epoch}: {e}")

# ==========================================
# 5. PHASE 2 TRAINING LOOP
# ==========================================
def run_phase_2_finetune(save_dir, resume_from_dir):
    # Setup directories
    CHECKPOINT_DIR = os.path.join(save_dir, 'checkpoints_v1.0_phase2')
    os.makedirs(CHECKPOINT_DIR, exist_ok=True)
    print(f" Saving Phase 2 checkpoints to: {CHECKPOINT_DIR}")

    # Setup Alphas & Metrics (Same as before)
    alpha_balanced = np.array([1.0, 1.0, 1.0, 1.0, 1.0], dtype=float)
    def get_metrics(name_prefix):
        return [
            'accuracy',
            tf.keras.metrics.AUC(name=f'{name_prefix}_auc'),
            tf.keras.metrics.F1Score(average='macro', name=f'{name_prefix}_f1')
        ]

    # --- LOAD DATA ---
    print(" Initializing Generators...")
    train_gen = QuadStreamGenerator(
        X_train_global, X_train_local, X_train_metadata, y_train,
        batch_size=12, augment=True, oversample_ratio=1
    )
    val_gen = QuadStreamGenerator(
        X_val_global, X_val_local, X_val_metadata, y_val,
        batch_size=12, augment=False, oversample_ratio=0.0
    )

    # --- SMART RESUMPTION LOGIC ---
    initial_epoch = 0
    model = None

    # 1. Check if we have interrupted Phase 2 checkpoints
    phase2_checkpoints = glob.glob(os.path.join(CHECKPOINT_DIR, "p2_epoch_*.keras"))

    if phase2_checkpoints:
        # Helper to find the highest epoch number
        def get_epoch_num(filepath):
            match = re.search(r"p2_epoch_(\d+).keras", filepath)
            return int(match.group(1)) if match else -1

        latest_ckpt = max(phase2_checkpoints, key=get_epoch_num)
        print(f" Found interrupted Phase 2 training! Resuming from: {latest_ckpt}")

        try:
            # Load the checkpoint.
            # This restores: Weights, Optimizer State, and "Unfrozen" status.
            model = tf.keras.models.load_model(
                latest_ckpt,
                custom_objects={'focal_loss_fixed': class_weighted_focal_loss(gamma=2.5, alpha_weights=alpha_balanced)}
            )
            found_epoch = get_epoch_num(latest_ckpt)
            initial_epoch = found_epoch
            print(f" Jumping ahead to Epoch {initial_epoch + 1}")
        except Exception as e:
            print(f" Error loading checkpoint {e}. Falling back to start.")
            model = None

    # 2. If no Phase 2 checkpoint found (or error), Start Fresh from Phase 1 Model
    if model is None:
        print("✨ No Phase 2 progress found. Starting Fine-Tuning from scratch...")

        phase1_path = os.path.join(resume_from_dir, 'phase1_v1.0_final.keras')
        if not os.path.exists(phase1_path):
            phase1_path = os.path.join(resume_from_dir, 'phase1_v1.0_best.keras')

        if not os.path.exists(phase1_path):
            raise FileNotFoundError(" Could not find Phase 1 model to fine-tune!")

        print(f" Loading Phase 1 Model from: {phase1_path}")
        model = tf.keras.models.load_model(
            phase1_path,
            custom_objects={'focal_loss_fixed': class_weighted_focal_loss(gamma=2.5, alpha_weights=alpha_balanced)}
        )

        # --- UNFREEZE & COMPILE (Only needed for fresh start) ---
        # UNFREEZE ONLY BLOCK 14 (The top-most block)
        # Unfreeze the entire last dense block (conv5)
        model = unfreeze_densenet_blocks(model, target_block_str="conv5")

        fine_tune_lr = 1e-6
        print(f" Recompiling with Fine-Tuning LR: {fine_tune_lr}")

        loss_fn = class_weighted_focal_loss(gamma=2.5, alpha_weights=alpha_balanced)
        metrics_dict = {
            'final_output': get_metrics('final'),
            'aux_t1_global_gamma': get_metrics('aux1'),
            'aux_t2_global_clahe': get_metrics('aux2'),
            'aux_t3_local_sigmoid': get_metrics('aux3'),
            'aux_t4_local_clahe': get_metrics('aux4')
        }

        model.compile(
            optimizer=tf.keras.optimizers.Adam(learning_rate=fine_tune_lr),
            loss=[loss_fn] * 5,
            loss_weights=[1.0, 0.1, 0.1, 0.1, 0.1],
            metrics=metrics_dict
        )
    else:
        print(" Optimizer state restored from checkpoint (Compilation skipped).")

    # --- HISTORY HANDLING ---
    phase1_hist_path = os.path.join(resume_from_dir, 'phase1_v1.0_history.pkl')
    phase2_hist_path = os.path.join(save_dir, 'phase2_v1.0_history.pkl')

    previous_best_f1 = -np.inf
    if os.path.exists(phase1_hist_path):
        with open(phase1_hist_path, 'rb') as f:
            p1_hist = pickle.load(f)
            if 'val_final_output_final_f1' in p1_hist:
                previous_best_f1 = max(p1_hist['val_final_output_final_f1'])

    # Load existing Phase 2 history if available
    current_p2_history = {}
    if os.path.exists(phase2_hist_path):
        with open(phase2_hist_path, 'rb') as f:
            current_p2_history = pickle.load(f)

    # --- CALLBACKS ---
    best_ckpt = tf.keras.callbacks.ModelCheckpoint(
        filepath=os.path.join(save_dir, 'phase2_v1.0_best.keras'),
        save_best_only=True,
        monitor='val_final_output_final_f1',
        verbose=1,
        mode='max'
    )
    if previous_best_f1 > -np.inf:
        best_ckpt.best = previous_best_f1

    history_saver = ContinuousHistorySaver(filepath=phase2_hist_path, previous_history=current_p2_history)

    callbacks = [
        tf.keras.callbacks.EarlyStopping(monitor='val_final_output_final_f1', patience=8, restore_best_weights=True, mode='max'),
        tf.keras.callbacks.ReduceLROnPlateau(monitor='val_final_output_final_f1', factor=0.2, patience=3, min_lr=1e-7, mode='max'),
        tf.keras.callbacks.ModelCheckpoint(filepath=os.path.join(CHECKPOINT_DIR, "p2_epoch_{epoch:02d}.keras"), save_best_only=False, verbose=1),
        best_ckpt,
        history_saver
    ]

    print(f" Running Phase 2 (Fine-Tuning) starting from Epoch {initial_epoch + 1}...")

    history = model.fit(
        train_gen,
        validation_data=val_gen,
        initial_epoch=initial_epoch,
        epochs=20,
        callbacks=callbacks
    )

    print(f"\n Saving Final Phase 2 Model to: {save_dir}")
    model.save(os.path.join(save_dir, 'phase2_v1.0_final.keras'))

    return model, history

# ==========================================
# 6. RUN EXECUTION
# ==========================================
MY_DRIVE_PATH = '/content/drive/My Drive/DenseNet121 model/v1.0'

model_p2, history_p2 = run_phase_2_finetune(save_dir=MY_DRIVE_PATH, resume_from_dir=MY_DRIVE_PATH)

#X. ResNet50V2 model

##Training model v1.17.1

###Phase 1 training

In [ ]:
import os
import glob
import re
import pickle
import numpy as np
import cv2
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.utils import Sequence
from tensorflow.keras.applications.resnet_v2 import ResNet50V2, preprocess_input
from tensorflow.keras import layers, Model, Input
import albumentations as A

# ==========================================
# 1. CUSTOM LOSS FUNCTION (CLASS-WEIGHTED FOCAL LOSS)
# ==========================================
def class_weighted_focal_loss(gamma=2.5, alpha_weights=None):
    """
    gamma: Reduced to 2.5. Gamma 3.0 was too aggressive and suppressed gradients
           for "moderately hard" samples (like Luminal A), hurting their learning.
    """
    # Convert alpha to a tensor constant for efficiency
    if alpha_weights is not None:
        alpha_tensor = tf.constant(alpha_weights, dtype=tf.float32)
    else:
        alpha_tensor = None

    def focal_loss_fixed(y_true, y_pred):
        epsilon = tf.keras.backend.epsilon()
        y_pred = tf.clip_by_value(y_pred, epsilon, 1. - epsilon)

        # Step 1: Get probability of TRUE class only
        # y_true: (batch, 5), y_pred: (batch, 5)
        y_true = tf.cast(y_true, tf.float32)

        # p_t = probability assigned to the true class for each sample
        # Shape: (batch_size,)
        pt = tf.reduce_sum(y_true * y_pred, axis=-1)  # sum(y_true_i * y_pred_i) = p_true_class

        # Step 2: Basic cross entropy for true class
        ce = -tf.math.log(pt + epsilon)  # Shape: (batch_size,)

        # Step 3: Focal weighting based on how confident we are in TRUE class
        focal_weight = tf.math.pow(1 - pt, gamma)  # Shape: (batch_size,)

        # 3. Calculate Class Weight Term
        # Multiply by alpha corresponding to the true class
        if alpha_tensor is not None:
            # Get alpha for the true class of each sample
            # y_true * alpha gives (batch, 5) with alpha only at true class position
            # Then sum to get shape (batch_size,)
            alpha_t = tf.reduce_sum(y_true * alpha_tensor, axis=-1)
            loss = alpha_t * focal_weight * ce
        else:
            loss = focal_weight * ce

        return tf.reduce_mean(loss)  # Mean over batch

    return focal_loss_fixed

# ==========================================
# 2. QUAD-STREAM GENERATOR
# ==========================================
class QuadStreamGenerator(Sequence):
    def __init__(self, x_global, x_local, x_meta, y, batch_size,
                 augment=False, shuffle=True,
                 oversample_ratio=0.0):
        super().__init__()
        self.x_global = x_global
        self.x_local = x_local
        self.x_meta = x_meta
        self.y = y
        self.batch_size = batch_size
        self.augment = augment
        self.shuffle = shuffle

        # Oversampling Setup
        self.oversample_ratio = oversample_ratio
        self.indexes = np.arange(len(self.x_global))

        # Calculate Sampling Probabilities if oversampling is enabled
        self.sampling_probs = None
        if self.oversample_ratio > 0:
            self.sampling_probs = self._calculate_sampling_probs()

        # --- ENHANCEMENT OBJECTS ---
        self.clahe_mild = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
        self.clahe_strong = cv2.createCLAHE(clipLimit=4.0, tileGridSize=(8,8))
        self.gamma_mild_table = self._build_gamma_table(0.6)
        self.gamma_strong_table = self._build_gamma_table(1.5)

        # --- SPECIALIZED AUGMENTATION PIPELINES ---
        # Using Albumentations to treat each class biologically correctly
        if self.augment:
            self.aug_pipelines = self._build_augmentation_pipelines()

        self.on_epoch_end()

    def _build_augmentation_pipelines(self):
        # 1. Base Geometric (Safe for all)
        # We use additional_targets to apply SAME transform to all 4 towers
        base_aug = [
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.5),
            A.Affine(scale=(0.9, 1.1), translate_percent=(0.1, 0.1), rotate=(-20, 20), p=0.5),
        ]

        # 2. Define Class-Specific Pipelines
        pipelines = {}

        # --- Class 0: BENIGN ---
        pipelines[0] = A.Compose(base_aug + [
            A.MedianBlur(blur_limit=5, p=0.3),
        ], additional_targets={'image2': 'image', 'image3': 'image', 'image4': 'image'})

        # --- Class 1: LUMINAL A ---
        pipelines[1] = A.Compose(base_aug + [
             A.ISONoise(color_shift=(0.01, 0.05), intensity=(0.1, 0.5), p=0.2),
        ], additional_targets={'image2': 'image', 'image3': 'image', 'image4': 'image'})

        # --- Class 2: LUMINAL B ---
        pipelines[2] = A.Compose(base_aug + [
            # 1. Geometry: Create the spicule shape physically
            A.ElasticTransform(alpha=1, sigma=50, p=0.5),

            # 2. Structure: Highlight the edges (MILD settings)
            A.Emboss(alpha=(0.2, 0.5), strength=(0.3, 0.5), p=0.4),
        ], additional_targets={'image2': 'image', 'image3': 'image', 'image4': 'image'})

        # --- Class 3: HER2+ ---
        pipelines[3] = A.Compose(base_aug + [
            A.Sharpen(alpha=(0.2, 0.5), lightness=(0.5, 1.0), p=0.5),
        ], additional_targets={'image2': 'image', 'image3': 'image', 'image4': 'image'})

        # --- Class 4: TN ---
        pipelines[4] = A.Compose(base_aug + [
            # 1. Geometry: Create the "Bulge" / Mass Effect
            A.GridDistortion(num_steps=5, distort_limit=0.3, p=0.5),

            # 2. Structure: Define the sharp margin (MILD)
            A.Sharpen(alpha=(0.1, 0.3), lightness=(0.5, 1.0), p=0.4),

            # 3. Density: High Density (Dark/Bright shift)
            # Safe Range: (85, 115)
            A.RandomGamma(gamma_limit=(85, 115), p=0.5),
        ], additional_targets={'image2': 'image', 'image3': 'image', 'image4': 'image'})

        return pipelines

    def _calculate_sampling_probs(self):
        # 1. Get integer labels
        y_int = np.argmax(self.y, axis=1)
        classes, counts = np.unique(y_int, return_counts=True)
        # Calculate weight using Inverse Frequency (1.0 / count)
        # This ensures every class has roughly EQUAL probability of being picked per batch.
        class_weight_map = {c: 1.0/count for c, count in zip(classes, counts)}

        sample_weights = np.array([class_weight_map[label] for label in y_int])

        # Normalize to sum to 1.0 for np.random.choice
        return sample_weights / sample_weights.sum()

    def _build_gamma_table(self, gamma):
        invGamma = 1.0 / gamma
        return np.array([((i / 255.0) ** invGamma) * 255 for i in np.arange(0, 256)]).astype("uint8")

    def apply_gamma(self, img, table):
        return cv2.LUT(img, table)

    def apply_sigmoid(self, img, gain, cutoff):
        img_f = img.astype(np.float32) / 255.0
        sigmoid = 1.0 / (1.0 + np.exp(-gain * (img_f - cutoff)))
        return (sigmoid * 255).astype(np.uint8)

    def __len__(self):
        return int(np.ceil(len(self.x_global) / self.batch_size))

    def on_epoch_end(self):
        # Logic to handle Weighted Random Sampling
        if self.sampling_probs is not None and self.shuffle:
            # Sample WITH REPLACEMENT based on calculated probabilities
            # This creates a new list of indexes where minority classes appear more often
            self.indexes = np.random.choice(
                np.arange(len(self.x_global)),
                size=len(self.x_global),
                p=self.sampling_probs,
                replace=True
            )
        elif self.shuffle:
            np.random.shuffle(self.indexes)

    def __getitem__(self, index):
        idxs = self.indexes[index*self.batch_size:(index+1)*self.batch_size]

        s1, s2, s3, s4 = [], [], [], []
        bm = self.x_meta[idxs]
        by = self.y[idxs]

        for i in range(len(idxs)):
            # Identify Class for Specialization
            current_label_idx = np.argmax(by[i])

            # Load raw images
            ig_raw = self.x_global[idxs[i]]
            il_raw = self.x_local[idxs[i]]

            if ig_raw.ndim == 3: ig = ig_raw[:,:,0].astype(np.uint8)
            else: ig = ig_raw.astype(np.uint8)

            if il_raw.ndim == 3: il = il_raw[:,:,0].astype(np.uint8)
            else: il = il_raw.astype(np.uint8)

            # --- PREPARE STACKS ---
            # Tower 1: Global Gamma
            stack_1 = np.stack([
                ig,
                self.apply_gamma(ig, self.gamma_mild_table),
                self.apply_gamma(ig, self.gamma_strong_table)
            ], axis=-1)

            # Tower 2: Global CLAHE
            stack_2 = np.stack([
                ig,
                self.clahe_mild.apply(ig),
                self.clahe_strong.apply(ig)
            ], axis=-1)

            # Tower 3: Local Sigmoid
            stack_3 = np.stack([
                il,
                self.apply_sigmoid(il, gain=8, cutoff=0.75),
                self.apply_sigmoid(il, gain=15, cutoff=0.60)
            ], axis=-1)

            # Tower 4: Local CLAHE
            stack_4 = np.stack([
                il,
                self.clahe_mild.apply(il),
                self.clahe_strong.apply(il)
            ], axis=-1)

            # --- APPLY CLASS-SPECIFIC AUGMENTATION ---
            if self.augment:
                # Select the correct pipeline for this specific image's class
                pipeline = self.aug_pipelines[current_label_idx]

                # Apply to all 4 towers simultaneously to keep them geometrically aligned
                # Albumentations 'image' is primary, 'image2/3/4' are additional
                augmented = pipeline(
                    image=stack_1,
                    image2=stack_2,
                    image3=stack_3,
                    image4=stack_4
                )

                stack_1 = augmented['image']
                stack_2 = augmented['image2']
                stack_3 = augmented['image3']
                stack_4 = augmented['image4']

            s1.append(preprocess_input(stack_1.astype(np.float32)))
            s2.append(preprocess_input(stack_2.astype(np.float32)))
            s3.append(preprocess_input(stack_3.astype(np.float32)))
            s4.append(preprocess_input(stack_4.astype(np.float32)))

        inputs = (np.array(s1), np.array(s2), np.array(s3), np.array(s4), np.array(bm))

        # Targets must be a TUPLE (), not a List []
        targets = (by, by, by, by, by)

        return inputs, targets

# ==========================================
# 3. DEFINE THE QUAD MODEL
# ==========================================
def create_quad_model(num_classes=5):
    def build_tower(input_shape, name_suffix):
        inp = Input(shape=input_shape, name=f"in_{name_suffix}")

        # Use ResNet50V2
        base = ResNet50V2(
            weights='imagenet',
            include_top=False,
            input_shape=input_shape,
            name=f"resnet_{name_suffix}"
        )
        base.trainable = False

        x = base(inp)
        x = layers.GlobalAveragePooling2D()(x)

        # Tower Layer 1
        x = layers.Dense(1024, kernel_initializer='he_normal', use_bias=False)(x) # he_normal for Swish/ReLU stability
        x = layers.BatchNormalization()(x) # Normalize before activation (Pre-Activation)
        x = layers.Activation('swish')(x)  # Swish is superior to ReLU for deep networks
        x = layers.Dropout(0.5)(x)

        # Tower Layer 2
        # Compress to standard feature size
        x = layers.Dense(512, kernel_initializer='he_normal', use_bias=False)(x)
        x = layers.BatchNormalization()(x)
        x = layers.Activation('swish')(x)
        x = layers.Dropout(0.5)(x)

        # Attention Gate (Standard)
        # This allows the model to gate specific features instead of the whole image
        att = layers.Dense(512, activation='sigmoid', name=f"att_gate_{name_suffix}")(x)
        boost = layers.Multiply()([x, att])
        x_res = layers.Add(name=f"feat_{name_suffix}")([x, boost])

        aux_out = layers.Dense(num_classes, activation='softmax', name=f"aux_{name_suffix}")(x_res)
        return inp, x_res, aux_out

    # This function allows "Source" tower to look at "Target" tower and borrow features
    def apply_cross_attention(source_feat, target_feat, suffix):
        # 1. Compress target features (The "Key" / "Value")
        context = layers.Dense(128, activation='relu', name=f"ctx_ex_{suffix}")(target_feat)

        # 2. Compress source features (The "Query")
        query = layers.Dense(128, activation='relu', name=f"query_ex_{suffix}")(source_feat)

        # 3. Compute Relevance: Concatenate Query and Key -> Sigmoid Gate
        # (Using Concatenate is more stable than Dot product for batch vectors)
        concat_qk = layers.Concatenate()([query, context])
        attention_score = layers.Dense(512, activation='sigmoid', name=f"att_score_{suffix}")(concat_qk)

        # 4. Gated Context: Scale the target features by how relevant they are
        # Rescale context back to 512 to match source
        context_scaled = layers.Dense(512, activation='relu', kernel_initializer='zeros', name=f"ctx_up_{suffix}")(context)
        gated_context = layers.Multiply()([context_scaled, attention_score])

        return gated_context

    in1, feat1, aux1 = build_tower((299,299,3), "t1_global_gamma")
    in2, feat2, aux2 = build_tower((299,299,3), "t2_global_clahe")
    in3, feat3, aux3 = build_tower((299,299,3), "t3_local_sigmoid")
    in4, feat4, aux4 = build_tower((299,299,3), "t4_local_clahe")

    # TOWER 1 Enriched: T1 + Context from T2, T3, T4
    c1_2 = apply_cross_attention(feat1, feat2, "t1_from_t2")
    c1_3 = apply_cross_attention(feat1, feat3, "t1_from_t3")
    c1_4 = apply_cross_attention(feat1, feat4, "t1_from_t4")
    feat1_enriched = layers.Add(name="t1_enriched")([feat1, c1_2, c1_3, c1_4])

    # TOWER 2 Enriched: T2 + Context from T1, T3, T4
    c2_1 = apply_cross_attention(feat2, feat1, "t2_from_t1")
    c2_3 = apply_cross_attention(feat2, feat3, "t2_from_t3")
    c2_4 = apply_cross_attention(feat2, feat4, "t2_from_t4")
    feat2_enriched = layers.Add(name="t2_enriched")([feat2, c2_1, c2_3, c2_4])

    # TOWER 3 Enriched: T3 + Context from T1, T2, T4
    c3_1 = apply_cross_attention(feat3, feat1, "t3_from_t1")
    c3_2 = apply_cross_attention(feat3, feat2, "t3_from_t2")
    c3_4 = apply_cross_attention(feat3, feat4, "t3_from_t4")
    feat3_enriched = layers.Add(name="t3_enriched")([feat3, c3_1, c3_2, c3_4])

    # TOWER 4 Enriched: T4 + Context from T1, T2, T3
    c4_1 = apply_cross_attention(feat4, feat1, "t4_from_t1")
    c4_2 = apply_cross_attention(feat4, feat2, "t4_from_t2")
    c4_3 = apply_cross_attention(feat4, feat3, "t4_from_t3")
    feat4_enriched = layers.Add(name="t4_enriched")([feat4, c4_1, c4_2, c4_3])

    in_meta = Input(shape=(5,), name="in_meta")
    feat_meta = layers.Dense(64, activation='relu')(in_meta)

    # --- CONCATENATION ---
    # We concatenate the ENRICHED features, not the raw ones
    concat = layers.Concatenate()([feat1_enriched, feat2_enriched, feat3_enriched, feat4_enriched, feat_meta])

    # Stabilized SE Block
    se_filters = 2112 // 16

    # Reduction (Squeeze)
    se = layers.Dense(se_filters, kernel_initializer='he_normal', use_bias=False)(concat)
    se = layers.BatchNormalization()(se) # BN added here to stabilize reduction
    se = layers.Activation('swish')(se)

    # Expansion (Excitation)
    se = layers.Dense(2112, kernel_initializer='he_normal')(se)
    se = layers.Activation('sigmoid')(se) # Sigmoid must be last for gating

    fused_weighted = layers.Multiply()([concat, se])

    # Applied "Dense -> BN -> Swish -> Dropout" pattern
    # Layer 1
    z = layers.Dense(1024, kernel_initializer='he_normal', use_bias=False)(fused_weighted)
    z = layers.BatchNormalization()(z)
    z = layers.Activation('swish')(z)
    z = layers.Dropout(0.3)(z)

    # Layer 2
    z = layers.Dense(512, kernel_initializer='he_normal', use_bias=False)(z)
    z = layers.BatchNormalization()(z)
    z = layers.Activation('swish')(z)
    z = layers.Dropout(0.3)(z)

    final_out = layers.Dense(num_classes, activation='softmax', name="final_output")(z)

    model = Model(
        inputs=[in1, in2, in3, in4, in_meta],
        outputs=[final_out, aux1, aux2, aux3, aux4],
        name="v1.0_Quad_Expert_System"
    )
    return model

# ==========================================
# 4. CUSTOM CALLBACK FOR SAFE SAVING
# ==========================================
class ContinuousHistorySaver(tf.keras.callbacks.Callback):
    def __init__(self, filepath, previous_history=None):
        super().__init__()
        self.filepath = filepath
        # If we have old history, start with that. Otherwise, empty dict.
        self.history_storage = previous_history if previous_history else {}

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        # Update storage with the latest metrics from this epoch
        for k, v in logs.items():
            if k not in self.history_storage:
                self.history_storage[k] = []
            self.history_storage[k].append(v)

        # Save immediately to Drive
        try:
            with open(self.filepath, 'wb') as f:
                pickle.dump(self.history_storage, f)
        except Exception as e:
            print(f"\n Warning: Could not save history for epoch {epoch}: {e}")

# ==========================================
# 5. TRAINING LOOP
# ==========================================
def run_phase_1_quad(save_dir):
    CHECKPOINT_DIR = os.path.join(save_dir, 'checkpoints_v1.0')
    os.makedirs(CHECKPOINT_DIR, exist_ok=True)
    print(f" Saving checkpoints to: {CHECKPOINT_DIR}")

    # Since the Generator now guarantees a balanced stream (1:1:1:1:1),
    # Set the loss weights to be equal [1.0] for all classes.
    alpha_balanced = np.array([1.0, 1.0, 1.0, 1.0, 1.0], dtype=float)
    print("\n Balanced Alphas:", alpha_balanced)

    # --- FRESH METRIC INSTANCES ---
    # We create a function to generate FRESH metrics for each head
    def get_metrics(name_prefix):
        return [
            'accuracy',
            tf.keras.metrics.AUC(name=f'{name_prefix}_auc'),
            tf.keras.metrics.F1Score(average='macro', name=f'{name_prefix}_f1')
        ]

    # oversample_ratio is technically not used in the new calculation formula,
    # But we pass > 0 to trigger the "balanced" sampling mode logic.

    # 2. Pass Weights to Generator
    train_gen = QuadStreamGenerator(
        X_train_global, X_train_local, X_train_metadata, y_train,
        batch_size=12, augment=True,
        oversample_ratio=1 # Triggers the balanced sampling
    )

    val_gen = QuadStreamGenerator(
        X_val_global, X_val_local, X_val_metadata, y_val,
        batch_size=12, augment=False,
        oversample_ratio=0.0 # Validation must remain natural distribution
    )

    # Resume Logic
    initial_epoch = 0
    model = None
    existing_checkpoints = glob.glob(os.path.join(CHECKPOINT_DIR, "epoch_*.keras"))

    if existing_checkpoints:
        def get_epoch_num(filepath):
            match = re.search(r"epoch_(\d+).keras", filepath)
            return int(match.group(1)) if match else -1
        latest_ckpt = max(existing_checkpoints, key=get_epoch_num)
        print(f" Found checkpoint! Resuming from: {latest_ckpt}")
        try:
            model = tf.keras.models.load_model(
                latest_ckpt,
                # Ensure Gamma=2.5 is passed when loading custom object
                custom_objects={'focal_loss_fixed': class_weighted_focal_loss(gamma=2.5, alpha_weights=alpha_balanced)}
            )
            found_epoch = get_epoch_num(latest_ckpt)
            initial_epoch = found_epoch
            print(f" Resuming training at Epoch {initial_epoch + 1}")
        except Exception as e:
            print(f" Error loading checkpoint {e}. Starting fresh.")
            model = None

    if model is None:
        print(" Starting v1.0 training from scratch...")
        model = create_quad_model()

        # Compile with Gamma=2.5
        # We pass the alpha_final list computed above
        loss_fn = class_weighted_focal_loss(gamma=2.5, alpha_weights=alpha_balanced)
        loss_list = [loss_fn for _ in range(5)]
        loss_weights_list = [1.0, 0.1, 0.1, 0.1, 0.1]

        # Apply specific metrics to specific outputs using a Dictionary
        # This is much safer than a list of lists
        metrics_dict = {
            'final_output': get_metrics('final'),
            'aux_t1_global_gamma': get_metrics('aux1'),
            'aux_t2_global_clahe': get_metrics('aux2'),
            'aux_t3_local_sigmoid': get_metrics('aux3'),
            'aux_t4_local_clahe': get_metrics('aux4')
        }

        model.compile(
            optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
            loss=loss_list,
            loss_weights=loss_weights_list,
            metrics=metrics_dict,
            run_eagerly=False
        )
    else:
        print(" Optimizer state restored from checkpoint (Compilation skipped).")

    # --- 5. HISTORY RETRIEVAL ---
    # We load history NOW to find the previous best score
    history_path = os.path.join(save_dir, 'phase1_v1.0_history.pkl')
    previous_best_f1 = -np.inf
    old_history = {}

    if os.path.exists(history_path):
        print(" Found existing history file. Scanning for previous best F1...")
        try:
            with open(history_path, 'rb') as f:
                old_history = pickle.load(f)

            target_metric = 'val_final_output_final_f1'
            if target_metric in old_history and len(old_history[target_metric]) > 0:
                previous_best_f1 = max(old_history[target_metric])
                print(f" Previous Best Val F1 was: {previous_best_f1:.4f}")
            else:
                print(" Metric not found in history. Assuming -inf.")
        except Exception as e:
            print(f" Could not read history file: {e}")

    # --- 6. DEFINE CALLBACKS WITH MEMORY ---

    # Manually create the checkpoint callback so we can modify it
    best_ckpt = tf.keras.callbacks.ModelCheckpoint(
        filepath=os.path.join(save_dir, 'phase1_v1.0_best.keras'),
        save_best_only=True,
        monitor='val_final_output_final_f1',
        verbose=1,
        mode='max'
    )

    # INJECT THE MEMORY
    # If we found a previous best, we force the callback to respect it.
    if previous_best_f1 > -np.inf:
        best_ckpt.best = previous_best_f1
        print(f" Injected previous best F1 ({previous_best_f1:.4f}) into ModelCheckpoint.")

    # Continuous History Saver
    # We pass 'old_history' so it continues appending to it, rather than starting fresh
    history_saver = ContinuousHistorySaver(filepath=history_path, previous_history=old_history)

    callbacks = [
        tf.keras.callbacks.EarlyStopping(
            monitor='val_final_output_final_f1', # Note: Name changed due to get_metrics()
            patience=6,
            restore_best_weights=True,
            mode='max'
        ),

        tf.keras.callbacks.ReduceLROnPlateau(
            monitor='val_final_output_final_f1',
            factor=0.2,
            patience=2,
            min_lr=1e-6,
            mode='max'
        ),

        tf.keras.callbacks.ModelCheckpoint(
            filepath=os.path.join(CHECKPOINT_DIR, "epoch_{epoch:02d}.keras"),
            save_best_only=False, verbose=1
        ),
        best_ckpt,
        history_saver   # Saves history.pkl every epoch
    ]

    print(f" Running Phase 1 v1.0 (Quad-Expert)...")

    # 3. Fit without class_weight arg
    history = model.fit(
        train_gen,
        validation_data=val_gen,
        initial_epoch=initial_epoch,
        epochs=30, # Increased epochs slightly as oversampling changes epoch dynamics
        callbacks=callbacks
    )

    print(f"\n Saving Final v1.0 Model to: {save_dir}")
    model.save(os.path.join(save_dir, 'phase1_v1.0_final.keras'))
    new_history = history.history

    if os.path.exists(history_path):
        print("found existing history, merging...")
        with open(history_path, 'rb') as f:
            old_history = pickle.load(f)

        # IMPROVED MERGE LOGIC:
        # Loop through NEW history to ensure we capture everything
        for key, val in new_history.items():
            if key in old_history:
                # Append new epochs to existing list
                old_history[key].extend(val)
            else:
                # If a new metric appeared (rare, but possible), add it
                old_history[key] = val

        final_history = old_history
    else:
        final_history = new_history

    # Save the combined history
    with open(history_path, 'wb') as f:
        pickle.dump(final_history, f)

    return model, history

# RUN
MY_DRIVE_PATH = '/content/drive/My Drive/ResNet50V2 model/v1.0'
os.makedirs(MY_DRIVE_PATH, exist_ok=True)

model, history = run_phase_1_quad(save_dir=MY_DRIVE_PATH)

###Phase 2 training (Unfreeze block 14 for fine-tuning)

In [ ]:
import os
import glob
import re
import pickle
import numpy as np
import cv2
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.utils import Sequence
from tensorflow.keras.applications.resnet_v2 import ResNet50V2, preprocess_input
from tensorflow.keras import layers, Model, Input
import albumentations as A

# ==========================================
# 1. RE-DEFINE CUSTOM OBJECTS (REQUIRED FOR LOADING)
# ==========================================
def class_weighted_focal_loss(gamma=2.5, alpha_weights=None):
    """
    gamma: Reduced to 2.5. Gamma 3.0 was too aggressive and suppressed gradients
           for "moderately hard" samples (like Luminal A), hurting their learning.
    """
    # Convert alpha to a tensor constant for efficiency
    if alpha_weights is not None:
        alpha_tensor = tf.constant(alpha_weights, dtype=tf.float32)
    else:
        alpha_tensor = None

    def focal_loss_fixed(y_true, y_pred):
        epsilon = tf.keras.backend.epsilon()
        y_pred = tf.clip_by_value(y_pred, epsilon, 1. - epsilon)

        # Step 1: Get probability of TRUE class only
        # y_true: (batch, 5), y_pred: (batch, 5)
        y_true = tf.cast(y_true, tf.float32)

        # p_t = probability assigned to the true class for each sample
        # Shape: (batch_size,)
        pt = tf.reduce_sum(y_true * y_pred, axis=-1)  # sum(y_true_i * y_pred_i) = p_true_class

        # Step 2: Basic cross entropy for true class
        ce = -tf.math.log(pt + epsilon)  # Shape: (batch_size,)

        # Step 3: Focal weighting based on how confident we are in TRUE class
        focal_weight = tf.math.pow(1 - pt, gamma)  # Shape: (batch_size,)

        # 3. Calculate Class Weight Term
        # Multiply by alpha corresponding to the true class
        if alpha_tensor is not None:
            # Get alpha for the true class of each sample
            # y_true * alpha gives (batch, 5) with alpha only at true class position
            # Then sum to get shape (batch_size,)
            alpha_t = tf.reduce_sum(y_true * alpha_tensor, axis=-1)
            loss = alpha_t * focal_weight * ce
        else:
            loss = focal_weight * ce

        return tf.reduce_mean(loss)  # Mean over batch

    return focal_loss_fixed

# ==========================================
# 2. RE-DEFINE GENERATOR
# ==========================================
class QuadStreamGenerator(Sequence):
    def __init__(self, x_global, x_local, x_meta, y, batch_size,
                 augment=False, shuffle=True,
                 oversample_ratio=0.0):
        super().__init__()
        self.x_global = x_global
        self.x_local = x_local
        self.x_meta = x_meta
        self.y = y
        self.batch_size = batch_size
        self.augment = augment
        self.shuffle = shuffle

        # Oversampling Setup
        self.oversample_ratio = oversample_ratio
        self.indexes = np.arange(len(self.x_global))

        # Calculate Sampling Probabilities if oversampling is enabled
        self.sampling_probs = None
        if self.oversample_ratio > 0:
            self.sampling_probs = self._calculate_sampling_probs()

        # --- ENHANCEMENT OBJECTS ---
        self.clahe_mild = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
        self.clahe_strong = cv2.createCLAHE(clipLimit=4.0, tileGridSize=(8,8))
        self.gamma_mild_table = self._build_gamma_table(0.6)
        self.gamma_strong_table = self._build_gamma_table(1.5)

        # --- SPECIALIZED AUGMENTATION PIPELINES ---
        # Using Albumentations to treat each class biologically correctly
        if self.augment:
            self.aug_pipelines = self._build_augmentation_pipelines()

        self.on_epoch_end()

    def _build_augmentation_pipelines(self):
        # 1. Base Geometric (Safe for all)
        # We use additional_targets to apply SAME transform to all 4 towers
        base_aug = [
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.5),
            A.Affine(scale=(0.9, 1.1), translate_percent=(0.1, 0.1), rotate=(-20, 20), p=0.5),
        ]

        # 2. Define Class-Specific Pipelines
        pipelines = {}

        # --- Class 0: BENIGN ---
        pipelines[0] = A.Compose(base_aug + [
            A.MedianBlur(blur_limit=5, p=0.3),
        ], additional_targets={'image2': 'image', 'image3': 'image', 'image4': 'image'})

        # --- Class 1: LUMINAL A ---
        pipelines[1] = A.Compose(base_aug + [
             A.ISONoise(color_shift=(0.01, 0.05), intensity=(0.1, 0.5), p=0.2),
        ], additional_targets={'image2': 'image', 'image3': 'image', 'image4': 'image'})

        # --- Class 2: LUMINAL B ---
        pipelines[2] = A.Compose(base_aug + [
            # 1. Geometry: Create the spicule shape physically
            # Removed alpha_affine to fix warning
            A.ElasticTransform(alpha=1, sigma=50, p=0.5),

            # 2. Structure: Highlight the edges (MILD settings)
            A.Emboss(alpha=(0.2, 0.5), strength=(0.3, 0.5), p=0.4),
        ], additional_targets={'image2': 'image', 'image3': 'image', 'image4': 'image'})

        # --- Class 3: HER2+ ---
        pipelines[3] = A.Compose(base_aug + [
            A.Sharpen(alpha=(0.2, 0.5), lightness=(0.5, 1.0), p=0.5),
        ], additional_targets={'image2': 'image', 'image3': 'image', 'image4': 'image'})

        # --- Class 4: TN ---
        pipelines[4] = A.Compose(base_aug + [
            # 1. Geometry: Create the "Bulge" / Mass Effect
            A.GridDistortion(num_steps=5, distort_limit=0.3, p=0.5),

            # 2. Structure: Define the sharp margin (MILD)
            A.Sharpen(alpha=(0.1, 0.3), lightness=(0.5, 1.0), p=0.4),

            # 3. Density: High Density (Dark/Bright shift)
            # Safe Range: (85, 115)
            A.RandomGamma(gamma_limit=(85, 115), p=0.5),
        ], additional_targets={'image2': 'image', 'image3': 'image', 'image4': 'image'})

        return pipelines

    def _calculate_sampling_probs(self):
        # 1. Get integer labels
        y_int = np.argmax(self.y, axis=1)
        classes, counts = np.unique(y_int, return_counts=True)
        # Calculate weight using Inverse Frequency (1.0 / count)
        # This ensures every class has roughly EQUAL probability of being picked per batch.
        class_weight_map = {c: 1.0/count for c, count in zip(classes, counts)}

        sample_weights = np.array([class_weight_map[label] for label in y_int])

        # Normalize to sum to 1.0 for np.random.choice
        return sample_weights / sample_weights.sum()

    def _build_gamma_table(self, gamma):
        invGamma = 1.0 / gamma
        return np.array([((i / 255.0) ** invGamma) * 255 for i in np.arange(0, 256)]).astype("uint8")

    def apply_gamma(self, img, table):
        return cv2.LUT(img, table)

    def apply_sigmoid(self, img, gain, cutoff):
        img_f = img.astype(np.float32) / 255.0
        sigmoid = 1.0 / (1.0 + np.exp(-gain * (img_f - cutoff)))
        return (sigmoid * 255).astype(np.uint8)

    def __len__(self):
        return int(np.ceil(len(self.x_global) / self.batch_size))

    def on_epoch_end(self):
        # Logic to handle Weighted Random Sampling
        if self.sampling_probs is not None and self.shuffle:
            # Sample WITH REPLACEMENT based on calculated probabilities
            # This creates a new list of indexes where minority classes appear more often
            self.indexes = np.random.choice(
                np.arange(len(self.x_global)),
                size=len(self.x_global),
                p=self.sampling_probs,
                replace=True
            )
        elif self.shuffle:
            np.random.shuffle(self.indexes)

    def __getitem__(self, index):
        idxs = self.indexes[index*self.batch_size:(index+1)*self.batch_size]

        s1, s2, s3, s4 = [], [], [], []
        bm = self.x_meta[idxs]
        by = self.y[idxs]

        for i in range(len(idxs)):
            # Identify Class for Specialization
            # You must extract the integer label (0-4) to pick the right pipeline
            current_label_idx = np.argmax(by[i])

            # Load raw images
            ig_raw = self.x_global[idxs[i]]
            il_raw = self.x_local[idxs[i]]

            if ig_raw.ndim == 3: ig = ig_raw[:,:,0].astype(np.uint8)
            else: ig = ig_raw.astype(np.uint8)

            if il_raw.ndim == 3: il = il_raw[:,:,0].astype(np.uint8)
            else: il = il_raw.astype(np.uint8)

            # --- PREPARE STACKS ---
            # Tower 1: Global Gamma
            stack_1 = np.stack([
                ig,
                self.apply_gamma(ig, self.gamma_mild_table),
                self.apply_gamma(ig, self.gamma_strong_table)
            ], axis=-1)

            # Tower 2: Global CLAHE
            stack_2 = np.stack([
                ig,
                self.clahe_mild.apply(ig),
                self.clahe_strong.apply(ig)
            ], axis=-1)

            # Tower 3: Local Sigmoid
            stack_3 = np.stack([
                il,
                self.apply_sigmoid(il, gain=8, cutoff=0.75),
                self.apply_sigmoid(il, gain=15, cutoff=0.60)
            ], axis=-1)

            # Tower 4: Local CLAHE
            stack_4 = np.stack([
                il,
                self.clahe_mild.apply(il),
                self.clahe_strong.apply(il)
            ], axis=-1)

            # --- APPLY CLASS-SPECIFIC AUGMENTATION ---
            if self.augment:
                # Select the correct pipeline for this specific image's class
                pipeline = self.aug_pipelines[current_label_idx]

                # Apply to all 4 towers simultaneously to keep them geometrically aligned
                # Albumentations 'image' is primary, 'image2/3/4' are additional
                augmented = pipeline(
                    image=stack_1,
                    image2=stack_2,
                    image3=stack_3,
                    image4=stack_4
                )

                stack_1 = augmented['image']
                stack_2 = augmented['image2']
                stack_3 = augmented['image3']
                stack_4 = augmented['image4']

            s1.append(preprocess_input(stack_1.astype(np.float32)))
            s2.append(preprocess_input(stack_2.astype(np.float32)))
            s3.append(preprocess_input(stack_3.astype(np.float32)))
            s4.append(preprocess_input(stack_4.astype(np.float32)))

        inputs = (np.array(s1), np.array(s2), np.array(s3), np.array(s4), np.array(bm))

        # Targets must be a TUPLE (), not a List []
        targets = (by, by, by, by, by)

        return inputs, targets

# ==========================================
# 3. HELPER: UNFREEZE LOGIC
# ==========================================
def unfreeze_resnet_blocks(model, target_block_str="conv5_block3"):
    """
    Unfreezes the top layers of ResNet50V2.
    ResNet50V2 has stages: conv1, conv2, conv3, conv4, conv5.
    'conv5' is the final stage.
    """
    print(f"\n Unfreezing '{target_block_str}' layers in all ResNet towers...")

    unfrozen_count = 0

    # Iterate through the main model layers to find the Xception Bases
    for layer in model.layers:
        # Look for our ResNet base models
        if "resnet" in layer.name and isinstance(layer, Model):
            print(f"   -> Processing Tower: {layer.name}")

            layer.trainable = True

            # Now traverse the layers INSIDE the Xception Base
            for inner_layer in layer.layers:
                # Logic: Unfreeze the specific block (e.g. conv5)
                if target_block_str in inner_layer.name:
                    # CRITICAL: KEEP BATCH NORM FROZEN
                    if isinstance(inner_layer, layers.BatchNormalization):
                        inner_layer.trainable = False
                    else:
                        inner_layer.trainable = True
                        unfrozen_count += 1
                else:
                    inner_layer.trainable = False

    print(f" Total layers unfrozen across all towers: {unfrozen_count}")
    return model

# ==========================================
# 4. CUSTOM CALLBACK FOR SAFE SAVING
# ==========================================
class ContinuousHistorySaver(tf.keras.callbacks.Callback):
    def __init__(self, filepath, previous_history=None):
        super().__init__()
        self.filepath = filepath
        # If we have old history, start with that. Otherwise, empty dict.
        self.history_storage = previous_history if previous_history else {}

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        # Update storage with the latest metrics from this epoch
        for k, v in logs.items():
            if k not in self.history_storage:
                self.history_storage[k] = []
            self.history_storage[k].append(v)

        # Save immediately to Drive
        try:
            with open(self.filepath, 'wb') as f:
                pickle.dump(self.history_storage, f)
        except Exception as e:
            print(f"\n Warning: Could not save history for epoch {epoch}: {e}")

# ==========================================
# 5. PHASE 2 TRAINING LOOP
# ==========================================
def run_phase_2_finetune(save_dir, resume_from_dir):
    # Setup directories
    CHECKPOINT_DIR = os.path.join(save_dir, 'checkpoints_v1.0_phase2')
    os.makedirs(CHECKPOINT_DIR, exist_ok=True)
    print(f" Saving Phase 2 checkpoints to: {CHECKPOINT_DIR}")

    # Setup Alphas & Metrics (Same as before)
    alpha_balanced = np.array([1.0, 1.0, 1.0, 1.0, 1.0], dtype=float)
    def get_metrics(name_prefix):
        return [
            'accuracy',
            tf.keras.metrics.AUC(name=f'{name_prefix}_auc'),
            tf.keras.metrics.F1Score(average='macro', name=f'{name_prefix}_f1')
        ]

    # --- LOAD DATA ---
    print(" Initializing Generators...")
    train_gen = QuadStreamGenerator(
        X_train_global, X_train_local, X_train_metadata, y_train,
        batch_size=12, augment=True, oversample_ratio=1
    )
    val_gen = QuadStreamGenerator(
        X_val_global, X_val_local, X_val_metadata, y_val,
        batch_size=12, augment=False, oversample_ratio=0.0
    )

    # --- SMART RESUMPTION LOGIC ---
    initial_epoch = 0
    model = None

    # 1. Check if we have interrupted Phase 2 checkpoints
    phase2_checkpoints = glob.glob(os.path.join(CHECKPOINT_DIR, "p2_epoch_*.keras"))

    if phase2_checkpoints:
        # Helper to find the highest epoch number
        def get_epoch_num(filepath):
            match = re.search(r"p2_epoch_(\d+).keras", filepath)
            return int(match.group(1)) if match else -1

        latest_ckpt = max(phase2_checkpoints, key=get_epoch_num)
        print(f" Found interrupted Phase 2 training! Resuming from: {latest_ckpt}")

        try:
            model = tf.keras.models.load_model(
                latest_ckpt,
                custom_objects={'focal_loss_fixed': class_weighted_focal_loss(gamma=2.5, alpha_weights=alpha_balanced)}
            )
            found_epoch = get_epoch_num(latest_ckpt)
            initial_epoch = found_epoch
            print(f" Jumping ahead to Epoch {initial_epoch + 1}")
        except Exception as e:
            print(f" Error loading checkpoint {e}. Falling back to start.")
            model = None

    # 2. If no Phase 2 checkpoint found (or error), Start Fresh from Phase 1 Model
    if model is None:
        print(" No Phase 2 progress found. Starting Fine-Tuning from scratch...")

        phase1_path = os.path.join(resume_from_dir, 'phase1_v1.0_final.keras')
        if not os.path.exists(phase1_path):
            phase1_path = os.path.join(resume_from_dir, 'phase1_v1.0_best.keras')

        if not os.path.exists(phase1_path):
            raise FileNotFoundError(" Could not find Phase 1 model to fine-tune!")

        print(f" Loading Phase 1 Model from: {phase1_path}")
        model = tf.keras.models.load_model(
            phase1_path,
            custom_objects={'focal_loss_fixed': class_weighted_focal_loss(gamma=2.5, alpha_weights=alpha_balanced)}
        )

        # --- UNFREEZE & COMPILE ---
        # Unfreeze the very top block of ResNet50V2
        model = unfreeze_resnet_blocks(model, target_block_str="conv5_block3")

        fine_tune_lr = 1e-6
        print(f" Recompiling with Fine-Tuning LR: {fine_tune_lr}")

        loss_fn = class_weighted_focal_loss(gamma=2.5, alpha_weights=alpha_balanced)
        metrics_dict = {
            'final_output': get_metrics('final'),
            'aux_t1_global_gamma': get_metrics('aux1'),
            'aux_t2_global_clahe': get_metrics('aux2'),
            'aux_t3_local_sigmoid': get_metrics('aux3'),
            'aux_t4_local_clahe': get_metrics('aux4')
        }

        model.compile(
            optimizer=tf.keras.optimizers.Adam(learning_rate=fine_tune_lr),
            loss=[loss_fn] * 5,
            loss_weights=[1.0, 0.1, 0.1, 0.1, 0.1],
            metrics=metrics_dict
        )
    else:
        print(" Optimizer state restored from checkpoint (Compilation skipped).")

    # --- HISTORY HANDLING ---
    phase1_hist_path = os.path.join(resume_from_dir, 'phase1_v1.0_history.pkl')
    phase2_hist_path = os.path.join(save_dir, 'phase2_v1.0_history.pkl')

    previous_best_f1 = -np.inf
    if os.path.exists(phase1_hist_path):
        with open(phase1_hist_path, 'rb') as f:
            p1_hist = pickle.load(f)
            if 'val_final_output_final_f1' in p1_hist:
                previous_best_f1 = max(p1_hist['val_final_output_final_f1'])

    # Load existing Phase 2 history if available
    current_p2_history = {}
    if os.path.exists(phase2_hist_path):
        with open(phase2_hist_path, 'rb') as f:
            current_p2_history = pickle.load(f)

    # --- CALLBACKS ---
    best_ckpt = tf.keras.callbacks.ModelCheckpoint(
        filepath=os.path.join(save_dir, 'phase2_v1.0_best.keras'),
        save_best_only=True,
        monitor='val_final_output_final_f1',
        verbose=1,
        mode='max'
    )
    if previous_best_f1 > -np.inf:
        best_ckpt.best = previous_best_f1

    history_saver = ContinuousHistorySaver(filepath=phase2_hist_path, previous_history=current_p2_history)

    callbacks = [
        tf.keras.callbacks.EarlyStopping(monitor='val_final_output_final_f1', patience=8, restore_best_weights=True, mode='max'),
        tf.keras.callbacks.ReduceLROnPlateau(monitor='val_final_output_final_f1', factor=0.2, patience=3, min_lr=1e-7, mode='max'),
        tf.keras.callbacks.ModelCheckpoint(filepath=os.path.join(CHECKPOINT_DIR, "p2_epoch_{epoch:02d}.keras"), save_best_only=False, verbose=1),
        best_ckpt,
        history_saver
    ]

    print(f" Running Phase 2 (Fine-Tuning) starting from Epoch {initial_epoch + 1}...")

    history = model.fit(
        train_gen,
        validation_data=val_gen,
        initial_epoch=initial_epoch,
        epochs=20,
        callbacks=callbacks
    )

    print(f"\n Saving Final Phase 2 Model to: {save_dir}")
    model.save(os.path.join(save_dir, 'phase2_v1.0_final.keras'))

    return model, history

# ==========================================
# 6. RUN EXECUTION
# ==========================================
MY_DRIVE_PATH = '/content/drive/My Drive/ResNet50V2 model/v1.17'

model_p2, history_p2 = run_phase_2_finetune(save_dir=MY_DRIVE_PATH, resume_from_dir=MY_DRIVE_PATH)

#XI. Check augmentation

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import cv2
import albumentations as A
from tensorflow.keras.utils import Sequence
from tensorflow.keras.applications.xception import preprocess_input

# ==========================================
# 1. DEFINE GENERATOR
# ==========================================
class QuadStreamGenerator(Sequence):
    def __init__(self, x_global, x_local, x_meta, y, batch_size,
                 augment=False, shuffle=True,
                 oversample_mode='none'):
        super().__init__()
        self.x_global = x_global
        self.x_local = x_local
        self.x_meta = x_meta
        self.y = y
        self.batch_size = batch_size
        self.augment = augment
        self.shuffle = shuffle
        self.oversample_mode = oversample_mode
        self.indexes = np.arange(len(self.x_global))

        # Calculate Sampling Probabilities
        self.sampling_probs = None
        if self.oversample_mode != 'none':
            self.sampling_probs = self._calculate_sampling_probs()

        # --- ENHANCEMENT OBJECTS ---
        self.clahe_mild = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
        self.clahe_strong = cv2.createCLAHE(clipLimit=4.0, tileGridSize=(8,8))
        self.gamma_mild_table = self._build_gamma_table(0.6)
        self.gamma_strong_table = self._build_gamma_table(1.5)

        if self.augment:
            self.aug_pipelines = self._build_augmentation_pipelines()

        self.on_epoch_end()

    def _build_augmentation_pipelines(self):
        base_aug = [
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.5),
            A.Affine(scale=(0.9, 1.1), translate_percent=(0.1, 0.1), rotate=(-20, 20), p=0.5),
        ]

        pipelines = {}

        # --- Class 0: BENIGN ---
        pipelines[0] = A.Compose(base_aug + [
            A.MedianBlur(blur_limit=5, p=0.3),
        ], additional_targets={'image2': 'image', 'image3': 'image', 'image4': 'image'})

        # --- Class 1: LUMINAL A ---
        # Switched to ISO_Noise to fix 'var_limit' warning and improve stability
        pipelines[1] = A.Compose(base_aug + [
             A.ISONoise(color_shift=(0.01, 0.05), intensity=(0.1, 0.5), p=0.2),
        ], additional_targets={'image2': 'image', 'image3': 'image', 'image4': 'image'})

        # --- Class 2: LUMINAL B ---
        pipelines[2] = A.Compose(base_aug + [
            # 1. Geometry: Create the spicule shape physically
            # Removed alpha_affine to fix warning
            A.ElasticTransform(alpha=1, sigma=50, p=0.5),

            # 2. Structure: Highlight the edges (MILD settings)
            A.Emboss(alpha=(0.2, 0.5), strength=(0.3, 0.5), p=0.4),
        ], additional_targets={'image2': 'image', 'image3': 'image', 'image4': 'image'})

        # --- Class 3: HER2+ ---
        pipelines[3] = A.Compose(base_aug + [
            A.Sharpen(alpha=(0.2, 0.5), lightness=(0.5, 1.0), p=0.5),
        ], additional_targets={'image2': 'image', 'image3': 'image', 'image4': 'image'})

        # --- Class 4: TN ---
        pipelines[4] = A.Compose(base_aug + [
            # 1. Geometry: Create the "Bulge" / Mass Effect
            A.GridDistortion(num_steps=5, distort_limit=0.3, p=0.5),

            # 2. Structure: Define the sharp margin (MILD)
            A.Sharpen(alpha=(0.1, 0.3), lightness=(0.5, 1.0), p=0.4),

            # 3. Density: High Density (Dark/Bright shift)
            # Safe Range: (85, 115)
            A.RandomGamma(gamma_limit=(85, 115), p=0.5),
        ], additional_targets={'image2': 'image', 'image3': 'image', 'image4': 'image'})

        return pipelines

    def _calculate_sampling_probs(self):
        y_int = np.argmax(self.y, axis=1)
        classes, counts = np.unique(y_int, return_counts=True)
        # Uniform 1:1:1:1:1
        class_weight_map = {c: 1.0/count for c, count in zip(classes, counts)}
        sample_weights = np.array([class_weight_map[label] for label in y_int])
        return sample_weights / sample_weights.sum()

    def _build_gamma_table(self, gamma):
        invGamma = 1.0 / gamma
        return np.array([((i / 255.0) ** invGamma) * 255 for i in np.arange(0, 256)]).astype("uint8")

    def apply_gamma(self, img, table):
        return cv2.LUT(img, table)

    def apply_sigmoid(self, img, gain, cutoff):
        img_f = img.astype(np.float32) / 255.0
        sigmoid = 1.0 / (1.0 + np.exp(-gain * (img_f - cutoff)))
        return (sigmoid * 255).astype(np.uint8)

    def __len__(self):
        return int(np.ceil(len(self.x_global) / self.batch_size))

    def on_epoch_end(self):
        if self.sampling_probs is not None and self.shuffle:
            self.indexes = np.random.choice(
                np.arange(len(self.x_global)),
                size=len(self.x_global),
                p=self.sampling_probs,
                replace=True
            )
        elif self.shuffle:
            np.random.shuffle(self.indexes)

    def __getitem__(self, index):
        # Dummy for Vis
        return None, None

# ==========================================
# 2. VISUALIZATION LOGIC
# ==========================================
CLASS_NAMES = {
    0: "Benign (Blur)",
    1: "Luminal A (Noise)",
    2: "Luminal B (Elastic/Emboss)",
    3: "HER2+ (Sharpen)",
    4: "Triple Neg (Grid/Sharpen/Gamma)"
}

def visualize_augmentations(X_global, X_local, X_meta, y_data):
    print("Initializing Generator for Visualization...")
    # 1. Instantiate Generator
    gen = QuadStreamGenerator(
        X_global, X_local, X_meta, y_data,
        batch_size=1,
        augment=True,
        oversample_mode='none' # Vis doesn't need sampling
    )

    fig, axes = plt.subplots(5, 2, figsize=(12, 22))
    plt.subplots_adjust(hspace=0.4)

    for class_idx in range(5):
        print(f"Processing Class {class_idx}...")
        indices = np.where(np.argmax(y_data, axis=1) == class_idx)[0]

        if len(indices) == 0:
            print(f"Skipping Class {class_idx}: No samples found.")
            continue

        sample_idx = indices[0]

        # A. Load Raw Images
        ig_raw = X_global[sample_idx]
        il_raw = X_local[sample_idx]

        if ig_raw.ndim == 3: ig = ig_raw[:,:,0].astype(np.uint8)
        else: ig = ig_raw.astype(np.uint8)

        if il_raw.ndim == 3: il = il_raw[:,:,0].astype(np.uint8)
        else: il = il_raw.astype(np.uint8)

        # B. Prepare Stacks
        stack_1 = np.stack([
            ig,
            gen.apply_gamma(ig, gen.gamma_mild_table),
            gen.apply_gamma(ig, gen.gamma_strong_table)
        ], axis=-1)

        stack_2 = np.stack([
            ig,
            gen.clahe_mild.apply(ig),
            gen.clahe_strong.apply(ig)
        ], axis=-1)

        stack_3 = np.stack([
            il,
            gen.apply_sigmoid(il, gain=8, cutoff=0.75),
            gen.apply_sigmoid(il, gain=15, cutoff=0.60)
        ], axis=-1)

        stack_4 = np.stack([
            il,
            gen.clahe_mild.apply(il),
            gen.clahe_strong.apply(il)
        ], axis=-1)

        # C. Apply Class-Specific Pipeline
        pipeline = gen.aug_pipelines[class_idx]
        augmented = pipeline(
            image=stack_1,
            image2=stack_2,
            image3=stack_3,
            image4=stack_4
        )

        # Retrieve augmented (Viewing Channel 0 is fine for shape check)
        aug_global = augmented['image'][:,:,0]
        aug_local = augmented['image3'][:,:,0]

        # Column 0: Augmented Global
        ax_g = axes[class_idx, 0]
        ax_g.imshow(aug_global, cmap='gray')
        ax_g.set_title(f"{CLASS_NAMES[class_idx]} - GLOBAL", fontsize=12, fontweight='bold')
        ax_g.axis('off')

        # Column 1: Augmented Local
        ax_l = axes[class_idx, 1]
        ax_l.imshow(aug_local, cmap='gray')
        ax_l.set_title(f"{CLASS_NAMES[class_idx]} - LOCAL", fontsize=12, fontweight='bold')
        ax_l.axis('off')

    print(" Visualization Complete.")
    plt.show()

# ==========================================
# 3. RATIO CHECK LOGIC
# ==========================================
def verify_ratio(X_global, X_local, X_meta, y_data):
    print("\n Running UNIFORM Ratio Stress Test...")

    # Init Generator in Uniform Mode
    gen = QuadStreamGenerator(
        X_global, X_local, X_meta, y_data,
        batch_size=32,
        augment=False,
        oversample_mode='balanced_uniform'
    )

    # Simulate 50 batches
    class_counts = {0:0, 1:0, 2:0, 3:0, 4:0}
    total = 0

    print("Sampling 50 batches...")
    for i in range(50):
        # Manually invoke sampling logic since __getitem__ is dummy in this script
        idxs = gen.indexes[i*32 : (i+1)*32]
        if len(idxs) == 0: break # End of epoch

        batch_y = gen.y[idxs]
        y_ints = np.argmax(batch_y, axis=1)
        unique, counts = np.unique(y_ints, return_counts=True)
        for u, c in zip(unique, counts):
            class_counts[u] += c
        total += len(batch_y)

    print("-" * 30)
    print(f"{'Class':<10} | {'Actual %':<10} | {'Target %':<10}")
    print("-" * 30)
    for c in sorted(class_counts.keys()):
        perc = (class_counts[c] / total) * 100
        target = "20.0%" # Uniform Target
        print(f"{c:<10} | {perc:.1f}%      | {target}")
    print("-" * 30)

# RUN BOTH CHECKS
visualize_augmentations(X_train_global, X_train_local, X_train_metadata, y_train)
verify_ratio(X_train_global, X_train_local, X_train_metadata, y_train)

#XII. Xception Evaluation

##Cleaning history file

In [ ]:
import pickle
import pandas as pd

# ==========================================
# 1. LOAD HISTORY DATA
# ==========================================
history_path = '/content/drive/MyDrive/Xception model/v1.0/phase1_v1.0_history.pkl'

print(f" Loading training history from: {history_path}\n")

with open(history_path, 'rb') as f:
    loaded_history = pickle.load(f)

# ==========================================
# 2. VERIFY METRIC LENGTHS
# ==========================================
print("=== Total Epochs Logged Per Metric ===")
for key, values in loaded_history.items():
    print(f"{key}: {len(values)} epochs")
print("\n")

# ==========================================
# 3. EPOCH-BY-EPOCH BREAKDOWN
# ==========================================
# Extracting the main metrics for a readable table
try:
    main_metrics = {
        'Train Acc': loaded_history['final_output_accuracy'],
        'Val Acc': loaded_history['val_final_output_accuracy'],
        'Train Loss': loaded_history['final_output_loss'],
        'Val Loss': loaded_history['val_final_output_loss'],
        'Train F1': loaded_history['final_output_final_f1'],
        'Val F1': loaded_history['val_final_output_final_f1']
    }

    # Convert to Pandas DataFrame for a clean visual table
    df = pd.DataFrame(main_metrics)
    df.index.name = 'Epoch Index'

    # Configure Pandas to show all rows without truncating
    pd.set_option('display.max_rows', None)

    print("=== Epoch-by-Epoch Data (Main Head) ===")
    print(df)

except KeyError as e:
    print(f" Could not build table. Missing key: {e}")

In [ ]:
import pickle
import os

# ==========================================
# 1. SETUP PATHS
# ==========================================
history_path = '/content/drive/MyDrive/Xception model/v1.0/phase1_v1.0_history.pkl'
backup_path = '/content/drive/MyDrive/Xception model/v1.0/phase1_v1.0_history_BACKUP.pkl'

print(f" Loading history from: {history_path}")

with open(history_path, 'rb') as f:
    history_data = pickle.load(f)

# ==========================================
# 2. CREATE BACKUP
# ==========================================
if not os.path.exists(backup_path):
    with open(backup_path, 'wb') as f:
        pickle.dump(history_data, f)
    print(" Backup created successfully.")

# ==========================================
# 3. SLICE THE ARRAYS
# ==========================================
fixed_history = {}
target_epochs = 30

for metric, values in history_data.items():
    # Keep only the first 30 elements (index 0 to 29)
    fixed_history[metric] = values[:target_epochs]

print(f" Sliced all metrics down to exactly {target_epochs} epochs.")

# ==========================================
# 4. OVERWRITE THE FILE
# ==========================================
with open(history_path, 'wb') as f:
    pickle.dump(fixed_history, f)

print(" Finish cleaning history file!")

##Acc, loss, AUC and F1 plotting v1.0

###Phase 1 model

In [ ]:
import pickle
import matplotlib.pyplot as plt

# ==========================================
# 1. LOAD HISTORY DATA
# ==========================================
# Corrected path based on your training script
history_path = '/content/drive/MyDrive/Xception model/v1.0/phase1_v1.0_history.pkl'

print(f" Loading training history from: {history_path}")

with open(history_path, 'rb') as f:
    loaded_history = pickle.load(f)

# Print keys to verify the exact names of your metrics
print(f" Available metrics in history: \n{list(loaded_history.keys())}")

# ==========================================
# 2. SETUP METRIC KEYS
# ==========================================
# Pulling metrics specifically for the main "final_output" head
acc_key = 'final_output_accuracy'
val_acc_key = 'val_final_output_accuracy'

loss_key = 'final_output_loss'
val_loss_key = 'val_final_output_loss'

auc_key = 'final_output_final_auc'
val_auc_key = 'val_final_output_final_auc'

f1_key = 'final_output_final_f1'
val_f1_key = 'val_final_output_final_f1'

# ==========================================
# 3. PLOT GRAPHS
# ==========================================
# Create a 2x2 grid for the 4 metrics
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# --- Plot 1: Accuracy ---
axes[0, 0].plot(loaded_history[acc_key], label='Train', color='blue')
axes[0, 0].plot(loaded_history[val_acc_key], label='Validation', color='orange')
axes[0, 0].set_title('Main Head Accuracy', fontsize=14)
axes[0, 0].set_ylabel('Accuracy')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylim(0.0, 1.0) # Locked range
axes[0, 0].legend(loc='lower right')
axes[0, 0].grid(True, linestyle='--', alpha=0.6)

# --- Plot 2: Loss ---
axes[0, 1].plot(loaded_history[loss_key], label='Train', color='blue')
axes[0, 1].plot(loaded_history[val_loss_key], label='Validation', color='orange')
axes[0, 1].set_title('Main Head Loss', fontsize=14)
axes[0, 1].set_ylabel('Loss')
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylim(bottom=0.0) # Locked bottom, dynamic top
axes[0, 1].legend(loc='upper right')
axes[0, 1].grid(True, linestyle='--', alpha=0.6)

# --- Plot 3: AUC ---
axes[1, 0].plot(loaded_history[auc_key], label='Train', color='blue')
axes[1, 0].plot(loaded_history[val_auc_key], label='Validation', color='orange')
axes[1, 0].set_title('Main Head AUC', fontsize=14)
axes[1, 0].set_ylabel('AUC')
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylim(0.0, 1.0) # Locked range
axes[1, 0].legend(loc='lower right')
axes[1, 0].grid(True, linestyle='--', alpha=0.6)

# --- Plot 4: F1 Score ---
axes[1, 1].plot(loaded_history[f1_key], label='Train', color='blue')
axes[1, 1].plot(loaded_history[val_f1_key], label='Validation', color='orange')
axes[1, 1].set_title('Main Head F1-Score', fontsize=14)
axes[1, 1].set_ylabel('F1-Score')
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylim(0.0, 1.0) # Locked range
axes[1, 1].legend(loc='lower right')
axes[1, 1].grid(True, linestyle='--', alpha=0.6)

# Adjust layout so titles and labels don't overlap
plt.tight_layout()
plt.show()

###Phase 2 model (fine-tuning)

In [ ]:
import pickle
import matplotlib.pyplot as plt

# ==========================================
# 1. LOAD HISTORY DATA
# ==========================================
# Corrected path based on your training script
history_path = '/content/drive/MyDrive/Xception model/v1.0/phase2_v1.0_history.pkl'

print(f" Loading training history from: {history_path}")

with open(history_path, 'rb') as f:
    loaded_history = pickle.load(f)

# Print keys to verify the exact names of your metrics
print(f" Available metrics in history: \n{list(loaded_history.keys())}")

# ==========================================
# 2. SETUP METRIC KEYS
# ==========================================
# Pulling metrics specifically for the main "final_output" head
acc_key = 'final_output_accuracy'
val_acc_key = 'val_final_output_accuracy'

loss_key = 'final_output_loss'
val_loss_key = 'val_final_output_loss'

auc_key = 'final_output_final_auc'
val_auc_key = 'val_final_output_final_auc'

f1_key = 'final_output_final_f1'
val_f1_key = 'val_final_output_final_f1'

# ==========================================
# 3. PLOT GRAPHS
# ==========================================
# Create a 2x2 grid for the 4 metrics
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# --- Plot 1: Accuracy ---
axes[0, 0].plot(loaded_history[acc_key], label='Train', color='blue')
axes[0, 0].plot(loaded_history[val_acc_key], label='Validation', color='orange')
axes[0, 0].set_title('Main Head Accuracy', fontsize=14)
axes[0, 0].set_ylabel('Accuracy')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylim(0.0, 1.0) # Locked range
axes[0, 0].legend(loc='lower right')
axes[0, 0].grid(True, linestyle='--', alpha=0.6)

# --- Plot 2: Loss ---
axes[0, 1].plot(loaded_history[loss_key], label='Train', color='blue')
axes[0, 1].plot(loaded_history[val_loss_key], label='Validation', color='orange')
axes[0, 1].set_title('Main Head Loss', fontsize=14)
axes[0, 1].set_ylabel('Loss')
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylim(bottom=0.0) # Locked bottom, dynamic top
axes[0, 1].legend(loc='upper right')
axes[0, 1].grid(True, linestyle='--', alpha=0.6)

# --- Plot 3: AUC ---
axes[1, 0].plot(loaded_history[auc_key], label='Train', color='blue')
axes[1, 0].plot(loaded_history[val_auc_key], label='Validation', color='orange')
axes[1, 0].set_title('Main Head AUC', fontsize=14)
axes[1, 0].set_ylabel('AUC')
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylim(0.0, 1.0) # Locked range
axes[1, 0].legend(loc='lower right')
axes[1, 0].grid(True, linestyle='--', alpha=0.6)

# --- Plot 4: F1 Score ---
axes[1, 1].plot(loaded_history[f1_key], label='Train', color='blue')
axes[1, 1].plot(loaded_history[val_f1_key], label='Validation', color='orange')
axes[1, 1].set_title('Main Head F1-Score', fontsize=14)
axes[1, 1].set_ylabel('F1-Score')
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylim(0.0, 1.0) # Locked range
axes[1, 1].legend(loc='lower right')
axes[1, 1].grid(True, linestyle='--', alpha=0.6)

# Adjust layout so titles and labels don't overlap
plt.tight_layout()
plt.show()

##Confusion matrix v1.0

####Phase 1 model

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_fscore_support, accuracy_score, roc_auc_score
from tensorflow.keras.applications.xception import preprocess_input

# ==========================================
# 1. LOAD DATA
# ==========================================
# Set your path
files_path = '/content/drive/MyDrive/CMMD_preprocessed/v1.0'

print(f" Loading data from: {files_path}")

# Load Test Data
X_test_global   = np.load(os.path.join(files_path, 'X_test_global.npy'))
X_test_local    = np.load(os.path.join(files_path, 'X_test_local.npy'))
X_test_metadata = np.load(os.path.join(files_path, 'X_test_metadata.npy'))
y_test          = np.load(os.path.join(files_path, 'y_test.npy'))

print(f" Loaded Test Data: Global {X_test_global.shape}, Local {X_test_local.shape}, Meta {X_test_metadata.shape}")

# ==========================================
# 2. HELPER FUNCTIONS
# ==========================================
def build_gamma_table(gamma):
    invGamma = 1.0 / gamma
    return np.array([((i / 255.0) ** invGamma) * 255 for i in np.arange(0, 256)]).astype("uint8")

def apply_gamma(img, table):
    return cv2.LUT(img, table)

def apply_sigmoid(img, gain, cutoff):
    img_f = img.astype(np.float32) / 255.0
    sigmoid = 1.0 / (1.0 + np.exp(-gain * (img_f - cutoff)))
    return (sigmoid * 255).astype(np.uint8)

# Initialize Tables/CLAHE exactly as in training
gamma_mild_table = build_gamma_table(0.6)
gamma_strong_table = build_gamma_table(1.5)
clahe_mild = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
clahe_strong = cv2.createCLAHE(clipLimit=4.0, tileGridSize=(8,8))

# ==========================================
# 3. PREPROCESSING FUNCTION
# ==========================================
def preprocess_test_data_quad(x_global, x_local, x_meta):
    print(" Preprocessing Test Data for Quad-Stream Architecture...")

    s1_list, s2_list, s3_list, s4_list = [], [], [], []

    for i in range(len(x_global)):
        # --- Prepare Raw Images ---
        ig_raw = x_global[i]
        il_raw = x_local[i]

        if ig_raw.ndim == 3: ig = ig_raw[:,:,0].astype(np.uint8)
        else: ig = ig_raw.astype(np.uint8)

        if il_raw.ndim == 3: il = il_raw[:,:,0].astype(np.uint8)
        else: il = il_raw.astype(np.uint8)

        # --- Tower 1: Global Gamma ---
        stack_1 = np.stack([
            ig,
            apply_gamma(ig, gamma_mild_table),
            apply_gamma(ig, gamma_strong_table)
        ], axis=-1)

        # --- Tower 2: Global CLAHE ---
        stack_2 = np.stack([
            ig,
            clahe_mild.apply(ig),
            clahe_strong.apply(ig)
        ], axis=-1)

        # --- Tower 3: Local Sigmoid ---
        stack_3 = np.stack([
            il,
            apply_sigmoid(il, gain=8, cutoff=0.75),
            apply_sigmoid(il, gain=15, cutoff=0.60)
        ], axis=-1)

        # --- Tower 4: Local CLAHE ---
        stack_4 = np.stack([
            il,
            clahe_mild.apply(il),
            clahe_strong.apply(il)
        ], axis=-1)

        # --- Xception Preprocessing (-1 to 1) ---
        s1_list.append(preprocess_input(stack_1.astype(np.float32)))
        s2_list.append(preprocess_input(stack_2.astype(np.float32)))
        s3_list.append(preprocess_input(stack_3.astype(np.float32)))
        s4_list.append(preprocess_input(stack_4.astype(np.float32)))

    return [
        np.array(s1_list),
        np.array(s2_list),
        np.array(s3_list),
        np.array(s4_list),
        np.array(x_meta)
    ]

# Run Preprocessing
X_test_processed = preprocess_test_data_quad(X_test_global, X_test_local, X_test_metadata)

print(" Preprocessing Complete.")

# ==========================================
# 4. MODEL PREDICTION & EVALUATION
# ==========================================
model_path = '/content/drive/MyDrive/Xception model/v1.0/phase1_v1.0_final.keras'

if os.path.exists(model_path):
    print(f" Loading Model from: {model_path}")
    # For evaluation, we only need the weights to predict, not the loss function.
    model = tf.keras.models.load_model(model_path, compile=False)
else:
    raise FileNotFoundError(f" Model file not found at {model_path}")

print(" Making Predictions...")

# The model returns a list of 5 outputs: [Final, Aux1, Aux2, Aux3, Aux4]
# We only care about index 0 (Final Output)
all_preds_p1 = model.predict(X_test_processed, verbose=1)
y_pred_p1 = all_preds_p1[0]

# --- Calculate Global Metrics using Scikit-Learn ---
# Since we didn't compile the model, we calculate these manually instead of model.evaluate
y_true_indices = np.argmax(y_test, axis=1)
y_pred_indices = np.argmax(y_pred_p1, axis=1)

test_acc = accuracy_score(y_true_indices, y_pred_indices)
# Calculate AUC (Macro vs Weighted - usually Macro for multiclass balance check)
try:
    test_auc = roc_auc_score(y_test, y_pred_p1, average='macro', multi_class='ovr')
except ValueError:
    test_auc = 0.0 # Handle edge cases if class missing in test

print(f"\n Test Accuracy: {test_acc:.4f}")
print(f" Test AUC (Macro): {test_auc:.4f}")

# ==========================================
# 5. METRICS & VISUALIZATION
# ==========================================
predictions = y_pred_p1
truth = y_test

# 0:Benign, 1:LumA, 2:LumB, 3:Her2, 4:TN
target_names = ["Benign", "Luminal A", "Luminal B", "HER2", "TN"]

# A. Classification Report
print("\n=== Classification Report ===")
print(classification_report(np.argmax(truth, axis=1), np.argmax(predictions,axis=1), target_names=target_names, digits=4))
print('*****************')

# B. Sensitivity & Specificity Analysis
res = []
for l in range(len(target_names)):
    # Calculate metrics for class 'l' vs all other classes (One-vs-Rest)
    # pos_label=True ensures index 1 is the positive class (Sensitivity)
    prec, recall, _, _ = precision_recall_fscore_support(
        np.array(np.argmax(truth, axis=1)) == l,
        np.array(np.argmax(predictions, axis=1)) == l,
        pos_label=True,
        average=None
    )

    # recall[0] is Specificity (Recall of the Negative class)
    # recall[1] is Sensitivity (Recall of the Positive class)
    res.append([target_names[l], recall[1], recall[0]])

aa = pd.DataFrame(res, columns=['class', 'sensitivity', 'specificity'])

# Calculate Macro Averages for Sensitivity and Specificity
macro_sens = aa['sensitivity'].mean()
macro_spec = aa['specificity'].mean()

summary_row = pd.DataFrame({
    'class': ['Macro Avg'],
    'sensitivity': [macro_sens],
    'specificity': [macro_spec]
})

# Append the Macro Avg row to the bottom of the dataframe
aa = pd.concat([aa, summary_row], ignore_index=True)

print("\n=== Sensitivity & Specificity Report ===")
# Print the dataframe with 4 decimal places using string formatters
print(aa.to_string(
    formatters={
        'sensitivity': lambda x: f"{x:.4f}" if isinstance(x, float) else x,
        'specificity': lambda x: f"{x:.4f}" if isinstance(x, float) else x
    },
    index=False
))
print('*****************')

# C. Confusion Matrix
cmtx = confusion_matrix(np.argmax(truth, axis=1), np.argmax(predictions, axis=1))
print("\nConfusion Matrix (Raw Counts):")
print(cmtx)

# D. Plot Confusion Matrix
def plot_conf(cm, labels, figsize=(10,8)):
    fig, ax = plt.subplots(figsize=figsize)
    sns.heatmap(cm, annot=True, fmt='d', linewidths=.5, ax=ax, cmap='Blues')
    ax.set_title('v1.0 Quad-Stream Expert System\nConfusion Matrix', fontsize=14)
    ax.set_xlabel('\nPredicted Values', fontsize=12)
    ax.set_ylabel('Actual Values', fontsize=12)
    ax.xaxis.set_ticklabels(labels, rotation=45)
    ax.yaxis.set_ticklabels(labels, rotation=0)
    plt.show()

plot_conf(cmtx, target_names)

####Phase 2 model (fine-tuning)

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_fscore_support, accuracy_score, roc_auc_score
from tensorflow.keras.applications.xception import preprocess_input

# ==========================================
# 1. LOAD DATA
# ==========================================
# Set your path
files_path = '/content/drive/MyDrive/CMMD_preprocessed/v1.0'

print(f" Loading data from: {files_path}")

# Load Test Data
X_test_global   = np.load(os.path.join(files_path, 'X_test_global.npy'))
X_test_local    = np.load(os.path.join(files_path, 'X_test_local.npy'))
X_test_metadata = np.load(os.path.join(files_path, 'X_test_metadata.npy'))
y_test          = np.load(os.path.join(files_path, 'y_test.npy'))

print(f" Loaded Test Data: Global {X_test_global.shape}, Local {X_test_local.shape}, Meta {X_test_metadata.shape}")

# ==========================================
# 2. HELPER FUNCTIONS
# ==========================================
def build_gamma_table(gamma):
    invGamma = 1.0 / gamma
    return np.array([((i / 255.0) ** invGamma) * 255 for i in np.arange(0, 256)]).astype("uint8")

def apply_gamma(img, table):
    return cv2.LUT(img, table)

def apply_sigmoid(img, gain, cutoff):
    img_f = img.astype(np.float32) / 255.0
    sigmoid = 1.0 / (1.0 + np.exp(-gain * (img_f - cutoff)))
    return (sigmoid * 255).astype(np.uint8)

# Initialize Tables/CLAHE
gamma_mild_table = build_gamma_table(0.6)
gamma_strong_table = build_gamma_table(1.5)
clahe_mild = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
clahe_strong = cv2.createCLAHE(clipLimit=4.0, tileGridSize=(8,8))

# ==========================================
# 3. PREPROCESSING FUNCTION
# ==========================================
def preprocess_test_data_quad(x_global, x_local, x_meta):
    print(" Preprocessing Test Data for Quad-Stream Architecture...")

    s1_list, s2_list, s3_list, s4_list = [], [], [], []

    for i in range(len(x_global)):
        # --- Prepare Raw Images ---
        ig_raw = x_global[i]
        il_raw = x_local[i]

        if ig_raw.ndim == 3: ig = ig_raw[:,:,0].astype(np.uint8)
        else: ig = ig_raw.astype(np.uint8)

        if il_raw.ndim == 3: il = il_raw[:,:,0].astype(np.uint8)
        else: il = il_raw.astype(np.uint8)

        # --- Tower 1: Global Gamma ---
        stack_1 = np.stack([
            ig,
            apply_gamma(ig, gamma_mild_table),
            apply_gamma(ig, gamma_strong_table)
        ], axis=-1)

        # --- Tower 2: Global CLAHE ---
        stack_2 = np.stack([
            ig,
            clahe_mild.apply(ig),
            clahe_strong.apply(ig)
        ], axis=-1)

        # --- Tower 3: Local Sigmoid ---
        stack_3 = np.stack([
            il,
            apply_sigmoid(il, gain=8, cutoff=0.75),
            apply_sigmoid(il, gain=15, cutoff=0.60)
        ], axis=-1)

        # --- Tower 4: Local CLAHE ---
        stack_4 = np.stack([
            il,
            clahe_mild.apply(il),
            clahe_strong.apply(il)
        ], axis=-1)

        # --- Xception Preprocessing (-1 to 1) ---
        s1_list.append(preprocess_input(stack_1.astype(np.float32)))
        s2_list.append(preprocess_input(stack_2.astype(np.float32)))
        s3_list.append(preprocess_input(stack_3.astype(np.float32)))
        s4_list.append(preprocess_input(stack_4.astype(np.float32)))

    return [
        np.array(s1_list),
        np.array(s2_list),
        np.array(s3_list),
        np.array(s4_list),
        np.array(x_meta)
    ]

# Run Preprocessing
X_test_processed = preprocess_test_data_quad(X_test_global, X_test_local, X_test_metadata)

print(" Preprocessing Complete.")

# ==========================================
# 4. MODEL PREDICTION & EVALUATION
# ==========================================
model_path = '/content/drive/MyDrive/Xception model/v1.0/phase2_v1.0_final.keras'

if os.path.exists(model_path):
    print(f" Loading Model from: {model_path}")
    # For evaluation, we only need the weights to predict, not the loss function.
    model = tf.keras.models.load_model(model_path, compile=False)
else:
    raise FileNotFoundError(f" Model file not found at {model_path}")

print(" Making Predictions...")

# The model returns a list of 5 outputs: [Final, Aux1, Aux2, Aux3, Aux4]
# We only care about index 0 (Final Output)
all_preds_p2 = model.predict(X_test_processed, verbose=1)
y_pred_p2 = all_preds_p2[0]

# --- Calculate Global Metrics using Scikit-Learn ---
# Since we didn't compile the model, we calculate these manually instead of model.evaluate
y_true_indices = np.argmax(y_test, axis=1)
y_pred_indices = np.argmax(y_pred_p2, axis=1)

test_acc = accuracy_score(y_true_indices, y_pred_indices)
# Calculate AUC (Macro vs Weighted - usually Macro for multiclass balance check)
try:
    test_auc = roc_auc_score(y_test, y_pred_p2, average='macro', multi_class='ovr')
except ValueError:
    test_auc = 0.0 # Handle edge cases if class missing in test

print(f"\n Test Accuracy: {test_acc:.4f}")
print(f" Test AUC (Macro): {test_auc:.4f}")

# ==========================================
# 5. METRICS & VISUALIZATION
# ==========================================
predictions = y_pred_p2
truth = y_test

# 0:Benign, 1:LumA, 2:LumB, 3:Her2, 4:TN
target_names = ["Benign", "Luminal A", "Luminal B", "HER2", "TN"]

# A. Classification Report
print("\n=== Classification Report ===")
print(classification_report(np.argmax(truth, axis=1), np.argmax(predictions,axis=1), target_names=target_names))
print('*****************')

# B. Sensitivity & Specificity Analysis
res = []
for l in range(len(target_names)):
    # Calculate metrics for class 'l' vs all other classes (One-vs-Rest)
    # pos_label=True ensures index 1 is the positive class (Sensitivity)
    prec, recall, _, _ = precision_recall_fscore_support(
        np.array(np.argmax(truth, axis=1)) == l,
        np.array(np.argmax(predictions, axis=1)) == l,
        pos_label=True,
        average=None
    )

    # recall[0] is Specificity (Recall of the Negative class)
    # recall[1] is Sensitivity (Recall of the Positive class)
    res.append([target_names[l], recall[1], recall[0]])

aa = pd.DataFrame(res, columns=['class', 'sensitivity', 'specificity'])
print(aa)
print('*****************')

# C. Confusion Matrix
cmtx = confusion_matrix(np.argmax(truth, axis=1), np.argmax(predictions, axis=1))
print("\nConfusion Matrix (Raw Counts):")
print(cmtx)

# D. Plot Confusion Matrix
def plot_conf(cm, labels, figsize=(10,8)):
    fig, ax = plt.subplots(figsize=figsize)
    sns.heatmap(cm, annot=True, fmt='d', linewidths=.5, ax=ax, cmap='Blues')
    ax.set_title('v1.0 Quad-Stream Expert System\nConfusion Matrix', fontsize=14)
    ax.set_xlabel('\nPredicted Values', fontsize=12)
    ax.set_ylabel('Actual Values', fontsize=12)
    ax.xaxis.set_ticklabels(labels, rotation=45)
    ax.yaxis.set_ticklabels(labels, rotation=0)
    plt.show()

plot_conf(cmtx, target_names)

##Saving prediction v1.0

###Phase 1 model

In [ ]:
import os
import numpy as np

save_dir = '/content/drive/MyDrive/Xception model/v1.0/results'
os.makedirs(save_dir, exist_ok=True)

save_path = os.path.join(save_dir, 'v1.0_p1_test_predictions.npz')

print(f" Saving predictions to: {save_path}...")

# We use savez_compressed to store multiple arrays in one file
np.savez_compressed(
    save_path,
    y_pred=y_pred_p1,       # The probabilities
    y_test=y_test,       # The true labels
    target_names=target_names # The class names
)

print(" Saved successfully!")

###Phase 2 model (fine-tuning)

In [ ]:
import os
import numpy as np

save_dir = '/content/drive/MyDrive/Xception model/v1.0/results'
os.makedirs(save_dir, exist_ok=True)

save_path = os.path.join(save_dir, 'v1.0_p2_test_predictions.npz')

print(f" Saving predictions to: {save_path}...")

# We use savez_compressed to store multiple arrays in one file
np.savez_compressed(
    save_path,
    y_pred=y_pred_p2,       # The probabilities
    y_test=y_test,       # The true labels
    target_names=target_names # The class names
)

print(" Saved successfully!")

##ROC curves v1.0

###Phase 1 model

In [ ]:
# ==========================================
# 5. ROC / AUC VISUALIZATION
#    Loads from saved .npz file
# ==========================================
import os
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc
from itertools import cycle
from google.colab import drive

if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

print("\n Loading saved predictions and generating ROC Curves...")

# 1. Setup Variables (FROM FILE)
load_path = '/content/drive/MyDrive/Xception model/v1.0/results/v1.0_p1_test_predictions.npz'

if not os.path.exists(load_path):
    raise FileNotFoundError(f" Could not find the results file at: {load_path}\nPlease run the saving code block first.")

# Load the data
data = np.load(load_path)

truth = data['y_test']
predictions = data['y_pred']
target_names = list(data['target_names']) # Convert array back to list

print(f" Loaded {len(truth)} samples.")
print(f" Classes detected: {target_names}")

num_classes = len(target_names)

# 2. Compute ROC curve and ROC area for each class
fpr = dict()
tpr = dict()
roc_auc = dict()

# --- Micro-Average ROC ---
# Computes the metric globally by considering each element of the label indicator matrix as a label
fpr["micro"], tpr["micro"], _ = roc_curve(truth.ravel(), predictions.ravel())
roc_auc["micro"] = auc(fpr["micro"], tpr["micro"])

# --- Per-Class ROC ---
for i in range(num_classes):
    fpr[i], tpr[i], _ = roc_curve(truth[:, i], predictions[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])

# --- Macro-Average ROC ---
# Aggregates class-wise FPR/TPR by interpolating them on a common grid
all_fpr = np.unique(np.concatenate([fpr[i] for i in range(num_classes)]))

# Then interpolate all ROC curves at this points
mean_tpr = np.zeros_like(all_fpr)
for i in range(num_classes):
    mean_tpr += np.interp(all_fpr, fpr[i], tpr[i])

# Average it and compute AUC
mean_tpr /= num_classes

fpr["macro"] = all_fpr
tpr["macro"] = mean_tpr
roc_auc["macro"] = auc(fpr["macro"], tpr["macro"])

# 3. Plotting
plt.figure(figsize=(10, 8))
lw = 2 # Line width

# Plot Micro-Average
plt.plot(fpr["micro"], tpr["micro"],
         label='Micro-average ROC (area = {0:0.2f})'.format(roc_auc["micro"]),
         color='deeppink', linestyle=':', linewidth=4)

# Plot Macro-Average
plt.plot(fpr["macro"], tpr["macro"],
         label='Macro-average ROC (area = {0:0.2f})'.format(roc_auc["macro"]),
         color='navy', linestyle=':', linewidth=4)

# Plot Each Class
colors = cycle(["aqua", "darkorange", "cornflowerblue", "red", "green"])
for i, color in zip(range(num_classes), colors):
    plt.plot(fpr[i], tpr[i], color=color, lw=lw,
             label='ROC {0} (area = {1:0.2f})'.format(target_names[i], roc_auc[i]))

# Plot Diagonal (Chance)
plt.plot([0, 1], [0, 1], 'k--', lw=lw)

# Formatting
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Multi-Class ROC: v1.0 Quad Stream Model')
plt.legend(loc="lower right")
plt.grid(True, alpha=0.3)
plt.show()

###Phase 2 model

In [ ]:
# ==========================================
# 5. ROC / AUC VISUALIZATION
#    Loads from saved .npz file
# ==========================================
import os
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc
from itertools import cycle
from google.colab import drive

if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

print("\n Loading saved predictions and generating ROC Curves...")

# 1. Setup Variables (FROM FILE)
# This path matches where we saved it in the previous step
load_path = '/content/drive/MyDrive/Xception model/v1.0/results/v1.0_p2_test_predictions.npz'

if not os.path.exists(load_path):
    raise FileNotFoundError(f" Could not find the results file at: {load_path}\nPlease run the saving code block first.")

# Load the data
data = np.load(load_path)

truth = data['y_test']
predictions = data['y_pred']
target_names = list(data['target_names']) # Convert array back to list

print(f" Loaded {len(truth)} samples.")
print(f" Classes detected: {target_names}")

num_classes = len(target_names)

# 2. Compute ROC curve and ROC area for each class
fpr = dict()
tpr = dict()
roc_auc = dict()

# --- Micro-Average ROC ---
# Computes the metric globally by considering each element of the label indicator matrix as a label
fpr["micro"], tpr["micro"], _ = roc_curve(truth.ravel(), predictions.ravel())
roc_auc["micro"] = auc(fpr["micro"], tpr["micro"])

# --- Per-Class ROC ---
for i in range(num_classes):
    fpr[i], tpr[i], _ = roc_curve(truth[:, i], predictions[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])

# --- Macro-Average ROC ---
# Aggregates class-wise FPR/TPR by interpolating them on a common grid
all_fpr = np.unique(np.concatenate([fpr[i] for i in range(num_classes)]))

# Then interpolate all ROC curves at this points
mean_tpr = np.zeros_like(all_fpr)
for i in range(num_classes):
    mean_tpr += np.interp(all_fpr, fpr[i], tpr[i])

# Average it and compute AUC
mean_tpr /= num_classes

fpr["macro"] = all_fpr
tpr["macro"] = mean_tpr
roc_auc["macro"] = auc(fpr["macro"], tpr["macro"])

# 3. Plotting
plt.figure(figsize=(10, 8))
lw = 2 # Line width

# Plot Micro-Average
plt.plot(fpr["micro"], tpr["micro"],
         label='Micro-average ROC (area = {0:0.2f})'.format(roc_auc["micro"]),
         color='deeppink', linestyle=':', linewidth=4)

# Plot Macro-Average
plt.plot(fpr["macro"], tpr["macro"],
         label='Macro-average ROC (area = {0:0.2f})'.format(roc_auc["macro"]),
         color='navy', linestyle=':', linewidth=4)

# Plot Each Class
colors = cycle(["aqua", "darkorange", "cornflowerblue", "red", "green"])
for i, color in zip(range(num_classes), colors):
    plt.plot(fpr[i], tpr[i], color=color, lw=lw,
             label='ROC {0} (area = {1:0.2f})'.format(target_names[i], roc_auc[i]))

# Plot Diagonal (Chance)
plt.plot([0, 1], [0, 1], 'k--', lw=lw)

# Formatting
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Multi-Class ROC: v1.0 Quad Stream Model')
plt.legend(loc="lower right")
plt.grid(True, alpha=0.3)
plt.show()

#XIII. DenseNet121 Evaluation

##Confusion matrix v1.0

###Phase 1 model

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_fscore_support, accuracy_score, roc_auc_score
from tensorflow.keras.applications.densenet import preprocess_input

# ==========================================
# 1. LOAD DATA
# ==========================================
# Set your path
files_path = '/content/drive/MyDrive/CMMD_preprocessed/v1.0'

print(f" Loading data from: {files_path}")

# Load Test Data
X_test_global   = np.load(os.path.join(files_path, 'X_test_global.npy'))
X_test_local    = np.load(os.path.join(files_path, 'X_test_local.npy'))
X_test_metadata = np.load(os.path.join(files_path, 'X_test_metadata.npy'))
y_test          = np.load(os.path.join(files_path, 'y_test.npy'))

print(f" Loaded Test Data: Global {X_test_global.shape}, Local {X_test_local.shape}, Meta {X_test_metadata.shape}")

# ==========================================
# 2. HELPER FUNCTIONS
# ==========================================
def build_gamma_table(gamma):
    invGamma = 1.0 / gamma
    return np.array([((i / 255.0) ** invGamma) * 255 for i in np.arange(0, 256)]).astype("uint8")

def apply_gamma(img, table):
    return cv2.LUT(img, table)

def apply_sigmoid(img, gain, cutoff):
    img_f = img.astype(np.float32) / 255.0
    sigmoid = 1.0 / (1.0 + np.exp(-gain * (img_f - cutoff)))
    return (sigmoid * 255).astype(np.uint8)

# Initialize Tables/CLAHE exactly as in training
gamma_mild_table = build_gamma_table(0.6)
gamma_strong_table = build_gamma_table(1.5)
clahe_mild = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
clahe_strong = cv2.createCLAHE(clipLimit=4.0, tileGridSize=(8,8))

# ==========================================
# 3. PREPROCESSING FUNCTION
# ==========================================
def preprocess_test_data_quad(x_global, x_local, x_meta):
    print(" Preprocessing Test Data for Quad-Stream Architecture...")

    s1_list, s2_list, s3_list, s4_list = [], [], [], []

    for i in range(len(x_global)):
        # --- Prepare Raw Images ---
        ig_raw = x_global[i]
        il_raw = x_local[i]

        if ig_raw.ndim == 3: ig = ig_raw[:,:,0].astype(np.uint8)
        else: ig = ig_raw.astype(np.uint8)

        if il_raw.ndim == 3: il = il_raw[:,:,0].astype(np.uint8)
        else: il = il_raw.astype(np.uint8)

        # --- Tower 1: Global Gamma ---
        stack_1 = np.stack([
            ig,
            apply_gamma(ig, gamma_mild_table),
            apply_gamma(ig, gamma_strong_table)
        ], axis=-1)

        # --- Tower 2: Global CLAHE ---
        stack_2 = np.stack([
            ig,
            clahe_mild.apply(ig),
            clahe_strong.apply(ig)
        ], axis=-1)

        # --- Tower 3: Local Sigmoid ---
        stack_3 = np.stack([
            il,
            apply_sigmoid(il, gain=8, cutoff=0.75),
            apply_sigmoid(il, gain=15, cutoff=0.60)
        ], axis=-1)

        # --- Tower 4: Local CLAHE ---
        stack_4 = np.stack([
            il,
            clahe_mild.apply(il),
            clahe_strong.apply(il)
        ], axis=-1)

        # --- DenseNet Preprocessing ---
        s1_list.append(preprocess_input(stack_1.astype(np.float32)))
        s2_list.append(preprocess_input(stack_2.astype(np.float32)))
        s3_list.append(preprocess_input(stack_3.astype(np.float32)))
        s4_list.append(preprocess_input(stack_4.astype(np.float32)))

    return [
        np.array(s1_list),
        np.array(s2_list),
        np.array(s3_list),
        np.array(s4_list),
        np.array(x_meta)
    ]

# Run Preprocessing
X_test_processed = preprocess_test_data_quad(X_test_global, X_test_local, X_test_metadata)

print(" Preprocessing Complete.")

# ==========================================
# 4. MODEL PREDICTION & EVALUATION
# ==========================================
# Clear TensorFlow Memory
tf.keras.backend.clear_session()

model_path = '/content/drive/MyDrive/DenseNet121 model/v1.0/phase1_v1.0_final.keras'

if os.path.exists(model_path):
    print(f" Loading Model from: {model_path}")
    # For evaluation, we only need the weights to predict, not the loss function.
    model = tf.keras.models.load_model(model_path, compile=False)
else:
    raise FileNotFoundError(f" Model file not found at {model_path}")

print(" Making Predictions...")

# The model returns a list of 5 outputs: [Final, Aux1, Aux2, Aux3, Aux4]
# We only care about index 0 (Final Output)
all_preds_p1 = model.predict(X_test_processed, verbose=1)
y_pred_p1 = all_preds_p1[0]

# --- Calculate Global Metrics using Scikit-Learn ---
# (Since we didn't compile the model, we calculate these manually instead of model.evaluate)
y_true_indices = np.argmax(y_test, axis=1)
y_pred_indices = np.argmax(y_pred_p1, axis=1)

test_acc = accuracy_score(y_true_indices, y_pred_indices)
# Calculate AUC (Macro vs Weighted - usually Macro for multiclass balance check)
try:
    test_auc = roc_auc_score(y_test, y_pred_p1, average='macro', multi_class='ovr')
except ValueError:
    test_auc = 0.0 # Handle edge cases if class missing in test

print(f"\n Test Accuracy: {test_acc:.4f}")
print(f" Test AUC (Macro): {test_auc:.4f}")

# ==========================================
# 5. METRICS & VISUALIZATION
# ==========================================
predictions = y_pred_p1
truth = y_test

# 0:Benign, 1:LumA, 2:LumB, 3:Her2, 4:TN
target_names = ["Benign", "Luminal A", "Luminal B", "HER2", "TN"]

# A. Classification Report
print("\n=== Classification Report ===")
print(classification_report(np.argmax(truth, axis=1), np.argmax(predictions,axis=1), target_names=target_names, digits=4))
print('*****************')

# B. Sensitivity & Specificity Analysis
res = []
for l in range(len(target_names)):
    # Calculate metrics for class 'l' vs all other classes (One-vs-Rest)
    # pos_label=True ensures index 1 is the positive class (Sensitivity)
    prec, recall, _, _ = precision_recall_fscore_support(
        np.array(np.argmax(truth, axis=1)) == l,
        np.array(np.argmax(predictions, axis=1)) == l,
        pos_label=True,
        average=None
    )

    # recall[0] is Specificity (Recall of the Negative class)
    # recall[1] is Sensitivity (Recall of the Positive class)
    res.append([target_names[l], recall[1], recall[0]])

aa = pd.DataFrame(res, columns=['class', 'sensitivity', 'specificity'])

# Calculate Macro Averages for Sensitivity and Specificity
macro_sens = aa['sensitivity'].mean()
macro_spec = aa['specificity'].mean()

summary_row = pd.DataFrame({
    'class': ['Macro Avg'],
    'sensitivity': [macro_sens],
    'specificity': [macro_spec]
})

# Append the Macro Avg row to the bottom of the dataframe
aa = pd.concat([aa, summary_row], ignore_index=True)

print("\n=== Sensitivity & Specificity Report ===")
# Print the dataframe with 4 decimal places using string formatters
print(aa.to_string(
    formatters={
        'sensitivity': lambda x: f"{x:.4f}" if isinstance(x, float) else x,
        'specificity': lambda x: f"{x:.4f}" if isinstance(x, float) else x
    },
    index=False
))
print('*****************')

# C. Confusion Matrix
cmtx = confusion_matrix(np.argmax(truth, axis=1), np.argmax(predictions, axis=1))
print("\nConfusion Matrix (Raw Counts):")
print(cmtx)

# D. Plot Confusion Matrix
def plot_conf(cm, labels, figsize=(10,8)):
    fig, ax = plt.subplots(figsize=figsize)
    sns.heatmap(cm, annot=True, fmt='d', linewidths=.5, ax=ax, cmap='Blues')
    ax.set_title('v1.0 Quad-Stream Expert System\nConfusion Matrix', fontsize=14)
    ax.set_xlabel('\nPredicted Values', fontsize=12)
    ax.set_ylabel('Actual Values', fontsize=12)
    ax.xaxis.set_ticklabels(labels, rotation=45)
    ax.yaxis.set_ticklabels(labels, rotation=0)
    plt.show()

plot_conf(cmtx, target_names)

#XIV. ResNet50V2 Evaluation

##Confusion matrix v1.0

###Phase 1 model

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_fscore_support, accuracy_score, roc_auc_score
from tensorflow.keras.applications.resnet_v2 import preprocess_input

# ==========================================
# 1. LOAD DATA
# ==========================================
# Set your path
files_path = '/content/drive/MyDrive/CMMD_preprocessed/v1.0'

print(f" Loading data from: {files_path}")

# Load Test Data
X_test_global   = np.load(os.path.join(files_path, 'X_test_global.npy'))
X_test_local    = np.load(os.path.join(files_path, 'X_test_local.npy'))
X_test_metadata = np.load(os.path.join(files_path, 'X_test_metadata.npy'))
y_test          = np.load(os.path.join(files_path, 'y_test.npy'))

print(f" Loaded Test Data: Global {X_test_global.shape}, Local {X_test_local.shape}, Meta {X_test_metadata.shape}")

# ==========================================
# 2. HELPER FUNCTIONS
# ==========================================
def build_gamma_table(gamma):
    invGamma = 1.0 / gamma
    return np.array([((i / 255.0) ** invGamma) * 255 for i in np.arange(0, 256)]).astype("uint8")

def apply_gamma(img, table):
    return cv2.LUT(img, table)

def apply_sigmoid(img, gain, cutoff):
    img_f = img.astype(np.float32) / 255.0
    sigmoid = 1.0 / (1.0 + np.exp(-gain * (img_f - cutoff)))
    return (sigmoid * 255).astype(np.uint8)

# Initialize Tables/CLAHE exactly as in training
gamma_mild_table = build_gamma_table(0.6)
gamma_strong_table = build_gamma_table(1.5)
clahe_mild = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
clahe_strong = cv2.createCLAHE(clipLimit=4.0, tileGridSize=(8,8))

# ==========================================
# 3. PREPROCESSING FUNCTION
# ==========================================
def preprocess_test_data_quad(x_global, x_local, x_meta):
    print(" Preprocessing Test Data for Quad-Stream Architecture...")

    s1_list, s2_list, s3_list, s4_list = [], [], [], []

    for i in range(len(x_global)):
        # --- Prepare Raw Images ---
        ig_raw = x_global[i]
        il_raw = x_local[i]

        if ig_raw.ndim == 3: ig = ig_raw[:,:,0].astype(np.uint8)
        else: ig = ig_raw.astype(np.uint8)

        if il_raw.ndim == 3: il = il_raw[:,:,0].astype(np.uint8)
        else: il = il_raw.astype(np.uint8)

        # --- Tower 1: Global Gamma ---
        stack_1 = np.stack([
            ig,
            apply_gamma(ig, gamma_mild_table),
            apply_gamma(ig, gamma_strong_table)
        ], axis=-1)

        # --- Tower 2: Global CLAHE ---
        stack_2 = np.stack([
            ig,
            clahe_mild.apply(ig),
            clahe_strong.apply(ig)
        ], axis=-1)

        # --- Tower 3: Local Sigmoid ---
        stack_3 = np.stack([
            il,
            apply_sigmoid(il, gain=8, cutoff=0.75),
            apply_sigmoid(il, gain=15, cutoff=0.60)
        ], axis=-1)

        # --- Tower 4: Local CLAHE ---
        stack_4 = np.stack([
            il,
            clahe_mild.apply(il),
            clahe_strong.apply(il)
        ], axis=-1)

        # --- ResNet Preprocessing ---
        s1_list.append(preprocess_input(stack_1.astype(np.float32)))
        s2_list.append(preprocess_input(stack_2.astype(np.float32)))
        s3_list.append(preprocess_input(stack_3.astype(np.float32)))
        s4_list.append(preprocess_input(stack_4.astype(np.float32)))

    return [
        np.array(s1_list),
        np.array(s2_list),
        np.array(s3_list),
        np.array(s4_list),
        np.array(x_meta)
    ]

# Run Preprocessing
X_test_processed = preprocess_test_data_quad(X_test_global, X_test_local, X_test_metadata)

print(" Preprocessing Complete.")

# ==========================================
# 4. MODEL PREDICTION & EVALUATION
# ==========================================
# Clear memory before loading the new model
tf.keras.backend.clear_session()

model_path = '/content/drive/MyDrive/ResNet50V2 model/v1.0/phase1_v1.0_final.keras'

if os.path.exists(model_path):
    print(f" Loading Model from: {model_path}")
    # For evaluation, we only need the weights to predict, not the loss function.
    model = tf.keras.models.load_model(model_path, compile=False)
else:
    raise FileNotFoundError(f" Model file not found at {model_path}")

print(" Making Predictions...")

# The model returns a list of 5 outputs: [Final, Aux1, Aux2, Aux3, Aux4]
# We only care about index 0 (Final Output)
all_preds_p1 = model.predict(X_test_processed, verbose=1)
y_pred_p1 = all_preds_p1[0]

# --- Calculate Global Metrics using Scikit-Learn ---
# Since we didn't compile the model, we calculate these manually instead of model.evaluate
y_true_indices = np.argmax(y_test, axis=1)
y_pred_indices = np.argmax(y_pred_p1, axis=1)

test_acc = accuracy_score(y_true_indices, y_pred_indices)
# Calculate AUC (Macro vs Weighted - usually Macro for multiclass balance check)
try:
    test_auc = roc_auc_score(y_test, y_pred_p1, average='macro', multi_class='ovr')
except ValueError:
    test_auc = 0.0 # Handle edge cases if class missing in test

print(f"\n Test Accuracy: {test_acc:.4f}")
print(f" Test AUC (Macro): {test_auc:.4f}")

# ==========================================
# 5. METRICS & VISUALIZATION
# ==========================================
predictions = y_pred_p1
truth = y_test

# 0:Benign, 1:LumA, 2:LumB, 3:Her2, 4:TN
target_names = ["Benign", "Luminal A", "Luminal B", "HER2", "TN"]

# A. Classification Report
print("\n=== Classification Report ===")
print(classification_report(np.argmax(truth, axis=1), np.argmax(predictions,axis=1), target_names=target_names, digits=4))
print('*****************')

# B. Sensitivity & Specificity Analysis
res = []
for l in range(len(target_names)):
    # Calculate metrics for class 'l' vs all other classes (One-vs-Rest)
    # pos_label=True ensures index 1 is the positive class (Sensitivity)
    prec, recall, _, _ = precision_recall_fscore_support(
        np.array(np.argmax(truth, axis=1)) == l,
        np.array(np.argmax(predictions, axis=1)) == l,
        pos_label=True,
        average=None
    )

    # recall[0] is Specificity (Recall of the Negative class)
    # recall[1] is Sensitivity (Recall of the Positive class)
    res.append([target_names[l], recall[1], recall[0]])

aa = pd.DataFrame(res, columns=['class', 'sensitivity', 'specificity'])

# Calculate Macro Averages for Sensitivity and Specificity
macro_sens = aa['sensitivity'].mean()
macro_spec = aa['specificity'].mean()

summary_row = pd.DataFrame({
    'class': ['Macro Avg'],
    'sensitivity': [macro_sens],
    'specificity': [macro_spec]
})

# Append the Macro Avg row to the bottom of the dataframe
aa = pd.concat([aa, summary_row], ignore_index=True)

print("\n=== Sensitivity & Specificity Report ===")
# Print the dataframe with 4 decimal places using string formatters
print(aa.to_string(
    formatters={
        'sensitivity': lambda x: f"{x:.4f}" if isinstance(x, float) else x,
        'specificity': lambda x: f"{x:.4f}" if isinstance(x, float) else x
    },
    index=False
))
print('*****************')

# C. Confusion Matrix
cmtx = confusion_matrix(np.argmax(truth, axis=1), np.argmax(predictions, axis=1))
print("\nConfusion Matrix (Raw Counts):")
print(cmtx)

# D. Plot Confusion Matrix
def plot_conf(cm, labels, figsize=(10,8)):
    fig, ax = plt.subplots(figsize=figsize)
    sns.heatmap(cm, annot=True, fmt='d', linewidths=.5, ax=ax, cmap='Blues')
    ax.set_title('v1.0 Quad-Stream Expert System\nConfusion Matrix', fontsize=14)
    ax.set_xlabel('\nPredicted Values', fontsize=12)
    ax.set_ylabel('Actual Values', fontsize=12)
    ax.xaxis.set_ticklabels(labels, rotation=45)
    ax.yaxis.set_ticklabels(labels, rotation=0)
    plt.show()

plot_conf(cmtx, target_names)

#XV. GRAD-CAM++ on Images

##GRAD-CAM ++ v1.0

###Implementing GRAD-CAM++

In [ ]:
import os
import cv2
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow.keras.utils import Sequence
from tensorflow.keras.applications.xception import preprocess_input
from tensorflow.keras import layers, Model, Input

# ==========================================
# 1. GENERATOR
# ==========================================
class QuadStreamGenerator(Sequence):
    def __init__(self, x_global, x_local, x_meta, y, batch_size,
                 augment=False, shuffle=True, oversample_ratio=0.0):
        super().__init__()
        self.x_global = x_global
        self.x_local = x_local
        self.x_meta = x_meta
        self.y = y
        self.batch_size = batch_size
        self.indexes = np.arange(len(self.x_global))
        self.clahe_mild = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
        self.clahe_strong = cv2.createCLAHE(clipLimit=4.0, tileGridSize=(8,8))
        self.gamma_mild_table = self._build_gamma_table(0.6)
        self.gamma_strong_table = self._build_gamma_table(1.5)

    def _build_gamma_table(self, gamma):
        invGamma = 1.0 / gamma
        return np.array([((i / 255.0) ** invGamma) * 255 for i in np.arange(0, 256)]).astype("uint8")

    def apply_gamma(self, img, table):
        return cv2.LUT(img, table)

    def apply_sigmoid(self, img, gain, cutoff):
        img_f = img.astype(np.float32) / 255.0
        sigmoid = 1.0 / (1.0 + np.exp(-gain * (img_f - cutoff)))
        return (sigmoid * 255).astype(np.uint8)

    def __len__(self):
        return int(np.ceil(len(self.x_global) / self.batch_size))

    def __getitem__(self, index):
        idxs = self.indexes[index*self.batch_size:(index+1)*self.batch_size]
        s1, s2, s3, s4 = [], [], [], []
        bm = self.x_meta[idxs]
        by = self.y[idxs]

        for i in range(len(idxs)):
            ig_raw = self.x_global[idxs[i]]
            il_raw = self.x_local[idxs[i]]
            if ig_raw.ndim == 3: ig = ig_raw[:,:,0].astype(np.uint8)
            else: ig = ig_raw.astype(np.uint8)
            if il_raw.ndim == 3: il = il_raw[:,:,0].astype(np.uint8)
            else: il = il_raw.astype(np.uint8)

            stack_1 = np.stack([ig, self.apply_gamma(ig, self.gamma_mild_table), self.apply_gamma(ig, self.gamma_strong_table)], axis=-1)
            stack_2 = np.stack([ig, self.clahe_mild.apply(ig), self.clahe_strong.apply(ig)], axis=-1)
            stack_3 = np.stack([il, self.apply_sigmoid(il, gain=8, cutoff=0.75), self.apply_sigmoid(il, gain=15, cutoff=0.60)], axis=-1)
            stack_4 = np.stack([il, self.clahe_mild.apply(il), self.clahe_strong.apply(il)], axis=-1)

            s1.append(preprocess_input(stack_1.astype(np.float32)))
            s2.append(preprocess_input(stack_2.astype(np.float32)))
            s3.append(preprocess_input(stack_3.astype(np.float32)))
            s4.append(preprocess_input(stack_4.astype(np.float32)))

        inputs = (np.array(s1), np.array(s2), np.array(s3), np.array(s4), np.array(bm))
        return inputs, by

# ==========================================
# 2. MODEL DEFINITION (With Named Pooling Layers)
# ==========================================
def create_quad_model(num_classes=5):
    def build_tower(input_shape, name_suffix):
        inp = Input(shape=input_shape, name=f"in_{name_suffix}")
        base = tf.keras.applications.Xception(
            weights=None, include_top=False, input_shape=input_shape,
            name=f"xception_{name_suffix}"
        )
        base.trainable = False

        x = base(inp)

        # We name this layer so we can find its INPUT later
        x = layers.GlobalAveragePooling2D(name=f"gap_{name_suffix}")(x)

        x = layers.Dense(1024, kernel_initializer='he_normal', use_bias=False)(x)
        x = layers.BatchNormalization()(x)
        x = layers.Activation('swish')(x)
        x = layers.Dropout(0.5)(x)

        x = layers.Dense(512, kernel_initializer='he_normal', use_bias=False)(x)
        x = layers.BatchNormalization()(x)
        x = layers.Activation('swish')(x)
        x = layers.Dropout(0.5)(x)

        att = layers.Dense(512, activation='sigmoid', name=f"att_gate_{name_suffix}")(x)
        boost = layers.Multiply()([x, att])
        x_res = layers.Add(name=f"feat_{name_suffix}")([x, boost])

        aux_out = layers.Dense(num_classes, activation='softmax', name=f"aux_{name_suffix}")(x_res)
        return inp, x_res, aux_out

    def apply_cross_attention(source_feat, target_feat, suffix):
        context = layers.Dense(128, activation='relu', name=f"ctx_ex_{suffix}")(target_feat)
        query = layers.Dense(128, activation='relu', name=f"query_ex_{suffix}")(source_feat)
        concat_qk = layers.Concatenate()([query, context])
        attention_score = layers.Dense(512, activation='sigmoid', name=f"att_score_{suffix}")(concat_qk)
        context_scaled = layers.Dense(512, activation='relu', kernel_initializer='zeros', name=f"ctx_up_{suffix}")(context)
        gated_context = layers.Multiply()([context_scaled, attention_score])
        return gated_context

    in1, feat1, aux1 = build_tower((299,299,3), "t1_global_gamma")
    in2, feat2, aux2 = build_tower((299,299,3), "t2_global_clahe")
    in3, feat3, aux3 = build_tower((299,299,3), "t3_local_sigmoid")
    in4, feat4, aux4 = build_tower((299,299,3), "t4_local_clahe")

    c1_2, c1_3, c1_4 = apply_cross_attention(feat1, feat2, "t1_from_t2"), apply_cross_attention(feat1, feat3, "t1_from_t3"), apply_cross_attention(feat1, feat4, "t1_from_t4")
    feat1_enriched = layers.Add(name="t1_enriched")([feat1, c1_2, c1_3, c1_4])

    c2_1, c2_3, c2_4 = apply_cross_attention(feat2, feat1, "t2_from_t1"), apply_cross_attention(feat2, feat3, "t2_from_t3"), apply_cross_attention(feat2, feat4, "t2_from_t4")
    feat2_enriched = layers.Add(name="t2_enriched")([feat2, c2_1, c2_3, c2_4])

    c3_1, c3_2, c3_4 = apply_cross_attention(feat3, feat1, "t3_from_t1"), apply_cross_attention(feat3, feat2, "t3_from_t2"), apply_cross_attention(feat3, feat4, "t3_from_t4")
    feat3_enriched = layers.Add(name="t3_enriched")([feat3, c3_1, c3_2, c3_4])

    c4_1, c4_2, c4_3 = apply_cross_attention(feat4, feat1, "t4_from_t1"), apply_cross_attention(feat4, feat2, "t4_from_t2"), apply_cross_attention(feat4, feat3, "t4_from_t3")
    feat4_enriched = layers.Add(name="t4_enriched")([feat4, c4_1, c4_2, c4_3])

    in_meta = Input(shape=(5,), name="in_meta")
    feat_meta = layers.Dense(64, activation='relu')(in_meta)

    concat = layers.Concatenate()([feat1_enriched, feat2_enriched, feat3_enriched, feat4_enriched, feat_meta])

    se = layers.Dense(2112 // 16, kernel_initializer='he_normal', use_bias=False)(concat)
    se = layers.BatchNormalization()(se)
    se = layers.Activation('swish')(se)
    se = layers.Dense(2112, kernel_initializer='he_normal')(se)
    se = layers.Activation('sigmoid')(se)
    fused_weighted = layers.Multiply()([concat, se])

    z = layers.Dense(1024, kernel_initializer='he_normal', use_bias=False)(fused_weighted)
    z = layers.BatchNormalization()(z)
    z = layers.Activation('swish')(z)
    z = layers.Dropout(0.3)(z)

    z = layers.Dense(512, kernel_initializer='he_normal', use_bias=False)(z)
    z = layers.BatchNormalization()(z)
    z = layers.Activation('swish')(z)
    z = layers.Dropout(0.3)(z)

    final_out = layers.Dense(num_classes, activation='softmax', name="final_output")(z)

    model = Model(
        inputs=[in1, in2, in3, in4, in_meta],
        outputs=[final_out, aux1, aux2, aux3, aux4],
        name="v1.0_Quad_Expert_System"
    )
    return model

# ==========================================
# 3. GRAD-CAM++ FUNCTION
# ==========================================
def generate_grad_cam_plus_plus(model, inputs_list, label_idx, target_layer_name):
    # 1. FIND THE GAP LAYER
    # We construct the name of the GAP layer based on the tower name
    # e.g., 'xception_t1_global_gamma' -> 'gap_t1_global_gamma'
    gap_name = target_layer_name.replace("xception_", "gap_")

    try:
        gap_layer = model.get_layer(gap_name)
    except ValueError:
        print(f" Layer {gap_name} not found.")
        return np.zeros((299, 299))

    # 2. TARGET THE INPUT OF THE GAP LAYER
    # The input to GAP is the 4D Feature Map (Batch, 10, 10, 2048)
    target_tensor = gap_layer.input

    # 3. Identify Main Output
    if isinstance(model.output, list):
        main_output = model.output[0]
    else:
        main_output = model.output

    # 4. Build Gradient Model
    # We map Inputs -> [Conv Feature Map, Final Prediction]
    grad_model = tf.keras.models.Model(
        inputs=model.inputs,
        outputs=[target_tensor, main_output]
    )

    # 5. Prepare Inputs (List)
    if isinstance(inputs_list, tuple): inputs_list = list(inputs_list)
    data_tensors = [tf.convert_to_tensor(x, dtype=tf.float32) for x in inputs_list]

    # 6. Record Gradients (Using List Input to avoid Dictionary KeyError)
    with tf.GradientTape() as tape:
        tape.watch(data_tensors)
        # We pass the list directly. Since we rebuilt the model with list inputs, this works.
        results = grad_model(data_tensors)
        conv_outputs = results[0]
        predictions = results[1]
        loss = predictions[:, label_idx]

    # 7. Compute Gradients
    grads = tape.gradient(loss, conv_outputs)

    # Grad-CAM++ Logic
    grads_2 = tf.math.pow(grads, 2)
    grads_3 = tf.math.pow(grads, 3)
    sum_conv_outputs = tf.reduce_sum(conv_outputs, axis=(1, 2))

    alpha_denom = (2 * grads_2) + (sum_conv_outputs * grads_3)
    alpha_denom = tf.where(alpha_denom != 0.0, alpha_denom, tf.ones_like(alpha_denom))

    alphas = grads_2 / alpha_denom
    weights = tf.reduce_sum(alphas * tf.nn.relu(grads), axis=(1, 2))
    heatmap = tf.reduce_sum(weights[:, None, None, :] * conv_outputs, axis=-1)

    heatmap = tf.maximum(heatmap, 0)
    max_val = tf.reduce_max(heatmap)
    if max_val > 0: heatmap /= max_val

    return heatmap[0].numpy()

def plot_quad_gradcam(original_inputs, heatmaps, final_pred, true_label_vec):
    fig, axes = plt.subplots(2, 4, figsize=(24, 10))
    titles = ["T1: Global Gamma", "T2: Global CLAHE", "T3: Local Sigmoid", "T4: Local CLAHE"]
    class_names = ["Benign", "Luminal A", "Luminal B", "HER2+", "TN"]
    pred_idx = np.argmax(final_pred)
    true_idx = np.argmax(true_label_vec)
    pred_conf = final_pred[pred_idx] * 100
    title_color = 'green' if pred_idx == true_idx else 'red'

    fig.suptitle(f"Prediction: {class_names[pred_idx]} ({pred_conf:.1f}%) | True: {class_names[true_idx]}", fontsize=18, color=title_color, weight='bold')

    for i in range(4):
        img_array = original_inputs[i][0]
        img_disp = np.uint8((img_array + 1) * 127.5)
        axes[0, i].imshow(img_disp, cmap='gray')
        axes[0, i].set_title(f"{titles[i]}", fontsize=12)
        axes[0, i].axis('off')

        heatmap = heatmaps[i]
        heatmap_resized = cv2.resize(heatmap, (299, 299))
        heatmap_color = cv2.applyColorMap(np.uint8(255 * heatmap_resized), cv2.COLORMAP_JET)
        heatmap_color = cv2.cvtColor(heatmap_color, cv2.COLOR_BGR2RGB)
        if len(img_disp.shape) == 2: img_rgb = cv2.cvtColor(img_disp, cv2.COLOR_GRAY2RGB)
        else: img_rgb = img_disp
        superimposed = cv2.addWeighted(img_rgb, 0.6, heatmap_color, 0.4, 0)
        axes[1, i].imshow(superimposed)
        axes[1, i].set_title(f"Focus Region", fontsize=12)
        axes[1, i].axis('off')
    plt.tight_layout()
    plt.show()

# ==========================================
# 4. MAIN EXECUTION
# ==========================================
if __name__ == "__main__":
    print(" Initializing Generator...")
    val_gen = QuadStreamGenerator(
        X_val_global, X_val_local, X_val_metadata, y_val,
        batch_size=1, augment=False, shuffle=False
    )

    MODEL_PATH = '/content/drive/My Drive/Xception model/v1.0/phase2_v1.0_final.keras'
    print(f" Building Fresh Model Graph & Loading Weights from: {MODEL_PATH}")

    try:
        # 1. Build fresh graph
        model = create_quad_model(num_classes=5)

        # 2. Load Weights Only
        model.load_weights(MODEL_PATH)
        print(" Weights Loaded Successfully! Graph is healthy.")
    except Exception as e:
        print(f" Failed to load weights: {e}")
        exit()

    # 3. Run Grad-CAM++
    BATCH_INDEX = 0
    print(f"\n Running Grad-CAM++ on Batch #{BATCH_INDEX}...")

    inputs, targets = val_gen[BATCH_INDEX]

    # Predict
    preds = model.predict(inputs)
    final_pred = preds[0][0]
    predicted_class = np.argmax(final_pred)

    tower_names = [
        "xception_t1_global_gamma",
        "xception_t2_global_clahe",
        "xception_t3_local_sigmoid",
        "xception_t4_local_clahe"
    ]

    heatmaps = []
    for t_name in tower_names:
        print(f"   -> Processing Tower: {t_name}")
        hm = generate_grad_cam_plus_plus(model, inputs, predicted_class, t_name)
        heatmaps.append(hm)

    plot_quad_gradcam(inputs[:4], heatmaps, final_pred, targets[0])

###Class analysis

Note: Must run the script above first

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import math

# ==========================================
# 1. SEARCH FUNCTION
# ==========================================
def collect_true_positives(generator, model, num_per_class=2):
    """
    Scans the ENTIRE generator to find 'num_per_class' correctly classified examples.
    """
    total_batches = len(generator)
    print(f" Scanning full dataset ({total_batches} batches) for {num_per_class} True Positives per class...")

    # Storage: {class_idx: [ (inputs, prediction_probs), ... ] }
    found_examples = {i: [] for i in range(5)}
    classes_filled = 0
    class_names = ["Benign", "Luminal A", "Luminal B", "HER2+", "TN"]

    # Loop through ALL batches in the generator
    for i in range(total_batches):
        # Stop early if we have everything
        if classes_filled == 5:
            print(" Collection Complete! Found 2 examples for every class.")
            break

        # Get one batch
        inputs, targets = generator[i]

        # Predict
        preds = model.predict(inputs, verbose=0)

        # Get True and Pred classes
        true_idx = np.argmax(targets[0])
        pred_idx = np.argmax(preds[0])

        # Check if Correct (True Positive)
        if true_idx == pred_idx:
            # Check if we still need examples for this class
            if len(found_examples[true_idx]) < num_per_class:
                found_examples[true_idx].append((inputs, preds[0]))

                count = len(found_examples[true_idx])
                print(f"   -> Found TP for {class_names[true_idx]} ({count}/{num_per_class}) - Batch {i}")

                # Check if this class is now full
                if count == num_per_class:
                    classes_filled += 1

    # Report if we missed any
    if classes_filled < 5:
        print("\n Warning: Scanned entire dataset but could not fill all classes.")
        for cls_idx in range(5):
            if len(found_examples[cls_idx]) < num_per_class:
                print(f"   - Missing {num_per_class - len(found_examples[cls_idx])} cases for {class_names[cls_idx]}")

    return found_examples

# ==========================================
# 2. BATCH VISUALIZER
# ==========================================
def visualize_class_report(found_examples, model):
    class_names = ["Benign", "Luminal A", "Luminal B", "HER2+", "TN"]
    tower_names = [
        "xception_t1_global_gamma",
        "xception_t2_global_clahe",
        "xception_t3_local_sigmoid",
        "xception_t4_local_clahe"
    ]
    titles = ["Global Gamma", "Global CLAHE", "Local Sigmoid", "Local CLAHE"]

    # Loop through each class (0 to 4)
    for class_idx in range(5):
        examples = found_examples[class_idx]

        if not examples:
            print(f" No True Positives found for {class_names[class_idx]}. Skipping.")
            continue

        print(f"\n{'='*60}")
        print(f" ANALYSIS: CLASS {class_idx} - {class_names[class_idx].upper()}")
        print(f"{'='*60}")

        # Loop through the examples found for this class
        for case_num, (inputs, pred_probs) in enumerate(examples):
            print(f"   Generating heatmaps for Case #{case_num + 1}...")

            # Generate Heatmaps
            heatmaps = []
            for t_name in tower_names:
                hm = generate_grad_cam_plus_plus(model, inputs, class_idx, t_name)
                heatmaps.append(hm)

            # --- PLOTTING ---
            fig, axes = plt.subplots(2, 4, figsize=(20, 9))

            # Access [0] for batch dim, then [class_idx] for class
            # pred_probs shape is (1, 5), so we need pred_probs[0][class_idx]
            try:
                conf = float(pred_probs[0][class_idx]) * 100
            except IndexError:
                # Fallback if shape is flat (5,)
                conf = float(pred_probs[class_idx]) * 100

            # Main Title
            plt.suptitle(f"True Positive: {class_names[class_idx]} | Case #{case_num+1} | Confidence: {conf:.1f}%",
                         fontsize=16, weight='bold', color='navy')

            for i in range(4):
                # 1. Recover Image
                img_array = inputs[i][0]
                img_disp = np.uint8((img_array + 1) * 127.5)

                # Top Row: Input
                axes[0, i].imshow(img_disp, cmap='gray')
                axes[0, i].set_title(f"Input: {titles[i]}", fontsize=10)
                axes[0, i].axis('off')

                # Bottom Row: Heatmap Overlay
                heatmap = heatmaps[i]
                heatmap_resized = cv2.resize(heatmap, (299, 299))
                heatmap_color = cv2.applyColorMap(np.uint8(255 * heatmap_resized), cv2.COLORMAP_JET)
                heatmap_color = cv2.cvtColor(heatmap_color, cv2.COLOR_BGR2RGB)

                if len(img_disp.shape) == 2: img_rgb = cv2.cvtColor(img_disp, cv2.COLOR_GRAY2RGB)
                else: img_rgb = img_disp

                superimposed = cv2.addWeighted(img_rgb, 0.6, heatmap_color, 0.4, 0)

                axes[1, i].imshow(superimposed)
                axes[1, i].set_title(f"Grad-CAM++ Focus", fontsize=10)
                axes[1, i].axis('off')

            plt.tight_layout()
            plt.show()
            print("\n")

# ==========================================
# 3. EXECUTION
# ==========================================
if __name__ == "__main__":
    # 1. Run the Collector
    tp_data = collect_true_positives(val_gen, model, num_per_class=2)

    # 2. Run the Visualizer
    visualize_class_report(tp_data, model)

###Case by case analysis

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt
import numpy as np
import cv2
import tensorflow as tf

# ==========================================
# 1. SETUP HELPERS
# ==========================================
def get_generator(dataset_name):
    if dataset_name == 'Validation Set':
        if 'val_gen' not in globals():
            return QuadStreamGenerator(X_val_global, X_val_local, X_val_metadata, y_val, batch_size=1, shuffle=False)
        return val_gen
    elif dataset_name == 'Test Set':
        if 'test_gen' not in globals():
            global test_gen
            test_gen = QuadStreamGenerator(X_test_global, X_test_local, X_test_metadata, y_test, batch_size=1, shuffle=False)
        return test_gen
    return None

def find_cases_by_class(generator, target_class_idx):
    """Scans dataset and returns a list of valid indices for the class."""
    valid_ids = []
    scan_limit = min(len(generator), 1000)

    for i in range(scan_limit):
        _, targets = generator[i]
        true_label = np.argmax(targets[0])
        if true_label == target_class_idx:
            valid_ids.append(i)

    return valid_ids

# ==========================================
# 2. DASHBOARD VISUALIZATION LOGIC
# ==========================================
def render_dashboard(dataset_name, target_class_name, case_id):
    # Safety Check
    if case_id is None:
        print(" Please select a Case ID first.")
        return

    # Map class name to index
    class_map = {"Benign": 0, "Luminal A": 1, "Luminal B": 2, "HER2+": 3, "TN": 4}
    class_idx = class_map[target_class_name]

    # Get Generator
    gen = get_generator(dataset_name)

    print(f" Processing Case {case_id} from {dataset_name}...")

    try:
        inputs, targets = gen[case_id]
        preds = model.predict(inputs, verbose=0)

        # Handle Multi-Output Model Structure correctly
        # model.predict returns a list: [Main_Output, Aux1, Aux2, Aux3, Aux4]
        # Main_Output has shape (1, 5)
        # We need preds[0][0] to get the flat (5,) vector
        if isinstance(preds, list):
            pred_probs = preds[0][0]
        else:
            pred_probs = preds[0] # Fallback if single output

        true_idx = np.argmax(targets[0])
        pred_idx = np.argmax(pred_probs)

        class_names = ["Benign", "Luminal A", "Luminal B", "HER2+", "TN"]
        titles = ["Global Gamma", "Global CLAHE", "Local Sigmoid", "Local CLAHE"]
        tower_names = ["xception_t1_global_gamma", "xception_t2_global_clahe", "xception_t3_local_sigmoid", "xception_t4_local_clahe"]

        # Generate Heatmaps
        heatmaps = []
        for t_name in tower_names:
            hm = generate_grad_cam_plus_plus(model, inputs, pred_idx, t_name)
            heatmaps.append(hm)

        # DRAW DASHBOARD
        fig = plt.figure(figsize=(20, 12))
        gs = fig.add_gridspec(3, 4)

        # Status Header
        status_color = 'green' if true_idx == pred_idx else 'red'
        status_text = "[CORRECT]" if true_idx == pred_idx else "[MISCLASSIFIED]"

        # Ensure pred_probs is treated as float for formatting
        conf_score = float(pred_probs[pred_idx]) * 100

        plt.suptitle(f"CASE {case_id} REPORT | {status_text}\nTrue: {class_names[true_idx]} | Pred: {class_names[pred_idx]} ({conf_score:.1f}%)",
                     fontsize=18, weight='bold', color=status_color)

        # Row 1: Images
        for i in range(4):
            ax = fig.add_subplot(gs[0:2, i])
            img_array = inputs[i][0]
            img_disp = np.uint8((img_array + 1) * 127.5)

            heatmap = heatmaps[i]
            heatmap_resized = cv2.resize(heatmap, (299, 299))
            heatmap_color = cv2.applyColorMap(np.uint8(255 * heatmap_resized), cv2.COLORMAP_JET)
            heatmap_color = cv2.cvtColor(heatmap_color, cv2.COLOR_BGR2RGB)

            if len(img_disp.shape) == 2: img_rgb = cv2.cvtColor(img_disp, cv2.COLOR_GRAY2RGB)
            else: img_rgb = img_disp

            superimposed = cv2.addWeighted(img_rgb, 0.6, heatmap_color, 0.4, 0)
            ax.imshow(superimposed)
            ax.set_title(f"{titles[i]}", fontsize=12, weight='bold')
            ax.axis('off')

        # Row 2: Stats
        ax_stats = fig.add_subplot(gs[2, :])
        colors = ['gray'] * 5
        colors[true_idx] = 'green'
        if true_idx != pred_idx: colors[pred_idx] = 'red'

        bars = ax_stats.bar(class_names, pred_probs * 100, color=colors, alpha=0.7)
        for bar in bars:
            height = bar.get_height()
            ax_stats.text(bar.get_x() + bar.get_width()/2., height + 1, f'{height:.1f}%', ha='center', va='bottom', fontsize=12, weight='bold')

        ax_stats.set_ylim(0, 110)
        ax_stats.set_title("Confidence Distribution", fontsize=14, weight='bold')
        ax_stats.grid(axis='y', linestyle='--', alpha=0.5)

        plt.tight_layout()
        plt.show()

    except Exception as e:
        print(f" Error: {e}")
        # Debug print to help identify shape issues
        print(f"DEBUG info: Preds shape type: {type(preds)}")
        if isinstance(preds, list):
             print(f"DEBUG info: preds[0] shape: {preds[0].shape}")


# ==========================================
# 3. INTERACTIVE WIDGETS UI
# ==========================================

# A. Create Widgets
style = {'description_width': 'initial'}

dataset_dropdown = widgets.Dropdown(
    options=['Validation Set', 'Test Set'],
    value='Validation Set',
    description='1️⃣ Dataset:',
    style=style,
    layout=widgets.Layout(width='300px')
)

class_dropdown = widgets.Dropdown(
    options=['Benign', 'Luminal A', 'Luminal B', 'HER2+', 'TN'],
    value='Benign',
    description='2️⃣ Class:',
    style=style,
    layout=widgets.Layout(width='300px')
)

btn_scan = widgets.Button(
    description='Step 1: Scan for IDs',
    button_style='info', # 'success', 'info', 'warning', 'danger' or ''
    icon='search',
    layout=widgets.Layout(width='300px')
)

case_dropdown = widgets.Dropdown(
    options=[],
    description='3️⃣ Select ID:',
    disabled=True, # Disabled until scanned
    style=style,
    layout=widgets.Layout(width='300px')
)

btn_generate = widgets.Button(
    description='Step 2: Generate Report',
    button_style='success',
    icon='chart-bar',
    disabled=True, # Disabled until ID selected
    layout=widgets.Layout(width='300px')
)

output_log = widgets.Output()
output_plot = widgets.Output()

# B. Event Handlers
def on_scan_click(b):
    """Scans the dataset and populates the dropdown."""
    output_log.clear_output()
    output_plot.clear_output()

    with output_log:
        print(f" Scanning {dataset_dropdown.value} for {class_dropdown.value} cases...")
        gen = get_generator(dataset_dropdown.value)

        # Get Class Index
        class_map = {"Benign": 0, "Luminal A": 1, "Luminal B": 2, "HER2+": 3, "TN": 4}
        target_idx = class_map[class_dropdown.value]

        # Find IDs
        valid_ids = find_cases_by_class(gen, target_idx)

        if valid_ids:
            print(f" Found {len(valid_ids)} cases.")
            case_dropdown.options = valid_ids
            case_dropdown.disabled = False
            case_dropdown.value = valid_ids[0]
            btn_generate.disabled = False # Unlock the generate button
        else:
            print(" No cases found.")
            case_dropdown.options = []
            case_dropdown.disabled = True
            btn_generate.disabled = True

def on_generate_click(b):
    """Generates the dashboard."""
    output_plot.clear_output()
    with output_plot:
        render_dashboard(dataset_dropdown.value, class_dropdown.value, case_dropdown.value)

# C. Link Events
btn_scan.on_click(on_scan_click)
btn_generate.on_click(on_generate_click)

# D. Layout
ui = widgets.VBox([
    widgets.HBox([dataset_dropdown, class_dropdown]),
    widgets.HBox([btn_scan, case_dropdown]),
    btn_generate
])

print(" MEDICAL AI DIAGNOSTIC DASHBOARD")
display(ui)
display(output_log)
display(output_plot)

#XVI. SHAP on Text & Image

##SHAP v1.0

####Implementing SHAP

In [ ]:
import shap
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow.keras.applications.xception import preprocess_input
import pandas as pd
import os

# ==========================================
# 1. SETUP & CONFIGURATION
# ==========================================
# Define paths (Same as your previous scripts)
DATA_PATH = '/content/drive/MyDrive/CMMD_preprocessed/v1.0'
MODEL_PATH = '/content/drive/My Drive/Xception model/v1.0/phase2_v1.0_final.keras'

# Define Feature Names based on your preprocessing logic:
# You constructed the vector as: [Age, Ab_None, Ab_Mass, Ab_Calcification, Ab_Both]
feature_names = [
    'Age (Normalized)',
    'Abnormality: None',
    'Abnormality: Mass',
    'Abnormality: Calcification',
    'Abnormality: Both'
]

class_names = ['Benign', 'Luminal A', 'Luminal B', 'HER2', 'Triple Negative']

# ==========================================
# 2. CUSTOM LOSS
# ==========================================
def class_weighted_focal_loss(gamma=2.5, alpha_weights=None):
    if alpha_weights is not None:
        alpha_tensor = tf.constant(alpha_weights, dtype=tf.float32)
    else:
        alpha_tensor = None

    def focal_loss_fixed(y_true, y_pred):
        epsilon = tf.keras.backend.epsilon()
        y_pred = tf.clip_by_value(y_pred, epsilon, 1. - epsilon)
        y_true = tf.cast(y_true, tf.float32)
        pt = tf.reduce_sum(y_true * y_pred, axis=-1)
        ce = -tf.math.log(pt + epsilon)
        focal_weight = tf.math.pow(1 - pt, gamma)
        if alpha_tensor is not None:
            alpha_t = tf.reduce_sum(y_true * alpha_tensor, axis=-1)
            loss = alpha_t * focal_weight * ce
        else:
            loss = focal_weight * ce
        return tf.reduce_mean(loss)
    return focal_loss_fixed

# ==========================================
# 3. LOAD DATA & MODEL
# ==========================================
print(" Loading Data...")
# We only need the validation data for explanation
X_val_global = np.load(os.path.join(DATA_PATH, 'X_val_global.npy'))
X_val_local  = np.load(os.path.join(DATA_PATH, 'X_val_local.npy'))
X_val_metadata = np.load(os.path.join(DATA_PATH, 'X_val_metadata.npy'))
y_val = np.load(os.path.join(DATA_PATH, 'y_val.npy'))

print(f" Loading Model from: {MODEL_PATH}")
# Dummy alphas just to satisfy the custom object requirement
alpha_dummy = np.array([1.0, 1.0, 1.0, 1.0, 1.0])
model = tf.keras.models.load_model(
    MODEL_PATH,
    custom_objects={'focal_loss_fixed': class_weighted_focal_loss(gamma=2.5, alpha_weights=alpha_dummy)}
)
print(" Model Loaded.")

# ==========================================
# 4. PREPARE SHAP EXPLAINER
# ==========================================
print("\n Initializing SHAP KernelExplainer...")

# SHAP needs a "background" dataset to understand what "average" looks like.
# Using the whole dataset is too slow, so summarize it using K-Means (e.g., 50 representative points).
background_summary = shap.kmeans(X_val_metadata, 50)

# ==========================================
# 5. DEFINE RAM-SAFE PREDICTION WRAPPER
# ==========================================

# Global variables (Set these before running SHAP)
current_stacks = None

def predict_fn_for_shap(metadata_batch):
    """
    RAM-SAFE Wrapper.
    Instead of predicting all 500+ perturbed samples at once,
    we loop through them in small mini-batches (e.g., 16).
    """
    BATCH_SIZE = 16 # Small batch size
    n_samples = metadata_batch.shape[0]
    all_predictions = []

    # Loop through the metadata in chunks
    for i in range(0, n_samples, BATCH_SIZE):
        # 1. Slice the metadata chunk
        meta_chunk = metadata_batch[i : i + BATCH_SIZE]
        current_chunk_size = meta_chunk.shape[0]

        # 2. Repeat the fixed patient images ONLY for this small chunk
        # current_stacks is a tuple: (s1, s2, s3, s4) calculated outside
        # We assume current_stacks have shape (1, 299, 299, 3)
        s1_batch = np.repeat(current_stacks[0], current_chunk_size, axis=0)
        s2_batch = np.repeat(current_stacks[1], current_chunk_size, axis=0)
        s3_batch = np.repeat(current_stacks[2], current_chunk_size, axis=0)
        s4_batch = np.repeat(current_stacks[3], current_chunk_size, axis=0)

        # 3. Predict on this small batch
        # Returns [final, aux1, aux2, aux3, aux4]
        preds = model.predict([s1_batch, s2_batch, s3_batch, s4_batch, meta_chunk], verbose=0)

        # 4. Store only the final output
        all_predictions.append(preds[0])

    # 5. Combine all chunks back into one array
    return np.vstack(all_predictions)

# ==========================================
# 6. RUN EXPLANATION FOR A SPECIFIC PATIENT
# ==========================================
# Pick a patient index to explain (e.g., Index 0)
PATIENT_IDX = 0

print(f"\n Analyzing Patient Index: {PATIENT_IDX}")
print(f" True Label: {class_names[np.argmax(y_val[PATIENT_IDX])]}")
print(f" Input Metadata: {X_val_metadata[PATIENT_IDX]}")

# --- A. PRE-CALCULATE IMAGE STACKS FOR THIS PATIENT ---
# We do this once to ensure the images remain exactly the same while SHAP varies the text
import cv2

def get_patient_stacks(idx):
    # 1. Get Raw Images
    ig = X_val_global[idx] # (299, 299, 3)
    il = X_val_local[idx]  # (299, 299, 3)

    # Ensure uint8
    if ig.dtype != np.uint8: ig = ig.astype(np.uint8)
    if il.dtype != np.uint8: il = il.astype(np.uint8)

    # 2. Re-create the Generator Augmentations (Manual CV2)
    # Helpers
    def apply_gamma(img, gamma):
        invGamma = 1.0 / gamma
        table = np.array([((i / 255.0) ** invGamma) * 255 for i in np.arange(0, 256)]).astype("uint8")
        return cv2.LUT(img, table)

    def apply_sigmoid(img, gain, cutoff):
        img_f = img.astype(np.float32) / 255.0
        sigmoid = 1.0 / (1.0 + np.exp(-gain * (img_f - cutoff)))
        return (sigmoid * 255).astype(np.uint8)

    clahe_mild = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    clahe_strong = cv2.createCLAHE(clipLimit=4.0, tileGridSize=(8,8))

    # Stack 1
    s1 = np.stack([ig[:,:,0], apply_gamma(ig[:,:,0], 0.6), apply_gamma(ig[:,:,0], 1.5)], axis=-1)

    # Stack 2
    s2 = np.stack([ig[:,:,0], clahe_mild.apply(ig[:,:,0]), clahe_strong.apply(ig[:,:,0])], axis=-1)

    # Stack 3
    s3 = np.stack([il[:,:,0], apply_sigmoid(il[:,:,0], 8, 0.75), apply_sigmoid(il[:,:,0], 15, 0.60)], axis=-1)

    # Stack 4
    s4 = np.stack([il[:,:,0], clahe_mild.apply(il[:,:,0]), clahe_strong.apply(il[:,:,0])], axis=-1)

    # Add Batch Dimension and Preprocess
    s1 = preprocess_input(np.expand_dims(s1, axis=0).astype(np.float32))
    s2 = preprocess_input(np.expand_dims(s2, axis=0).astype(np.float32))
    s3 = preprocess_input(np.expand_dims(s3, axis=0).astype(np.float32))
    s4 = preprocess_input(np.expand_dims(s4, axis=0).astype(np.float32))

    return (s1, s2, s3, s4)

# Set the global variable
current_stacks = get_patient_stacks(PATIENT_IDX)

# --- B. COMPUTE SHAP VALUES ---
# Initialize Explainer with the prediction wrapper and background data
explainer = shap.KernelExplainer(predict_fn_for_shap, background_summary)

# Calculate SHAP values for the single instance
# nsamples=500 is usually enough for tabular data with few features
shap_values = explainer.shap_values(X_val_metadata[PATIENT_IDX:PATIENT_IDX+1], nsamples=500)

# ==========================================
# 7. VISUALIZE RESULTS (WITH REAL AGE)
# ==========================================
# Get the predicted class to focus the plot on that specific output
preds = predict_fn_for_shap(X_val_metadata[PATIENT_IDX:PATIENT_IDX+1])
top_class_idx = np.argmax(preds[0])
top_class_name = class_names[top_class_idx]

print(f"\n Model Prediction: {top_class_name} ({preds[0][top_class_idx]*100:.2f}%)")

# --- SAFE EXTRACTION LOGIC ---
vals = None
base_val = None

if isinstance(shap_values, list):
    print(" SHAP returned a List. Extracting specific class...")
    vals = shap_values[top_class_idx][0]
    base_val = explainer.expected_value[top_class_idx]
else:
    print(" SHAP returned an Array. Slicing correct dimensions...")
    vals = shap_values[0][:, top_class_idx]
    if hasattr(explainer.expected_value, '__iter__'):
        base_val = explainer.expected_value[top_class_idx]
    else:
        base_val = explainer.expected_value

# --- DYNAMIC AGE DENORMALIZATION ---
# Automatically extract min and max from the existing 'clin' DataFrame
valid_ages = clin['Age'].dropna()
AGE_MIN = valid_ages.min()
AGE_MAX = valid_ages.max()

print(f" Denormalizing Age using clinical range: {AGE_MIN} to {AGE_MAX} years.")

# Create a copy of the patient data specifically for the plot
display_data = X_val_metadata[PATIENT_IDX].copy()

# Reverse the Min-Max formula: Original = (Normalized * (Max - Min)) + Min
display_data[0] = (display_data[0] * (AGE_MAX - AGE_MIN)) + AGE_MIN
display_data[0] = np.round(display_data[0], 1) # Rounding for a cleaner display

# Update the feature names to reflect the real-world metric
display_feature_names = [
    'Age (Years)',
    'Abnormality: None',
    'Abnormality: Mass',
    'Abnormality: Calcification',
    'Abnormality: Both'
]

# --- PLOT WATERFALL ---
plt.figure()
shap.waterfall_plot(
    shap.Explanation(
        values=vals,
        base_values=base_val,
        data=display_data,                   # Use the un-normalized data
        feature_names=display_feature_names  # Use the updated labels
    ),
    show=False
)
plt.title(f"Why did the model predict {top_class_name}?", fontsize=14)
plt.show()

###Global analysis (Class by class)

In [ ]:
import shap
import numpy as np
import tensorflow as tf
import pickle
import cv2
import os
from tqdm import tqdm
from tensorflow.keras.applications.xception import preprocess_input

# ==========================================
# 1. SETUP & MODEL LOADING
# ==========================================
# Only Model Path is needed now
MODEL_PATH = '/content/drive/My Drive/Xception model/v1.0/phase2_v1.0_final.keras'
SAVE_PATH = '/content/drive/My Drive/Xception model/v1.0/SHAP/shap_results_v1.0.pkl'

# Define names for the plot later
feature_names = [
    'Age (Normalized)', 'Abnormality: None', 'Abnormality: Mass',
    'Abnormality: Calcification', 'Abnormality: Both'
]
class_names = ['Benign', 'Luminal A', 'Luminal B', 'HER2', 'Triple Negative']

# Custom Loss (Needed to load the model)
def class_weighted_focal_loss(gamma=2.5, alpha_weights=None):
    if alpha_weights is not None:
        alpha_tensor = tf.constant(alpha_weights, dtype=tf.float32)
    else:
        alpha_tensor = None
    def focal_loss_fixed(y_true, y_pred):
        epsilon = tf.keras.backend.epsilon()
        y_pred = tf.clip_by_value(y_pred, epsilon, 1. - epsilon)
        y_true = tf.cast(y_true, tf.float32)
        pt = tf.reduce_sum(y_true * y_pred, axis=-1)
        ce = -tf.math.log(pt + epsilon)
        focal_weight = tf.math.pow(1 - pt, gamma)
        if alpha_tensor is not None:
            alpha_t = tf.reduce_sum(y_true * alpha_tensor, axis=-1)
            loss = alpha_t * focal_weight * ce
        else:
            loss = focal_weight * ce
        return tf.reduce_mean(loss)
    return focal_loss_fixed

print(f" Loading Model from: {MODEL_PATH}")
alpha_dummy = np.array([1.0, 1.0, 1.0, 1.0, 1.0])
model = tf.keras.models.load_model(
    MODEL_PATH,
    custom_objects={'focal_loss_fixed': class_weighted_focal_loss(gamma=2.5, alpha_weights=alpha_dummy)}
)
print(" Model Loaded.")

# ==========================================
# 2. HELPER FUNCTIONS
# ==========================================
current_stacks = None

def get_patient_stacks(idx):
    ig = X_val_global[idx]; il = X_val_local[idx]
    if ig.dtype != np.uint8: ig = ig.astype(np.uint8)
    if il.dtype != np.uint8: il = il.astype(np.uint8)

    def apply_gamma(img, gamma):
        invGamma = 1.0 / gamma; table = np.array([((i / 255.0) ** invGamma) * 255 for i in np.arange(0, 256)]).astype("uint8")
        return cv2.LUT(img, table)
    def apply_sigmoid(img, gain, cutoff):
        img_f = img.astype(np.float32) / 255.0; sigmoid = 1.0 / (1.0 + np.exp(-gain * (img_f - cutoff)))
        return (sigmoid * 255).astype(np.uint8)
    clahe_mild = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    clahe_strong = cv2.createCLAHE(clipLimit=4.0, tileGridSize=(8,8))

    s1 = np.stack([ig[:,:,0], apply_gamma(ig[:,:,0], 0.6), apply_gamma(ig[:,:,0], 1.5)], axis=-1)
    s2 = np.stack([ig[:,:,0], clahe_mild.apply(ig[:,:,0]), clahe_strong.apply(ig[:,:,0])], axis=-1)
    s3 = np.stack([il[:,:,0], apply_sigmoid(il[:,:,0], 8, 0.75), apply_sigmoid(il[:,:,0], 15, 0.60)], axis=-1)
    s4 = np.stack([il[:,:,0], clahe_mild.apply(il[:,:,0]), clahe_strong.apply(il[:,:,0])], axis=-1)

    return (
        preprocess_input(np.expand_dims(s1, axis=0).astype(np.float32)),
        preprocess_input(np.expand_dims(s2, axis=0).astype(np.float32)),
        preprocess_input(np.expand_dims(s3, axis=0).astype(np.float32)),
        preprocess_input(np.expand_dims(s4, axis=0).astype(np.float32))
    )

def predict_fn_for_shap(metadata_batch):
    BATCH_SIZE = 16
    n_samples = metadata_batch.shape[0]
    all_predictions = []

    for i in range(0, n_samples, BATCH_SIZE):
        meta_chunk = metadata_batch[i : i + BATCH_SIZE]
        current_chunk_size = meta_chunk.shape[0]
        s1_batch = np.repeat(current_stacks[0], current_chunk_size, axis=0)
        s2_batch = np.repeat(current_stacks[1], current_chunk_size, axis=0)
        s3_batch = np.repeat(current_stacks[2], current_chunk_size, axis=0)
        s4_batch = np.repeat(current_stacks[3], current_chunk_size, axis=0)
        preds = model.predict([s1_batch, s2_batch, s3_batch, s4_batch, meta_chunk], verbose=0)
        all_predictions.append(preds[0])
    return np.vstack(all_predictions)

# ==========================================
# 3. RUN ANALYSIS (Balanced Sampling)
# ==========================================
N_PER_CLASS = 10
print(f"\n Selecting {N_PER_CLASS} samples per class from existing data...")

random_indices = []
y_val_integers = np.argmax(y_val, axis=1)

for class_idx in range(5):
    indices_for_class = np.where(y_val_integers == class_idx)[0]
    if len(indices_for_class) >= N_PER_CLASS:
        chosen = np.random.choice(indices_for_class, N_PER_CLASS, replace=False)
    else:
        chosen = indices_for_class
    random_indices.extend(chosen)

random_indices = np.array(random_indices)
np.random.shuffle(random_indices)

global_shap_values = [[] for _ in range(5)]
global_X_test = []

print(" Initializing Explainer...")
background_summary = shap.kmeans(X_val_metadata, 10)
current_stacks = get_patient_stacks(0)
explainer = shap.KernelExplainer(predict_fn_for_shap, background_summary)

for i, idx in enumerate(tqdm(random_indices, desc="Explaining")):
    current_stacks = get_patient_stacks(idx)
    shap_vals = explainer.shap_values(X_val_metadata[idx:idx+1], nsamples=100)
    global_X_test.append(X_val_metadata[idx])

    if isinstance(shap_vals, list):
        for class_idx in range(5):
            global_shap_values[class_idx].append(shap_vals[class_idx][0])
    else:
        for class_idx in range(5):
            global_shap_values[class_idx].append(shap_vals[0][:, class_idx])

# ==========================================
# 4. SAVE TO DRIVE
# ==========================================
print(f"\n Saving results to: {SAVE_PATH}")

data_to_save = {
    'shap_values': global_shap_values,
    'X_test': np.array(global_X_test),
    'feature_names': feature_names,
    'class_names': class_names
}

with open(SAVE_PATH, 'wb') as f:
    pickle.dump(data_to_save, f)

print(" Saved successfully! Run the Viewer script to see plots.")

####Sample global SHAP result

In [ ]:
import shap
import pickle
import numpy as np
import matplotlib.pyplot as plt

# ==========================================
# 1. DYNAMIC AGE DENORMALIZATION
# ==========================================
print(" Please upload your clinical Excel file to denormalize the global ages...")
uploaded = files.upload()
clinical_xlsx = list(uploaded.keys())[0]
clin = pd.read_excel(clinical_xlsx)

valid_ages = clin['Age'].dropna()
AGE_MIN = valid_ages.min()
AGE_MAX = valid_ages.max()
print(f" Clinical data loaded. Age Range: {AGE_MIN} to {AGE_MAX} years.")

# ==========================================
# 2. LOAD FROM DRIVE
# ==========================================
LOAD_PATH = '/content/drive/My Drive/Xception model/v1.0/SHAP/shap_results_v1.0.pkl'

print(f" Loading SHAP results from: {LOAD_PATH}")
with open(LOAD_PATH, 'rb') as f:
    data = pickle.load(f)

# Unpack
raw_shap_list = data['shap_values']
X_test = data['X_test']
feature_names = data['feature_names']
class_names = data['class_names']
final_shap_values = [np.array(raw_shap_list[c]) for c in range(5)]

# --- INTERCEPT AND REVERSE NORMALIZATION ---
# X_test[:, 0] grabs the Age column for all patients in the global summary
X_test[:, 0] = (X_test[:, 0] * (AGE_MAX - AGE_MIN)) + AGE_MIN
feature_names[0] = 'Age (Years)' # Update the label dynamically

print(" Data Loaded! Generating Plots...")

# ==========================================
# 2. GENERATE PLOTS
# ==========================================
# A. Individual Class Plots
for i, class_name in enumerate(class_names):
    plt.figure(figsize=(10, 6))
    plt.title(f"Feature Impact on Class: {class_name}", fontsize=14, pad=20)
    shap.summary_plot(final_shap_values[i], X_test, feature_names=feature_names, show=False)
    plt.tight_layout(); plt.show()

# B. Combined Bar Plot
print(f"\n{'='*40}\n Global Feature Importance (All Classes)\n{'='*40}")

# 1. Increase figure width for legend space
plt.figure(figsize=(14, 8))
plt.title("Overall Feature Importance Ranking (mean |SHAP value|)", fontsize=16)

# 2. Create the plot (show=False)
shap.summary_plot(
    final_shap_values,
    X_test,
    feature_names=feature_names,
    class_names=class_names,
    plot_type="bar",
    show=False
)

# 3. ADD DATA LABELS
ax = plt.gca() # Get the current axes

# Loop through each "Class" (which are the stacked segments)
for c in ax.containers:
    # Optional: Create custom labels to hide very small values (clutter reduction)
    # We only label if the bar width is > 0.002
    labels = [f'{v.get_width():.3f}' if v.get_width() > 0.002 else '' for v in c]

    # Add the labels to the center of the bar segments
    ax.bar_label(c, labels=labels, label_type='center', fontsize=9, color='white', weight='bold')

# 4. Manually move the legend outside
plt.legend(loc='center left', bbox_to_anchor=(1.0, 0.5), fontsize=12)

# 5. Final Layout Adjustment
plt.xlabel("mean(|SHAP value|) (average impact on model output magnitude)")
plt.tight_layout()
plt.show()

###Textual case by case analysis

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt
import numpy as np
import cv2
import shap
import tensorflow as tf
import os
from tensorflow.keras.applications.xception import preprocess_input

# ==========================================
# 1. SETUP: LOAD MODEL
# ==========================================
MODEL_PATH = '/content/drive/My Drive/Xception model/v1.0/phase2_v1.0_final.keras'

def class_weighted_focal_loss(gamma=2.5, alpha_weights=None):
    if alpha_weights is not None:
        alpha_tensor = tf.constant(alpha_weights, dtype=tf.float32)
    else:
        alpha_tensor = None
    def focal_loss_fixed(y_true, y_pred):
        epsilon = tf.keras.backend.epsilon()
        y_pred = tf.clip_by_value(y_pred, epsilon, 1. - epsilon)
        y_true = tf.cast(y_true, tf.float32)
        pt = tf.reduce_sum(y_true * y_pred, axis=-1)
        ce = -tf.math.log(pt + epsilon)
        focal_weight = tf.math.pow(1 - pt, gamma)
        if alpha_tensor is not None:
            alpha_t = tf.reduce_sum(y_true * alpha_tensor, axis=-1)
            loss = alpha_t * focal_weight * ce
        else:
            loss = focal_weight * ce
        return tf.reduce_mean(loss)
    return focal_loss_fixed

print(f" Loading Model from: {MODEL_PATH}...")
alpha_dummy = np.array([1.0, 1.0, 1.0, 1.0, 1.0])
model = tf.keras.models.load_model(
    MODEL_PATH,
    custom_objects={'focal_loss_fixed': class_weighted_focal_loss(gamma=2.5, alpha_weights=alpha_dummy)}
)
print(" Model Loaded Successfully!")

# ==========================================
# 2. CONFIGURATION
# ==========================================
CLASS_NAMES = ['Benign', 'Luminal A', 'Luminal B', 'HER2', 'Triple Negative']
FEATURE_NAMES = ['Age', 'Abnorm: None', 'Abnorm: Mass', 'Abnorm: Calcification', 'Abnorm: Both']

def get_dataset_arrays(name):
    if name == 'Validation Set':
        if 'X_val_global' not in globals(): raise NameError(" X_val_global not found!")
        return X_val_global, X_val_local, X_val_metadata, y_val
    else:
        if 'X_test_global' not in globals(): raise NameError(" X_test_global not found!")
        return X_test_global, X_test_local, X_test_metadata, y_test

if 'X_val_metadata' in globals():
    print(" Initializing SHAP background...")
    background_summary = shap.kmeans(X_val_metadata, 10)
else:
    print(" Warning: Data not loaded yet.")

# ==========================================
# 3. ANALYSIS LOGIC
# ==========================================
def analyze_case(dataset_name, case_id, progress_bar=None):
    # 1. Fetch Data
    Xg, Xl, Xm, y = get_dataset_arrays(dataset_name)
    meta_vector = Xm[case_id]
    true_label_idx = np.argmax(y[case_id])

    # 2. Reconstruct Images
    def process_images(idx):
        ig = Xg[idx]; il = Xl[idx]
        if ig.dtype != np.uint8: ig = ig.astype(np.uint8)
        if il.dtype != np.uint8: il = il.astype(np.uint8)
        if ig.ndim == 3: ig = ig[:,:,0]
        if il.ndim == 3: il = il[:,:,0]

        clahe_mild = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
        clahe_strong = cv2.createCLAHE(clipLimit=4.0, tileGridSize=(8,8))
        def apply_gamma(img, gamma):
            invGamma = 1.0 / gamma; table = np.array([((i / 255.0) ** invGamma) * 255 for i in np.arange(0, 256)]).astype("uint8")
            return cv2.LUT(img, table)
        def apply_sigmoid(img, gain, cutoff):
            img_f = img.astype(np.float32) / 255.0; sigmoid = 1.0 / (1.0 + np.exp(-gain * (img_f - cutoff)))
            return (sigmoid * 255).astype(np.uint8)

        s1 = np.stack([ig, apply_gamma(ig, 0.6), apply_gamma(ig, 1.5)], axis=-1)
        s2 = np.stack([ig, clahe_mild.apply(ig), clahe_strong.apply(ig)], axis=-1)
        s3 = np.stack([il, apply_sigmoid(il, 8, 0.75), apply_sigmoid(il, 15, 0.60)], axis=-1)
        s4 = np.stack([il, clahe_mild.apply(il), clahe_strong.apply(il)], axis=-1)

        return (
            preprocess_input(np.expand_dims(s1, axis=0).astype(np.float32)),
            preprocess_input(np.expand_dims(s2, axis=0).astype(np.float32)),
            preprocess_input(np.expand_dims(s3, axis=0).astype(np.float32)),
            preprocess_input(np.expand_dims(s4, axis=0).astype(np.float32))
        ), (s1, s2, s3, s4)

    model_inputs, raw_stacks = process_images(case_id)

    # 3. Model Prediction
    meta_batch = np.expand_dims(meta_vector, axis=0)
    preds = model.predict(list(model_inputs) + [meta_batch], verbose=0)
    pred_probs = preds[0][0]
    pred_label_idx = np.argmax(pred_probs)

    # 4. SHAP Analysis
    if progress_bar:
        progress_bar.value = 0
        progress_bar.description = "Explaining..."

    def shap_wrapper(meta_chunk):
        BATCH_SIZE = 10 # Small batch to allow UI updates
        n_total = meta_chunk.shape[0]
        results = []

        for i in range(0, n_total, BATCH_SIZE):
            batch_meta = meta_chunk[i : i + BATCH_SIZE]
            current_n = batch_meta.shape[0]

            i1 = np.repeat(model_inputs[0], current_n, axis=0)
            i2 = np.repeat(model_inputs[1], current_n, axis=0)
            i3 = np.repeat(model_inputs[2], current_n, axis=0)
            i4 = np.repeat(model_inputs[3], current_n, axis=0)

            p = model.predict([i1, i2, i3, i4, batch_meta], verbose=0)
            results.append(p[0])

            if progress_bar:
                progress_bar.value += current_n

        return np.vstack(results)

    # Note: Initial 'dry run' of explainer might bump progress bar slightly,
    # but the main loop will be the 100 nsamples.
    explainer = shap.KernelExplainer(shap_wrapper, background_summary)

    # Set max to approx nsamples + background check overhead
    if progress_bar: progress_bar.max = 110

    shap_vals = explainer.shap_values(meta_batch, nsamples=100, silent=True)

    if progress_bar:
        progress_bar.value = progress_bar.max # Finish bar
        progress_bar.description = "Done!"

    # Handle SHAP output
    if isinstance(shap_vals, list):
        shap_values_target = shap_vals[pred_label_idx][0]
        base_value_target = explainer.expected_value[pred_label_idx]
    else:
        shap_values_target = shap_vals[0][:, pred_label_idx]
        base_value_target = explainer.expected_value[pred_label_idx]

    return {
        'images': raw_stacks,
        'true_idx': true_label_idx,
        'pred_idx': pred_label_idx,
        'probs': pred_probs,
        'shap_vals': shap_values_target,
        'base_val': base_value_target,
        'meta': meta_vector
    }

# ==========================================
# 4. VISUALIZATION REPORT
# ==========================================
def render_dashboard(data, case_id):
    # --- PART 1: DIAGNOSIS & IMAGES ---
    fig1 = plt.figure(figsize=(20, 10), layout='constrained')
    gs1 = fig1.add_gridspec(2, 4, height_ratios=[1.2, 0.8])

    true_name = CLASS_NAMES[data['true_idx']]
    pred_name = CLASS_NAMES[data['pred_idx']]
    conf = data['probs'][data['pred_idx']] * 100
    color = 'green' if data['true_idx'] == data['pred_idx'] else 'red'
    status = "CORRECT" if color == 'green' else "MISDIAGNOSIS"

    fig1.suptitle(f"CASE {case_id}: {status} | True: {true_name} vs Pred: {pred_name} ({conf:.1f}%)",
                 fontsize=20, weight='bold', color=color)

    titles = ["Global Gamma", "Global CLAHE", "Local Sigmoid", "Local CLAHE"]
    for i in range(4):
        ax = fig1.add_subplot(gs1[0, i])
        img = data['images'][i][:,:,1]
        ax.imshow(img, cmap='bone')
        ax.set_title(titles[i], fontsize=12)
        ax.axis('off')

    ax_stats = fig1.add_subplot(gs1[1, :])
    bar_colors = ['lightgray'] * 5
    bar_colors[data['true_idx']] = 'green'
    if data['true_idx'] != data['pred_idx']:
        bar_colors[data['pred_idx']] = 'red'

    bars = ax_stats.bar(CLASS_NAMES, data['probs']*100, color=bar_colors, alpha=0.8, width=0.5)

    for bar in bars:
        height = bar.get_height()
        ax_stats.text(bar.get_x() + bar.get_width()/2., height + 1,
                      f'{height:.1f}%', ha='center', va='bottom', fontsize=12, weight='bold')

    ax_stats.set_ylim(0, 115)
    ax_stats.set_ylabel("Confidence (%)")
    ax_stats.set_title("Model Confidence Distribution", fontsize=14, weight='bold')
    ax_stats.spines['top'].set_visible(False)
    ax_stats.spines['right'].set_visible(False)
    plt.show()

    # --- PART 2: EXPLAINABILITY (SHAP) ---
    print("\n" + "="*80 + "\n")

    # DYNAMIC AGE DENORMALIZATION
    valid_ages = clin['Age'].dropna()
    AGE_MIN = valid_ages.min()
    AGE_MAX = valid_ages.max()

    display_meta = data['meta'].copy()
    display_meta[0] = (display_meta[0] * (AGE_MAX - AGE_MIN)) + AGE_MIN
    display_meta[0] = np.round(display_meta[0], 1)

    fig2 = plt.figure(figsize=(14, 6), layout='constrained')
    ax_shap = fig2.add_subplot(1, 1, 1)
    plt.sca(ax_shap)

    shap.waterfall_plot(
        shap.Explanation(
            values=data['shap_vals'],
            base_values=data['base_val'],
            data=display_meta,            # <-- Use denormalized data
            feature_names=FEATURE_NAMES   # <-- Relies on the updated global variable
        ),
        show=False,
        max_display=5
    )

    ax_shap.set_title(f"Why did the model predict '{pred_name}'?", fontsize=16, weight='bold')
    plt.show()

# ==========================================
# 5. INTERACTIVE WIDGETS
# ==========================================
style = {'description_width': 'initial'}
layout = widgets.Layout(width='300px')

dd_dataset = widgets.Dropdown(options=['Validation Set', 'Test Set'], value='Validation Set', description='1️⃣ Dataset:', style=style, layout=layout)
dd_class = widgets.Dropdown(options=CLASS_NAMES, value='Benign', description='2️⃣ Filter Class:', style=style, layout=layout)
btn_scan = widgets.Button(description='Step 1: Scan for Cases', button_style='info', icon='search', layout=layout)
dd_case = widgets.Dropdown(options=[], description='3️⃣ Select ID:', disabled=True, style=style, layout=layout)
btn_run = widgets.Button(description='Step 2: Analyze Case', button_style='success', icon='user-md', disabled=True, layout=layout)

# Progress Bar Widget
prog_bar = widgets.IntProgress(value=0, min=0, max=100, description='Waiting:', bar_style='info', layout=widgets.Layout(width='98%'))

out_log = widgets.Output()
out_plot = widgets.Output()

def on_scan(b):
    out_log.clear_output(); out_plot.clear_output()
    with out_log:
        try:
            print(f" Scanning {dd_dataset.value} for {dd_class.value} cases...")
            _, _, _, y = get_dataset_arrays(dd_dataset.value)
            target_idx = CLASS_NAMES.index(dd_class.value)
            valid_ids = np.where(np.argmax(y, axis=1) == target_idx)[0]

            if len(valid_ids) > 0:
                print(f" Found {len(valid_ids)} cases.")
                dd_case.options = tuple(valid_ids)
                dd_case.disabled = False
                dd_case.value = valid_ids[0]
                btn_run.disabled = False
                prog_bar.value = 0; prog_bar.description = 'Ready'
            else:
                print(" No cases found.")
                dd_case.options = []; dd_case.disabled = True; btn_run.disabled = True
        except Exception as e: print(f" Error: {e}")

def on_run(b):
    out_plot.clear_output()
    with out_plot:
        # Pass the progress bar widget to the analysis function
        try:
            data = analyze_case(dd_dataset.value, dd_case.value, progress_bar=prog_bar)
            render_dashboard(data, dd_case.value)
        except Exception as e:
            print(f" Error during analysis: {e}")

btn_scan.on_click(on_scan)
btn_run.on_click(on_run)

ui = widgets.VBox([
    widgets.HBox([dd_dataset, dd_class]),
    widgets.HBox([btn_scan, dd_case]),
    btn_run,
    prog_bar # Display the bar here
])
print(" MEDICAL AI EXPLAINABILITY DASHBOARD")
display(ui)
display(out_log)
display(out_plot)

###Image case by case analysis

####Scan for case

In [ ]:
import ipywidgets as widgets
from IPython.display import display
import numpy as np

# ==========================================
# 1. CONFIGURATION
# ==========================================
CLASS_NAMES = ['Benign', 'Luminal A', 'Luminal B', 'HER2', 'Triple Negative']

def get_dataset_arrays(name):
    # We only need the 'y' array to find the cases, so we return None for the images
    if name == 'Validation Set':
        return None, None, None, y_val
    else:
        return None, None, None, y_test

# ==========================================
# 2. WIDGETS & UI
# ==========================================
style = {'description_width': 'initial'}
layout = widgets.Layout(width='300px')

dd_dataset = widgets.Dropdown(options=['Validation Set', 'Test Set'], value='Validation Set', description='1️⃣ Dataset:', style=style, layout=layout)
dd_class = widgets.Dropdown(options=CLASS_NAMES, value='Benign', description='2️⃣ Filter Class:', style=style, layout=layout)
btn_scan = widgets.Button(description='Scan for Case IDs', button_style='info', icon='search', layout=layout)
dd_case = widgets.Dropdown(options=[], description='3️⃣ Available IDs:', disabled=True, style=style, layout=layout)

out_log = widgets.Output()

def on_scan(b):
    out_log.clear_output()
    with out_log:
        try:
            # Fetch the labels
            _, _, _, y = get_dataset_arrays(dd_dataset.value)
            t_idx = CLASS_NAMES.index(dd_class.value)

            # Find all IDs matching the true class
            v_ids = np.where(np.argmax(y, axis=1) == t_idx)[0]

            if len(v_ids) > 0:
                dd_case.options = tuple(v_ids)
                dd_case.disabled = False
                dd_case.value = v_ids[0]
                print(f" Found {len(v_ids)} cases.")
                print(" Pick an ID from the dropdown, then type it into your Manual SHAP cell.")
            else:
                print(" No cases found.")
                dd_case.options = []
                dd_case.disabled = True
        except Exception as e:
            print(f" Error: {e}")
            print("Make sure your dataset arrays (y_val, y_test) are loaded into memory!")

btn_scan.on_click(on_scan)

ui = widgets.VBox([
    widgets.HBox([dd_dataset, dd_class]),
    widgets.HBox([btn_scan, dd_case])
])

print("🔍 FAST CASE ID SCANNER")
display(ui)
display(out_log)

####Running analysis

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt
import numpy as np
import cv2
import shap
import tensorflow as tf
import gc
from tensorflow.keras.applications.xception import preprocess_input

# ==========================================
# 1. SETUP: LOAD MODEL
# ==========================================
MODEL_PATH = '/content/drive/My Drive/Xception model/v1.0/phase2_v1.0_final.keras'

def class_weighted_focal_loss(gamma=2.5, alpha_weights=None):
    if alpha_weights is not None:
        alpha_tensor = tf.constant(alpha_weights, dtype=tf.float32)
    else:
        alpha_tensor = None
    def focal_loss_fixed(y_true, y_pred):
        epsilon = tf.keras.backend.epsilon()
        y_pred = tf.clip_by_value(y_pred, epsilon, 1. - epsilon)
        y_true = tf.cast(y_true, tf.float32)
        pt = tf.reduce_sum(y_true * y_pred, axis=-1)
        ce = -tf.math.log(pt + epsilon)
        focal_weight = tf.math.pow(1 - pt, gamma)
        if alpha_tensor is not None:
            alpha_t = tf.reduce_sum(y_true * alpha_tensor, axis=-1)
            loss = alpha_t * focal_weight * ce
        else:
            loss = focal_weight * ce
        return tf.reduce_mean(loss)
    return focal_loss_fixed

print(f" Loading Model...")
alpha_balanced = np.array([1.0, 1.0, 1.0, 1.0, 1.0], dtype=float)
full_model = tf.keras.models.load_model(
    MODEL_PATH,
    custom_objects={'focal_loss_fixed': class_weighted_focal_loss(gamma=2.5, alpha_weights=alpha_balanced)}
)

# Wrapper to isolate Main Output
shap_model = tf.keras.Model(inputs=full_model.inputs, outputs=full_model.outputs[0])
print(" Model Loaded!")

# ==========================================
# 2. CONFIGURATION & DATA
# ==========================================
CLASS_NAMES = ['Benign', 'Luminal A', 'Luminal B', 'HER2', 'Triple Negative']

def get_dataset_arrays(name):
    if name == 'Validation Set':
        return X_val_global, X_val_local, X_val_metadata, y_val
    else:
        return X_test_global, X_test_local, X_test_metadata, y_test

# ==========================================
# 3. ANALYSIS LOGIC (UNIFIED WIDE-IMAGE SHAP)
# ==========================================
def run_unified_missed_signal(dataset_name, case_id, progress_bar=None, output_widget=None):
    # 1. Fetch Data
    Xg, Xl, Xm, y = get_dataset_arrays(dataset_name)
    meta_vector = Xm[case_id]
    true_label_idx = np.argmax(y[case_id])

    # 2. Prepare the "Wide Image" (Global + Local side-by-side)
    # This forces SHAP to treat them as a single synchronized input
    raw_ig = Xg[case_id]
    raw_il = Xl[case_id]

    # Standardize to 3-channel uint8 for SHAP plotting compatibility
    if raw_ig.ndim == 2: raw_ig = cv2.cvtColor(raw_ig, cv2.COLOR_GRAY2RGB)
    elif raw_ig.shape[-1] == 1: raw_ig = cv2.cvtColor(raw_ig, cv2.COLOR_GRAY2RGB)
    if raw_il.ndim == 2: raw_il = cv2.cvtColor(raw_il, cv2.COLOR_GRAY2RGB)
    elif raw_il.shape[-1] == 1: raw_il = cv2.cvtColor(raw_il, cv2.COLOR_GRAY2RGB)

    raw_ig = raw_ig.astype(np.uint8)
    raw_il = raw_il.astype(np.uint8)

    # Stitch them horizontally -> Shape: (299, 598, 3)
    wide_image = np.concatenate([raw_ig, raw_il], axis=1)

    # 3. THE SMART WRAPPER
    # SHAP will feed this function batches of masked 'wide_images'
    def unified_prediction_wrapper(image_batch):
        N = image_batch.shape[0]
        batch_s1, batch_s2, batch_s3, batch_s4 = [], [], [], []

        # Initialize OpenCV tools once per batch
        clahe_mild = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
        clahe_strong = cv2.createCLAHE(clipLimit=4.0, tileGridSize=(8,8))

        invGamma1 = 1.0 / 0.6; t1 = np.array([((x / 255.0) ** invGamma1) * 255 for x in np.arange(0, 256)]).astype("uint8")
        invGamma2 = 1.0 / 1.5; t2 = np.array([((x / 255.0) ** invGamma2) * 255 for x in np.arange(0, 256)]).astype("uint8")

        for i in range(N):
            # SHAP returns floats sometimes, so convert to uint8 safely
            img = image_batch[i].astype(np.uint8)

            # Split back into Global and Local
            ig_plane = img[:, :299, 1]  # Left half, middle channel
            il_plane = img[:, 299:, 1]  # Right half, middle channel

            # --- TOWER 1: Global Gamma ---
            s1 = np.stack([ig_plane, cv2.LUT(ig_plane, t1), cv2.LUT(ig_plane, t2)], axis=-1)

            # --- TOWER 2: Global CLAHE ---
            s2 = np.stack([ig_plane, clahe_mild.apply(ig_plane), clahe_strong.apply(ig_plane)], axis=-1)

            # --- TOWER 3: Local Sigmoid ---
            img_f = il_plane.astype(np.float32) / 255.0
            sig1 = (1.0 / (1.0 + np.exp(-8 * (img_f - 0.75))) * 255).astype(np.uint8)
            sig2 = (1.0 / (1.0 + np.exp(-15 * (img_f - 0.60))) * 255).astype(np.uint8)
            s3 = np.stack([il_plane, sig1, sig2], axis=-1)

            # --- TOWER 4: Local CLAHE ---
            s4 = np.stack([il_plane, clahe_mild.apply(il_plane), clahe_strong.apply(il_plane)], axis=-1)

            batch_s1.append(preprocess_input(s1.astype(np.float32)))
            batch_s2.append(preprocess_input(s2.astype(np.float32)))
            batch_s3.append(preprocess_input(s3.astype(np.float32)))
            batch_s4.append(preprocess_input(s4.astype(np.float32)))

        in_meta = np.repeat(np.expand_dims(meta_vector, axis=0), N, axis=0)
        return shap_model.predict([np.array(batch_s1), np.array(batch_s2), np.array(batch_s3), np.array(batch_s4), in_meta], verbose=0)

    # 4. Get the Initial Prediction
    preds = unified_prediction_wrapper(np.expand_dims(wide_image, axis=0))
    pred_idx = np.argmax(preds[0])

    # 5. Dual-Class Logic (Missed Signal setup)
    if pred_idx != true_label_idx:
        indices_to_explain = [pred_idx, true_label_idx]
        labels_to_explain = [f"Pred: {CLASS_NAMES[pred_idx]}", f"True: {CLASS_NAMES[true_label_idx]}"]
        print_mode = "DUAL"
    else:
        indices_to_explain = [pred_idx]
        labels_to_explain = [f"Confirmed: {CLASS_NAMES[pred_idx]}"]
        print_mode = "SINGLE"

    # Render Header (Bar chart)
    with output_widget:
        render_diagnosis_header(preds[0], true_label_idx, pred_idx, case_id)

    if progress_bar:
        progress_bar.value = 30
        progress_bar.description = "Calculating Unified Gradients (Wait)..."

    # 6. RUN SHAP ON THE UNIFIED SYSTEM
    masker = shap.maskers.Image("inpaint_telea", wide_image.shape)
    explainer = shap.Explainer(unified_prediction_wrapper, masker, output_names=CLASS_NAMES)

    # 600 max_evals provides high resolution. Batch size kept at 10 to save RAM.
    shap_values = explainer(
        np.expand_dims(wide_image, axis=0),
        max_evals=1200,
        batch_size=10,
        outputs=indices_to_explain
    )

    # Extract the arrays for plotting
    shap_arrays = []
    for i in range(len(indices_to_explain)):
        vals = shap_values.values[..., i]
        shap_arrays.append(vals)

    # 7. Render Unified Output
    with output_widget:
        print(f"\n{'='*80}")
        print(" UNIFIED EXPERT SYSTEM CONSENSUS")
        print("Left Side: Global Crop | Right Side: Local Crop")
        if print_mode == "DUAL":
            print(f"Comparing Evidence: {labels_to_explain[0]} vs {labels_to_explain[1]}")
        else:
            print(f"Explaining Evidence: {labels_to_explain[0]}")

        shap.image_plot(
            shap_arrays,
            np.array([wide_image.astype(np.float32) / 255.0]), # SHAP expects 0-1 for background rendering
            labels=labels_to_explain,
            show=True,
            width=30
        )

    # Cleanup memory
    del explainer, shap_values, masker, shap_arrays
    gc.collect()

    if progress_bar:
        progress_bar.value = 100
        progress_bar.description = "Done!"

# ==========================================
# 4. VISUALIZATION HELPER
# ==========================================
def render_diagnosis_header(probs, true_idx, pred_idx, case_id):
    fig1 = plt.figure(figsize=(12, 4), layout='constrained')
    gs1 = fig1.add_gridspec(1, 1)

    true_name = CLASS_NAMES[true_idx]
    pred_name = CLASS_NAMES[pred_idx]
    conf = probs[pred_idx] * 100
    color = 'green' if true_idx == pred_idx else 'red'
    status = "CORRECT" if color == 'green' else "MISDIAGNOSIS"

    fig1.suptitle(f"CASE {case_id}: {status}\nTrue: {true_name} | Pred: {pred_name} ({conf:.1f}%)",
                 fontsize=16, weight='bold', color=color)

    ax = fig1.add_subplot(gs1[0])
    bars = ax.bar(CLASS_NAMES, probs*100, color=['lightgray']*5, width=0.6)
    bars[true_idx].set_color('green')
    if true_idx != pred_idx: bars[pred_idx].set_color('red')

    for bar in bars:
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+1, f'{bar.get_height():.1f}%', ha='center', va='bottom')

    ax.set_ylim(0, 110)
    ax.set_ylabel("Confidence %")
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
    plt.show()

# ==========================================
# 5. MANUAL EXECUTION CELL
# ==========================================
TARGET_DATASET = 'Test Set' # <-- Select 'Validation Set' or 'Test Set' here
TARGET_CASE_ID = 206        # <-- Select Case ID here

print(f" Initializing Unified Black Box Analysis for Case {TARGET_CASE_ID}...")

# Create UI Elements directly in the cell
manual_output = widgets.Output()
dummy_bar = widgets.IntProgress(min=0, max=100, description='Running:', bar_style='info', layout=widgets.Layout(width='98%'))

display(dummy_bar)
display(manual_output)

try:
    run_unified_missed_signal(
        TARGET_DATASET,
        TARGET_CASE_ID,
        progress_bar=dummy_bar,
        output_widget=manual_output
    )
    print("\n Unified Analysis Complete. Check the widget area above.")
except Exception as e:
    print(f" Error: {e}")

#XVII. TCAV on Concatenated Data

##TCAV v1.0

####Implementing TCAV

#####One-time Cache Builder

In [ ]:
import os
import gc
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, Model, Input
from tensorflow.keras.applications.xception import preprocess_input
import cv2

# ==========================================
# 1. SETUP & DATA LOADING
# ==========================================
DATA_PATH = '/content/drive/MyDrive/CMMD_preprocessed/v1.0'
MODEL_PATH = '/content/drive/My Drive/Xception model/v1.0/phase2_v1.0_final.keras'
CACHE_PATH = os.path.join(DATA_PATH, 'val_fusion_vectors.npy')

print(" Lazy Loading Data...")
X_val_global = np.load(os.path.join(DATA_PATH, 'X_val_global.npy'), mmap_mode='r')
X_val_local  = np.load(os.path.join(DATA_PATH, 'X_val_local.npy'), mmap_mode='r')
X_val_metadata = np.load(os.path.join(DATA_PATH, 'X_val_metadata.npy'), mmap_mode='r')
y_val = np.load(os.path.join(DATA_PATH, 'y_val.npy'), mmap_mode='r')

# ==========================================
# 2. DEFINE FULL ARCHITECTURE
# ==========================================
def create_full_quad_model(num_classes=5):
    def build_tower(input_shape, name_suffix):
        inp = Input(shape=input_shape, name=f"in_{name_suffix}")
        base = tf.keras.applications.Xception(weights=None, include_top=False, input_shape=input_shape, name=f"xception_{name_suffix}")
        base.trainable = False
        x = base(inp)
        x = layers.GlobalAveragePooling2D()(x)
        x = layers.Dense(1024, use_bias=False)(x)
        x = layers.BatchNormalization()(x)
        x = layers.Activation('swish')(x)
        x = layers.Dropout(0.5)(x)
        x = layers.Dense(512, use_bias=False)(x)
        x = layers.BatchNormalization()(x)
        x = layers.Activation('swish')(x)
        x = layers.Dropout(0.5)(x)
        att = layers.Dense(512, activation='sigmoid', name=f"att_gate_{name_suffix}")(x)
        boost = layers.Multiply()([x, att])
        x_res = layers.Add(name=f"feat_{name_suffix}")([x, boost])
        aux_out = layers.Dense(num_classes, activation='softmax', name=f"aux_{name_suffix}")(x_res)
        return inp, x_res, aux_out

    def apply_cross_attention(source_feat, target_feat, suffix):
        context = layers.Dense(128, activation='relu', name=f"ctx_ex_{suffix}")(target_feat)
        query = layers.Dense(128, activation='relu', name=f"query_ex_{suffix}")(source_feat)
        concat_qk = layers.Concatenate()([query, context])
        att_score = layers.Dense(512, activation='sigmoid', name=f"att_score_{suffix}")(concat_qk)
        ctx_scaled = layers.Dense(512, activation='relu', kernel_initializer='zeros', name=f"ctx_up_{suffix}")(context)
        return layers.Multiply()([ctx_scaled, att_score])

    in1, feat1, aux1 = build_tower((299,299,3), "t1_global_gamma")
    in2, feat2, aux2 = build_tower((299,299,3), "t2_global_clahe")
    in3, feat3, aux3 = build_tower((299,299,3), "t3_local_sigmoid")
    in4, feat4, aux4 = build_tower((299,299,3), "t4_local_clahe")

    c1_2, c1_3, c1_4 = apply_cross_attention(feat1, feat2, "t1_t2"), apply_cross_attention(feat1, feat3, "t1_t3"), apply_cross_attention(feat1, feat4, "t1_t4")
    feat1_enr = layers.Add()([feat1, c1_2, c1_3, c1_4])
    c2_1, c2_3, c2_4 = apply_cross_attention(feat2, feat1, "t2_t1"), apply_cross_attention(feat2, feat3, "t2_t3"), apply_cross_attention(feat2, feat4, "t2_t4")
    feat2_enr = layers.Add()([feat2, c2_1, c2_3, c2_4])
    c3_1, c3_2, c3_4 = apply_cross_attention(feat3, feat1, "t3_t1"), apply_cross_attention(feat3, feat2, "t3_t2"), apply_cross_attention(feat3, feat4, "t3_t4")
    feat3_enr = layers.Add()([feat3, c3_1, c3_2, c3_4])
    c4_1, c4_2, c4_3 = apply_cross_attention(feat4, feat1, "t4_t1"), apply_cross_attention(feat4, feat2, "t4_t2"), apply_cross_attention(feat4, feat3, "t4_t3")
    feat4_enr = layers.Add()([feat4, c4_1, c4_2, c4_3])

    in_meta = Input(shape=(5,), name="in_meta")
    feat_meta = layers.Dense(64, activation='relu')(in_meta)

    # Named Fusion Layer
    concat = layers.Concatenate(name='fusion_concat')([feat1_enr, feat2_enr, feat3_enr, feat4_enr, feat_meta])

    # Head layers (Needed just so load_weights doesn't crash)
    se = layers.Dense(2112//16, use_bias=False, name="head_se_reduce")(concat)
    se = layers.BatchNormalization(name="head_se_bn")(se)
    se = layers.Activation('swish')(se)
    se = layers.Dense(2112, activation='sigmoid', name="head_se_expand")(se)
    fused_weighted = layers.Multiply(name="head_se_mult")([concat, se])
    z = layers.Dense(1024, use_bias=False, name="head_d1")(fused_weighted)
    z = layers.BatchNormalization(name="head_bn1")(z)
    z = layers.Activation('swish')(z)
    z = layers.Dropout(0.3)(z)
    z = layers.Dense(512, use_bias=False, name="head_d2")(z)
    z = layers.BatchNormalization(name="head_bn2")(z)
    z = layers.Activation('swish')(z)
    z = layers.Dropout(0.3)(z)
    final_out = layers.Dense(num_classes, activation='softmax', name="final_output")(z)

    return Model(inputs=[in1, in2, in3, in4, in_meta], outputs=[final_out, aux1, aux2, aux3, aux4])

# ==========================================
# 3. LOAD WEIGHTS & EXTRACT BACKBONE
# ==========================================
print(" Initializing Full Model...")
full_model = create_full_quad_model()
try:
    full_model.load_weights(MODEL_PATH)
    print("   -> Full Model weights loaded successfully.")
except Exception as e:
    print(f"   -> CRITICAL ERROR: {e}")
    raise e # Stop execution if weights fail to load

print(" Extracting Backbone...")
backbone = Model(
    inputs=full_model.inputs,
    outputs=full_model.get_layer('fusion_concat').output
)
print("   -> Backbone extracted.")

# Delete full model to save RAM
del full_model
gc.collect()

# ==========================================
# 4. BUILD THE CACHE
# ==========================================
def build_and_save_cache(backbone_model, total_samples):
    print(f"\n Building Fusion Vector Cache for {total_samples} samples...")
    all_vectors = []
    batch_size = 32

    invGamma_mild = 1.0 / 0.6
    table_mild = np.array([((i / 255.0) ** invGamma_mild) * 255 for i in np.arange(0, 256)]).astype("uint8")
    table_strong = np.array([((i / 255.0) ** (1.0/1.5)) * 255 for i in np.arange(0, 256)]).astype("uint8")
    clahe_m = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    clahe_s = cv2.createCLAHE(clipLimit=4.0, tileGridSize=(8,8))

    for start in range(0, total_samples, batch_size):
        end = min(start + batch_size, total_samples)
        batch_idxs = range(start, end)

        s1, s2, s3, s4, metas = [], [], [], [], []

        for idx in batch_idxs:
            ig = np.array(X_val_global[idx]).astype(np.uint8)
            il = np.array(X_val_local[idx]).astype(np.uint8)
            if ig.ndim==3: ig=ig[:,:,0]
            if il.ndim==3: il=il[:,:,0]

            t1 = np.stack([ig, cv2.LUT(ig, table_mild), cv2.LUT(ig, table_strong)], axis=-1)
            t2 = np.stack([ig, clahe_m.apply(ig), clahe_s.apply(ig)], axis=-1)
            t3 = np.stack([il, il, il], axis=-1)
            t4 = np.stack([il, clahe_m.apply(il), clahe_s.apply(il)], axis=-1)

            s1.append(preprocess_input(t1.astype(np.float32)))
            s2.append(preprocess_input(t2.astype(np.float32)))
            s3.append(preprocess_input(t3.astype(np.float32)))
            s4.append(preprocess_input(t4.astype(np.float32)))
            metas.append(X_val_metadata[idx])

        inputs = [np.array(s1), np.array(s2), np.array(s3), np.array(s4), np.array(metas)]

        vectors = backbone_model.predict(inputs, verbose=0)
        all_vectors.append(vectors)

        print(f"   -> Processed {end}/{total_samples}...", end='\r')
        del inputs, s1, s2, s3, s4, metas
        gc.collect()

    final_cache = np.vstack(all_vectors)
    np.save(CACHE_PATH, final_cache)
    print(f"\n Cache saved successfully to: {CACHE_PATH} | Shape: {final_cache.shape}")

# Execute the caching process
build_and_save_cache(backbone, len(y_val))

# Clean up memory immediately
del backbone
tf.keras.backend.clear_session()
gc.collect()

####Fast Hypothesis Testing

In [ ]:
import os
import gc
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, Model, Input
from sklearn.linear_model import SGDClassifier
import scipy.stats as stats
import matplotlib.pyplot as plt
import seaborn as sns
import ipywidgets as widgets
from IPython.display import display, clear_output

# ==========================================
# 1. SETUP & INSTANT DATA LOADING
# ==========================================
DATA_PATH = '/content/drive/MyDrive/CMMD_preprocessed/v1.0'
MODEL_PATH = '/content/drive/My Drive/Xception model/v1.0/phase2_v1.0_final.keras'
CACHE_PATH = os.path.join(DATA_PATH, 'val_fusion_vectors.npy')

print(" Loading Metadata and Cache...")
X_val_metadata = np.load(os.path.join(DATA_PATH, 'X_val_metadata.npy'), mmap_mode='r')
y_labels = np.load(os.path.join(DATA_PATH, 'y_val.npy'), mmap_mode='r')

# Load the cached vectors directly into RAM
val_fusion_cache = np.load(CACHE_PATH)
print(f" Cache Loaded! Shape: {val_fusion_cache.shape}")

# Calculate real ages and define segments
AGE_MIN, AGE_MAX = 20.0, 90.0
real_ages = (X_val_metadata[:, 0] * (AGE_MAX - AGE_MIN)) + AGE_MIN

segments = {
    '< 40': (real_ages < 40),
    '40 - 60': (real_ages >= 40) & (real_ages <= 60),
    '> 60': (real_ages > 60)
}

# ==========================================
# 2. REBUILD TINY HEAD MODEL
# ==========================================
def create_head_model(input_dim=2112, num_classes=5):
    inp = Input(shape=(input_dim,), name="fusion_input")

    se_filters = input_dim // 16
    se = layers.Dense(se_filters, kernel_initializer='he_normal', use_bias=False)(inp)
    se = layers.BatchNormalization()(se)
    se = layers.Activation('swish')(se)
    se = layers.Dense(input_dim, kernel_initializer='he_normal')(se)
    se = layers.Activation('sigmoid')(se)

    fused_weighted = layers.Multiply()([inp, se])

    z = layers.Dense(1024, kernel_initializer='he_normal', use_bias=False)(fused_weighted)
    z = layers.BatchNormalization()(z)
    z = layers.Activation('swish')(z)
    z = layers.Dropout(0.3)(z)

    z = layers.Dense(512, kernel_initializer='he_normal', use_bias=False)(z)
    z = layers.BatchNormalization()(z)
    z = layers.Activation('swish')(z)
    z = layers.Dropout(0.3)(z)

    # Pure logits for mathematically accurate gradients
    out = layers.Dense(num_classes, activation=None, name="final_output")(z)
    return Model(inputs=inp, outputs=out)

print(" Extracting Head Weights via Signature Matching...")
dummy_model = tf.keras.models.load_model(MODEL_PATH, compile=False)
head_model = create_head_model(input_dim=2112)

dummy_weight_layers = [l for l in dummy_model.layers if len(l.get_weights()) > 0]
my_weight_layers = [l for l in head_model.layers if len(l.get_weights()) > 0]

for my_l in my_weight_layers:
    my_shapes = [w.shape for w in my_l.get_weights()]
    matching_dummy_layers = []
    for dum_l in dummy_weight_layers:
        dum_shapes = [w.shape for w in dum_l.get_weights()]
        if my_shapes == dum_shapes:
            matching_dummy_layers.append(dum_l)

    if not matching_dummy_layers:
        raise ValueError(f"CRITICAL: Could not find matching weights for shape {my_shapes}")

    chosen_dum_l = matching_dummy_layers[-1]
    my_l.set_weights(chosen_dum_l.get_weights())
    dummy_weight_layers.remove(chosen_dum_l)

print(" Head Model Weights Transferred Perfectly.")
del dummy_model
gc.collect()

# ==========================================
# 3. FAST TCAV ENGINE
# ==========================================
class Fast_TCAV:
    def __init__(self, head_model):
        self.head = head_model

    def train_cav(self, concept_indices, random_indices):
        self.classifier = SGDClassifier(alpha=0.01, max_iter=1000, tol=1e-3)
        c_vecs = val_fusion_cache[concept_indices]
        r_vecs = val_fusion_cache[random_indices]

        X = np.concatenate([c_vecs, r_vecs])
        y = np.concatenate([np.ones(len(c_vecs)), np.zeros(len(r_vecs))])

        self.classifier.fit(X, y)
        return self.classifier.coef_[0]

    def calculate_sensitivity(self, cav, target_indices, target_class_idx):
        vectors = val_fusion_cache[target_indices]
        vectors_tensor = tf.convert_to_tensor(vectors, dtype=tf.float32)

        with tf.GradientTape() as tape:
            tape.watch(vectors_tensor)
            preds = self.head(vectors_tensor)
            logit = preds[:, target_class_idx]

        grads = tape.gradient(logit, vectors_tensor)
        scores = tf.reduce_sum(grads * cav, axis=1).numpy()
        return scores

fast_tcav_engine = Fast_TCAV(head_model)

# ==========================================
# 4. INTERACTIVE DASHBOARD
# ==========================================
abnormality_map = {'Mass': 1, 'Calcification': 2, 'Both': 3}
class_map = {'Benign': 0, 'Luminal A': 1, 'Luminal B': 2, 'HER2': 3, 'Triple Negative': 4}

style = {'description_width': 'initial'}
AGE_OPTIONS = ['< 40', '40 - 60', '> 60']
ABNORM_OPTIONS = ['Mass', 'Calcification', 'Both']

w_category = widgets.Dropdown(options=['Age', 'Abnormality'], value='Age', description='1. Concept Category:', style=style)
w_value = widgets.Dropdown(options=AGE_OPTIONS, description='2. Concept Value:', style=style)
w_target = widgets.Dropdown(options=['Benign', 'Luminal A', 'Luminal B', 'HER2', 'Triple Negative'], value='Triple Negative', description='3. Target Class:', style=style)
w_run_btn = widgets.Button(description='Run Fast Robust Test (30 Splits)', button_style='primary', icon='bolt', layout=widgets.Layout(width='98%', height='50px'))
w_out = widgets.Output()

def on_category_change(change):
    if change['new'] == 'Age': w_value.options = AGE_OPTIONS
    else: w_value.options = ABNORM_OPTIONS
w_category.observe(on_category_change, names='value')

def on_button_click(b):
    w_out.clear_output()
    with w_out:
        cat = w_category.value
        val = w_value.value
        target_class = w_target.value

        print(f" Instant Analysis: '{val}' -> '{target_class}' (30 Iterations)")

        if cat == 'Age':
            if val == '< 40': mask = real_ages < 40
            elif val == '40 - 60': mask = (real_ages >= 40) & (real_ages <= 60)
            elif val == '> 60': mask = real_ages > 60
            c_idxs = np.where(mask)[0]
            r_pool = np.where(~mask)[0]
        else:
            col = abnormality_map.get(val) + 1
            c_idxs = np.where(X_val_metadata[:, col] == 1)[0]
            r_pool = np.where(X_val_metadata[:, col] == 0)[0]

        if len(c_idxs) < 10:
            print(f" Error: Not enough concept samples (Found {len(c_idxs)}). Need 10+.")
            return

        target_idx = class_map.get(target_class)
        t_idxs = np.where(np.argmax(y_labels, axis=1) == target_idx)[0]
        if len(t_idxs) > 50: t_idxs = t_idxs[:50]

        sample_size = min(len(c_idxs), len(r_pool))
        if sample_size > 100: sample_size = 100

        all_tcav_scores = []
        all_sensitivities = [] # Store raw gradients for KDE plot

        for i in range(30):
            c_batch = np.random.choice(c_idxs, size=sample_size, replace=False)
            r_batch = np.random.choice(r_pool, size=sample_size, replace=False)

            cav = fast_tcav_engine.train_cav(c_batch, r_batch)
            scores = fast_tcav_engine.calculate_sensitivity(cav, t_idxs, target_idx)

            all_tcav_scores.append(np.mean(scores > 0))
            all_sensitivities.extend(scores) # Collect all individual patient sensitivities

        all_tcav_scores = np.array(all_tcav_scores)
        mean_score = np.mean(all_tcav_scores)
        std_dev = np.std(all_tcav_scores)
        t_stat, p_value = stats.ttest_1samp(all_tcav_scores, 0.5)

        clear_output(wait=True)
        print(f" EXPERIMENT COMPLETE")
        print(f" Hypothesis: '{val}' -> '{target_class}'")
        print(f" Mean TCAV Score: {mean_score:.4f} ± {std_dev:.4f}")

        if p_value < 0.05:
            if mean_score > 0.5: print(f" CONCLUSION: Statistically Significant POSITIVE signal (p = {p_value:.4e}).")
            else: print(f" CONCLUSION: Statistically Significant NEGATIVE signal (p = {p_value:.4e}).")
        else:
            print(f" CONCLUSION: Not statistically significant (p = {p_value:.4e}).")

        # --- DUAL VISUALIZATION ---
        fig, axes = plt.subplots(1, 2, figsize=(14, 4))

        # Plot 1: Sensitivity Distribution (The "Why")
        sns.kdeplot(all_sensitivities, fill=True, color='teal', alpha=0.4, ax=axes[0])
        axes[0].axvline(0, color='red', linestyle='--', label='Neutral (0.0)')
        axes[0].set_title(f"Sensitivity Distribution (All Runs)\nShows magnitude of influence")
        axes[0].set_xlabel("Sensitivity (Gradient x CAV)")
        axes[0].set_ylabel("Density")
        axes[0].legend()

        # Plot 2: TCAV Score Stability (The "How Reliable")
        sns.boxplot(x=all_tcav_scores, color='lightblue', fliersize=5, ax=axes[1])
        sns.stripplot(x=all_tcav_scores, color='darkblue', alpha=0.5, jitter=True, ax=axes[1])
        axes[1].axvline(0.5, color='red', linestyle='--', label='Random Baseline (0.5)')
        axes[1].set_title(f"TCAV Score Stability (30 Splits)\nMean: {mean_score:.2f} ± {std_dev:.2f}")
        axes[1].set_xlabel("TCAV Score (>0.5 indicates positive correlation)")
        axes[1].set_xlim(-0.05, 1.05)
        axes[1].legend()

        plt.tight_layout()
        plt.show()

w_run_btn.on_click(on_button_click)

ui = widgets.VBox([
    widgets.HTML("<h3> Fast Robust TCAV Tester</h3>"),
    widgets.HBox([w_category, w_value]),
    w_target,
    w_run_btn,
    w_out
])

display(ui)

###Case by case analysis

In [ ]:
import os
import gc
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, Model, Input
from sklearn.linear_model import SGDClassifier
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt
import seaborn as sns

# ==========================================
# 1. SETUP & INSTANT DATA LOADING
# ==========================================
DATA_PATH = '/content/drive/MyDrive/CMMD_preprocessed/v1.0'
MODEL_PATH = '/content/drive/My Drive/Xception model/v1.0/phase2_v1.0_final.keras'
VAL_CACHE_PATH = os.path.join(DATA_PATH, 'val_fusion_vectors.npy')

print(" Loading Metadata and Validation Cache...")
# Global images are loaded just for plotting in the UI
X_val_global = np.load(os.path.join(DATA_PATH, 'X_val_global.npy'), mmap_mode='r')
X_val_metadata = np.load(os.path.join(DATA_PATH, 'X_val_metadata.npy'), mmap_mode='r')
y_val = np.load(os.path.join(DATA_PATH, 'y_val.npy'), mmap_mode='r')

val_fusion_cache = np.load(VAL_CACHE_PATH)
print(f" Cache Loaded! Val Shape: {val_fusion_cache.shape}")

# Calculate real ages
AGE_MIN, AGE_MAX = 20.0, 90.0
real_ages_val = (X_val_metadata[:, 0] * (AGE_MAX - AGE_MIN)) + AGE_MIN

# Maps
abnormality_map = {'Mass': 1, 'Calcification': 2, 'Both': 3}
class_map = {'Benign': 0, 'Luminal A': 1, 'Luminal B': 2, 'HER2': 3, 'Triple Negative': 4}
inv_class_map = {v: k for k, v in class_map.items()}

# ==========================================
# 2. REBUILD TINY HEAD MODEL
# ==========================================
def create_head_model(input_dim=2112, num_classes=5):
    inp = Input(shape=(input_dim,), name="fusion_input")
    se_filters = input_dim // 16
    se = layers.Dense(se_filters, kernel_initializer='he_normal', use_bias=False)(inp)
    se = layers.BatchNormalization()(se)
    se = layers.Activation('swish')(se)
    se = layers.Dense(input_dim, kernel_initializer='he_normal')(se)
    se = layers.Activation('sigmoid')(se)
    fused_weighted = layers.Multiply()([inp, se])
    z = layers.Dense(1024, kernel_initializer='he_normal', use_bias=False)(fused_weighted)
    z = layers.BatchNormalization()(z)
    z = layers.Activation('swish')(z)
    z = layers.Dropout(0.3)(z)
    z = layers.Dense(512, kernel_initializer='he_normal', use_bias=False)(z)
    z = layers.BatchNormalization()(z)
    z = layers.Activation('swish')(z)
    z = layers.Dropout(0.3)(z)

    # activation=None for pure logits
    out = layers.Dense(num_classes, activation=None, name="final_output")(z)
    return Model(inputs=inp, outputs=out)

print(" Extracting Head Weights via Signature Matching...")
dummy_model = tf.keras.models.load_model(MODEL_PATH, compile=False)
head_model = create_head_model(input_dim=2112)

dummy_weight_layers = [l for l in dummy_model.layers if len(l.get_weights()) > 0]
my_weight_layers = [l for l in head_model.layers if len(l.get_weights()) > 0]

for my_l in my_weight_layers:
    my_shapes = [w.shape for w in my_l.get_weights()]
    matching_dummy_layers = []
    for dum_l in dummy_weight_layers:
        dum_shapes = [w.shape for w in dum_l.get_weights()]
        if my_shapes == dum_shapes:
            matching_dummy_layers.append(dum_l)

    if not matching_dummy_layers:
        raise ValueError(f"CRITICAL: Could not find matching weights for shape {my_shapes}")

    chosen_dum_l = matching_dummy_layers[-1]
    my_l.set_weights(chosen_dum_l.get_weights())
    dummy_weight_layers.remove(chosen_dum_l)

print(" Head Model Weights Transferred Perfectly.")
del dummy_model
gc.collect()

# ==========================================
# 3. FAST SINGLE PATIENT ENGINE
# ==========================================
class Fast_Single_TCAV:
    def __init__(self, head_model):
        self.head = head_model
        # Optional: Attach softmax layer just for UI probability display
        self.softmax_layer = layers.Softmax()

    def train_cav(self, concept_indices, random_indices):
        c_vecs = val_fusion_cache[concept_indices]
        r_vecs = val_fusion_cache[random_indices]

        X = np.concatenate([c_vecs, r_vecs])
        y = np.concatenate([np.ones(len(c_vecs)), np.zeros(len(r_vecs))])

        self.classifier = SGDClassifier(alpha=0.01, max_iter=1000, tol=1e-3)
        self.classifier.fit(X, y)
        return self.classifier.coef_[0]

    def calculate_individual_sensitivity(self, cav, fusion_vec, target_class_idx):
        fusion_vec = fusion_vec[np.newaxis, ...]
        vectors_tensor = tf.convert_to_tensor(fusion_vec, dtype=tf.float32)

        with tf.GradientTape() as tape:
            tape.watch(vectors_tensor)
            logits = self.head(vectors_tensor)
            target_logit = logits[:, target_class_idx]

        grads = tape.gradient(target_logit, vectors_tensor)
        score = tf.reduce_sum(grads * cav).numpy()
        probs = self.softmax_layer(logits).numpy()[0]
        return score, probs

fast_engine = Fast_Single_TCAV(head_model)

def calculate_population_context(cav, target_class_idx, limit=50):
    target_indices = np.where(np.argmax(y_val, axis=1) == target_class_idx)[0]
    if len(target_indices) > limit:
        target_indices = np.random.choice(target_indices, size=limit, replace=False)

    scores = []
    for idx in target_indices:
        vec = val_fusion_cache[idx]
        s, _ = fast_engine.calculate_individual_sensitivity(cav, vec, target_class_idx)
        scores.append(s)
    return np.array(scores)

# ==========================================
# 4. INTERACTIVE DASHBOARD
# ==========================================
def get_patient_options():
    options = []
    total = min(len(y_val), 300)
    for i in range(total):
        true_class = inv_class_map[np.argmax(y_val[i])]
        options.append((f"{i} [{true_class}]", i))
    return options

style = {'description_width': 'initial'}

w_patient_idx = widgets.Dropdown(options=get_patient_options(), description='Patient (Val Set):', style=style, layout=widgets.Layout(width='50%'))
w_load_btn = widgets.Button(description='Load Info', button_style='info', icon='user', layout=widgets.Layout(width='20%'))

w_concept_cat = widgets.Dropdown(options=['Abnormality', 'Age'], value='Abnormality', description='Hypothesis:', style=style)
w_concept_val = widgets.Dropdown(options=['Mass', 'Calcification', 'Both'], description='Value:', style=style)

def update_options(change):
    if change['new'] == 'Age': w_concept_val.options = ['> 65', '> 50', '< 45', '< 35']
    else: w_concept_val.options = ['Mass', 'Calcification', 'Both']
w_concept_cat.observe(update_options, names='value')

w_test_btn = widgets.Button(description='Run Analysis', button_style='success', icon='chart-bar', layout=widgets.Layout(width='98%'))

out_patient_info = widgets.Output()
out_result = widgets.Output()

def on_load_click(b):
    out_patient_info.clear_output()
    out_result.clear_output()
    idx = w_patient_idx.value

    meta, label, img = X_val_metadata[idx], y_val[idx], X_val_global[idx]

    age = (meta[0] * (AGE_MAX - AGE_MIN)) + AGE_MIN
    ab_str = "None"
    if meta[2]==1: ab_str="Mass"
    elif meta[3]==1: ab_str="Calcification"
    elif meta[4]==1: ab_str="Both"

    with out_patient_info:
        fig, ax = plt.subplots(1, 1, figsize=(3, 3))
        disp_img = img.astype(np.float32)
        if disp_img.max() > 1.0: disp_img /= 255.0
        ax.imshow(disp_img, cmap='bone')
        ax.axis('off')
        ax.set_title(f"Patient #{idx}\n{inv_class_map[np.argmax(label)]}")
        plt.show()
        print(f" Profile: Age={age:.2f} | {ab_str}")

def on_test_click(b):
    out_result.clear_output()
    with out_result:
        print(" Analyzing Patient vs. Population Context...")

        cat, val = w_concept_cat.value, w_concept_val.value
        if cat == 'Age':
            thresh = float(val.replace('>','').replace('<','').strip())
            if '>' in val:
                c_idxs = np.where(real_ages_val > thresh)[0]
                r_idxs = np.where(real_ages_val < thresh - 15)[0]
            else:
                c_idxs = np.where(real_ages_val < thresh)[0]
                r_idxs = np.where(real_ages_val > thresh + 15)[0]
        else:
            col = abnormality_map.get(val) + 1
            c_idxs = np.where(X_val_metadata[:, col] == 1)[0]
            r_idxs = np.where(X_val_metadata[:, col] == 0)[0]

        if len(c_idxs) < 10: print(" Error: Not enough concept samples."); return
        if len(c_idxs) > 100: c_idxs = np.random.choice(c_idxs, size=100, replace=False)
        r_idxs = np.random.choice(r_idxs, size=len(c_idxs), replace=False)

        cav = fast_engine.train_cav(c_idxs, r_idxs)

        idx = w_patient_idx.value
        patient_vector = val_fusion_cache[idx]

        dummy_tensor = tf.convert_to_tensor(patient_vector[np.newaxis, ...], dtype=tf.float32)
        raw_logits = fast_engine.head(dummy_tensor)
        pred_class_idx = np.argmax(raw_logits.numpy()[0])
        pred_str = inv_class_map[pred_class_idx]

        patient_score, probs = fast_engine.calculate_individual_sensitivity(cav, patient_vector, pred_class_idx)

        pop_scores = calculate_population_context(cav, pred_class_idx)
        percentile = (np.sum(pop_scores < patient_score) / len(pop_scores)) * 100

        clear_output(wait=True)
        print(f" DIAGNOSIS: {pred_str} ({probs[pred_class_idx]*100:.1f}%)")
        print(f" Hypothesis: '{val}' influence")
        print(f" Patient Score: {patient_score:.4f}")
        print(f" Population Mean: {np.mean(pop_scores):.4f}")

        if patient_score > np.percentile(pop_scores, 75):
            verdict = "HIGH IMPACT (Top 25%)"
            color = 'green'
        elif patient_score < np.percentile(pop_scores, 25):
            verdict = "LOW IMPACT (Bottom 25%)"
            color = 'red'
        else:
            verdict = "AVERAGE IMPACT"
            color = 'orange'

        print(f" VERDICT: {verdict}")

        plt.figure(figsize=(10, 4))
        sns.kdeplot(pop_scores, fill=True, color='gray', alpha=0.3, label=f"Typical '{pred_str}' Patients")
        plt.axvline(0, color='black', linestyle='--', linewidth=1)
        plt.axvline(patient_score, color=color, linestyle='-', linewidth=3, label="THIS PATIENT")
        plt.text(patient_score, plt.ylim()[1]*0.9, f" {patient_score:.3f}", color=color, fontweight='bold')
        plt.title(f"Where does this patient stand? ({percentile:.0f}th Percentile)")
        plt.xlabel("Sensitivity Score (Right = Higher Impact)")
        plt.legend()
        plt.yticks([])
        plt.show()

w_load_btn.on_click(on_load_click)
w_test_btn.on_click(on_test_click)

ui = widgets.VBox([
    widgets.HTML("<h3> Fast Single Patient Contextualizer</h3>"),
    widgets.HBox([w_patient_idx, w_load_btn]),
    out_patient_info,
    widgets.HTML("<hr>"),
    widgets.HBox([w_concept_cat, w_concept_val]),
    w_test_btn,
    out_result
])

display(ui)